In [ ]:
import sys
import platform
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

print("=== Notebook 05: Initial Environment Check ===")
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Working directory:", Path.cwd())
print("Google Drive mounted:", Path("/content/drive/MyDrive").is_dir())

for package in (
    "torch", "torchvision", "ultralytics",
    "numpy", "pycocotools", "Pillow"
):
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("PyTorch is unavailable.")

print("No packages installed. No files modified. No inference performed.")

=== Notebook 05: Initial Environment Check ===
Python: 3.13.18
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
Working directory: /content
Google Drive mounted: False
torch: 2.14.1+cpu
torchvision: 0.29.1+cpu
ultralytics: NOT INSTALLED
numpy: 2.4.6
pycocotools: 2.0.11
Pillow: 11.3.0
CUDA available: False
No packages installed. No files modified. No inference performed.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path
import subprocess

project = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")
repo = project / "code"

if not (repo / ".git").is_dir():
    raise RuntimeError("Expected Git repository was not found.")

for label, args in [
    ("Branch", ["branch", "--show-current"]),
    ("HEAD", ["rev-parse", "HEAD"]),
    ("Working tree", [
        "--no-optional-locks", "status", "--short",
        "--untracked-files=all"
    ]),
]:
    result = subprocess.run(
        ["git", *args],
        cwd=repo, capture_output=True, text=True, check=True
    )
    print(f"\n--- {label} ---")
    print(result.stdout.strip() or "(clean)")

paths = {
    "Audit report": repo / (
        "docs/research_reports/Pre_Notebook05_Reproducibility_Audit_v1.md"
    ),
    "Checkpoint": project / "models/yolov8s.pt",
    "COCO images": project / "datasets/coco2017/images/val2017",
    "COCO annotations": project / (
        "datasets/coco2017/annotations/instances_val2017.json"
    ),
}

print("\n--- Required paths ---")
for label, path in paths.items():
    exists = path.is_dir() if label == "COCO images" else path.is_file()
    print(f"{label}: {'FOUND' if exists else 'MISSING'}")

print("\nNo files modified. No inference performed.")


--- Branch ---
main

--- HEAD ---
a8eaad0fdb3acd82320b99c3fe39cf82092f9c61

--- Working tree ---
(clean)

--- Required paths ---
Audit report: FOUND
Checkpoint: FOUND
COCO images: FOUND
COCO annotations: FOUND

No files modified. No inference performed.


In [ ]:
from pathlib import Path
import json

repo = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code"
)

paths = [
    "experiments/baseline/baseline_metadata.json",
    "results/baseline_validation/baseline_coco2017_metrics.json",
    "results/official_cocoeval/yolov8s_official_cocoeval_metrics.json",
]

for relative_path in paths:
    path = repo / relative_path
    print(f"\n--- {relative_path} ---")

    if not path.is_file():
        print("NOT FOUND")
        continue

    data = json.loads(path.read_text(encoding="utf-8"))
    print(json.dumps(data, indent=2, ensure_ascii=False))

print("\nRead-only inspection. No dataset split created.")


--- experiments/baseline/baseline_metadata.json ---
{
  "model": "YOLOv8s",
  "weights": "yolov8s.pt",
  "ultralytics_version": "8.4.189",
  "torch_version": "2.11.0+cu130",
  "python_version": "3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]",
  "gpu": "Tesla T4",
  "image_source": "https://ultralytics.com/images/bus.jpg",
  "image_size": 640,
  "confidence_threshold": 0.25,
  "detected_objects": 5,
  "wall_time_seconds": 13.6625
}

--- results/baseline_validation/baseline_coco2017_metrics.json ---
{
  "experiment": "yolov8s_coco2017_baseline",
  "timestamp_utc": "2026-10-28T10:55:06.875387+00:00",
  "model": "yolov8s.pt",
  "dataset": "COCO 2017 val",
  "num_images": 5000,
  "num_ground_truth_boxes_yolo": 36335,
  "evaluation_method": "Ultralytics YOLO labels",
  "official_cocoeval": false,
  "settings": {
    "imgsz": 640,
    "batch": 16,
    "device": 0,
    "conf": 0.001,
    "iou": 0.7,
    "max_det": 300
  },
  "environment": {
    "ultralytics": "8.4.189",
    "torch": "2.

In [ ]:
from pathlib import Path
import json

path = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/"
    "notebooks/03_baseline_validation.ipynb"
)
notebook = json.loads(path.read_text(encoding="utf-8"))

for number, cell in enumerate(notebook["cells"], start=1):
    source = cell.get("source", [])
    source = source if isinstance(source, str) else "".join(source)

    if cell["cell_type"] == "code" and any(
        term in source
        for term in ("iscrowd", "category_id", "category_mapping", "coco80")
    ):
        print(f"\n--- Cell {number} ---")
        print(source)

print("\nRead-only inspection. No files modified.")


--- Cell 11 ---

import json
from pathlib import Path
from collections import Counter

COCO_ROOT = Path("/content/datasets/coco2017")
json_path = COCO_ROOT / "annotations" / "instances_val2017.json"
labels_dir = COCO_ROOT / "labels" / "val2017"
labels_dir.mkdir(parents=True, exist_ok=True)

with open(json_path, "r") as f:
    coco = json.load(f)

# Official COCO category IDs in YOLO class order
coco91_to_coco80 = [
    1, 2, 3, 4, 5, 6, 7, 8, 9, 10,
    11, 13, 14, 15, 16, 17, 18, 19, 20, 21,
    22, 23, 24, 25, 27, 28, 31, 32, 33, 34,
    35, 36, 37, 38, 39, 40, 41, 42, 43, 44,
    46, 47, 48, 49, 50, 51, 52, 53, 54, 55,
    56, 57, 58, 59, 60, 61, 62, 63, 64, 65,
    67, 70, 72, 73, 74, 75, 76, 77, 78, 79,
    80, 81, 82, 84, 85, 86, 87, 88, 89, 90
]

category_map = {
    category_id: index
    for index, category_id in enumerate(coco91_to_coco80)
}

assert len(category_map) == 80
assert set(category_map) == {
    c["id"] for c in coco["categories"]
}

images = {img["id"]: img for i

In [ ]:
import json
from pathlib import Path
from collections import Counter

path = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/"
    "datasets/coco2017/annotations/instances_val2017.json"
)

with path.open(encoding="utf-8") as stream:
    coco = json.load(stream)

counts = Counter()

for annotation in coco["annotations"]:
    counts["total"] += 1

    if annotation.get("iscrowd", 0):
        counts["crowd_skipped"] += 1
        continue

    _, _, width, height = annotation["bbox"]
    if width <= 0 or height <= 0:
        counts["invalid_boxes_skipped"] += 1
        continue

    counts["retained_by_historical_conversion"] += 1

for key in (
    "total", "crowd_skipped", "invalid_boxes_skipped",
    "retained_by_historical_conversion"
):
    print(f"{key}: {counts[key]}")

print(
    "Matches recorded YOLO box count:",
    counts["retained_by_historical_conversion"] == 36335
)
print("No labels generated. No files modified.")

total: 36781
crowd_skipped: 446
invalid_boxes_skipped: 0
retained_by_historical_conversion: 36335
Matches recorded YOLO box count: True
No labels generated. No files modified.


In [ ]:
import json
from pathlib import Path

folder = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/notebooks"
)

terms = (
    "COCOeval(", ".evaluate(", ".accumulate(",
    "params.imgIds", "loadRes(", "model.val(", "yolo.val("
)

for name in (
    "03_baseline_validation.ipynb",
    "03_baseline_results_report.ipynb",
):
    path = folder / name
    print(f"\n=== {name} ===")

    if not path.is_file():
        print("NOT FOUND")
        continue

    notebook = json.loads(path.read_text(encoding="utf-8"))
    found = False

    for number, cell in enumerate(notebook["cells"], start=1):
        source = cell.get("source", [])
        source = source if isinstance(source, str) else "".join(source)

        if cell["cell_type"] == "code" and any(
            term in source for term in terms
        ):
            found = True
            print(f"\n--- Cell {number} ---")
            print(source)

    if not found:
        print("No matching evaluation cells found.")

print("\nRead-only inspection. No files modified.")


=== 03_baseline_validation.ipynb ===

--- Cell 7 ---

from ultralytics import YOLO
from pathlib import Path

# Quick dataset pipeline check (not final evaluation)
model = YOLO("yolov8s.pt")

results = model.val(
    data="coco8.yaml",
    imgsz=640,
    batch=4,
    device=0,
    workers=2,
    project=str(Path(CODE_DIR) / "results" / "baseline_validation"),
    name="coco8_smoke_test",
    exist_ok=True
)

print("=" * 50)
print("COCO8 Smoke Test Results")
print("=" * 50)
print(f"mAP50: {results.box.map50:.4f}")
print(f"mAP50-95: {results.box.map:.4f}")


--- Cell 14 ---

from pathlib import Path
import time

print("=" * 50)
print("YOLOv8s - COCO 2017 Baseline Validation")
print("=" * 50)

start_time = time.time()

metrics = model.val(
    data="/content/datasets/coco2017/coco2017_val.yaml",
    split="val",
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    conf=0.001,
    iou=0.7,
    max_det=300,
    project=str(Path(CODE_DIR) / "results" / "baseline_validation"),
    na

In [ ]:
import json

protocol_draft = {
    "status": "DRAFT_NOT_FROZEN",
    "model": "YOLOv8s",
    "dataset": "COCO2017 val2017",
    "split": {
        "method": "seeded_random_permutation_of_sorted_image_ids",
        "seed": 20261108,
        "development_images": 1000,
        "held_out_images": 4000,
        "pilot_images": 50,
        "pilot_source": "development_only",
        "prior_use": "All 5000 images have recorded baseline evaluation.",
        "restriction": (
            "Held-out images must not guide method selection, "
            "layer selection, thresholds or hyperparameter tuning."
        ),
    },
    "evaluation": {
        "primary": "pycocotools COCOeval bbox",
        "ground_truth": "original_instances_val2017.json",
        "imgsz": 640,
        "confidence_threshold": 0.001,
        "nms_iou_threshold": 0.7,
        "inference_max_det": 300,
        "cocoeval_max_dets": [1, 10, 100],
        "comparisons": [
            "FP32 clean",
            "target-layer logical INT4 clean",
            "same quantized baseline with one injected weight-bit fault",
        ],
        "primary_outcome": "AP50:95 change relative to matched clean baseline",
        "secondary_outcomes": [
            "AP50", "AP75", "AR100",
            "separate box-coordinate and class-score differences",
            "post-NMS detection changes",
        ],
    },
    "pending_before_freeze": [
        "Audit known development image overlap, including the bus image.",
        "Check split category coverage before any performance evaluation.",
        "Define layer, weight and bit sampling and replicate counts.",
        "Define fault persistence across images.",
        "Validate environment and evaluation implementation.",
        "Define uncertainty estimation and compute budget after the pilot.",
    ],
}

assert (
    protocol_draft["split"]["development_images"]
    + protocol_draft["split"]["held_out_images"]
) == 5000

print(json.dumps(protocol_draft, indent=2))
print("\nDraft only. No split created. No files modified.")

{
  "status": "DRAFT_NOT_FROZEN",
  "model": "YOLOv8s",
  "dataset": "COCO2017 val2017",
  "split": {
    "method": "seeded_random_permutation_of_sorted_image_ids",
    "seed": 20261108,
    "development_images": 1000,
    "held_out_images": 4000,
    "pilot_images": 50,
    "pilot_source": "development_only",
    "prior_use": "All 5000 images have recorded baseline evaluation.",
    "restriction": "Held-out images must not guide method selection, layer selection, thresholds or hyperparameter tuning."
  },
  "evaluation": {
    "primary": "pycocotools COCOeval bbox",
    "ground_truth": "original_instances_val2017.json",
    "imgsz": 640,
    "confidence_threshold": 0.001,
    "nms_iou_threshold": 0.7,
    "inference_max_det": 300,
    "cocoeval_max_dets": [
      1,
      10,
      100
    ],
    "comparisons": [
      "FP32 clean",
      "target-layer logical INT4 clean",
      "same quantized baseline with one injected weight-bit fault"
    ],
    "primary_outcome": "AP50:95 change 

In [ ]:
import json
import hashlib
from pathlib import Path
from PIL import Image

project = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")
bus_path = project / "data/single_image_test/ultralytics_bus.jpg"
root = project / "datasets/coco2017"

if not bus_path.is_file():
    raise FileNotFoundError(f"Historical image not found: {bus_path}")

with (root / "annotations/instances_val2017.json").open(
    encoding="utf-8"
) as stream:
    records = json.load(stream)["images"]

with Image.open(bus_path) as image:
    bus = image.convert("RGB")
    bus_size = bus.size
    bus_hash = hashlib.sha256(bus.tobytes()).hexdigest()

candidates = [
    record for record in records
    if (record["width"], record["height"]) == bus_size
]

matches = []
for record in candidates:
    path = root / "images/val2017" / record["file_name"]
    with Image.open(path) as image:
        pixels = image.convert("RGB")
        digest = hashlib.sha256(pixels.tobytes()).hexdigest()
        if pixels.size == bus_size and digest == bus_hash:
            matches.append({
                "image_id": record["id"],
                "file_name": record["file_name"],
            })

print("Historical image dimensions:", bus_size)
print("Same-dimension candidates:", len(candidates))
print("Exact decoded RGB matches:", len(matches))
for match in matches:
    print(match)

print("Resized, cropped or re-encoded variants are not excluded by this check.")
print("No split created. No files modified.")

Historical image dimensions: (810, 1080)
Same-dimension candidates: 0
Exact decoded RGB matches: 0
Resized, cropped or re-encoded variants are not excluded by this check.
No split created. No files modified.


In [ ]:
import json
import random
import hashlib
from pathlib import Path

annotation_path = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/"
    "datasets/coco2017/annotations/instances_val2017.json"
)

with annotation_path.open(encoding="utf-8") as stream:
    coco = json.load(stream)

all_ids = sorted(image["id"] for image in coco["images"])
if len(all_ids) != 5000 or len(set(all_ids)) != 5000:
    raise RuntimeError("Expected 5000 unique image IDs.")

shuffled_ids = all_ids.copy()
random.Random(20261108).shuffle(shuffled_ids)

development_ids = sorted(shuffled_ids[:1000])
held_out_ids = sorted(shuffled_ids[1000:])
pilot_ids = sorted(shuffled_ids[:50])

assert not set(development_ids) & set(held_out_ids)
assert set(development_ids) | set(held_out_ids) == set(all_ids)
assert set(pilot_ids) <= set(development_ids)

categories = {item["id"]: item["name"] for item in coco["categories"]}

def id_hash(ids):
    payload = json.dumps(sorted(ids), separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

for name, ids in (
    ("development", development_ids),
    ("held_out", held_out_ids),
    ("pilot", pilot_ids),
):
    selected = set(ids)
    covered = {
        ann["category_id"]
        for ann in coco["annotations"]
        if ann["image_id"] in selected
        and not ann.get("iscrowd", 0)
        and ann["bbox"][2] > 0
        and ann["bbox"][3] > 0
    }
    missing = [
        categories[category_id]
        for category_id in sorted(set(categories) - covered)
    ]

    print(f"\nSplit: {name}")
    print("Images:", len(ids))
    print("Image ID SHA256:", id_hash(ids))
    print("Categories with non-crowd valid boxes:", len(covered))
    print("Missing categories:", missing)

print("\nCandidate split only; not frozen or saved.")
print("No model evaluation performed. No files modified.")


Split: development
Images: 1000
Image ID SHA256: dfe9971cbbbe98ff2e22c5cc52929771c69e9a3b0483716428d0e335ee16d4a1
Categories with non-crowd valid boxes: 79
Missing categories: ['hair drier']

Split: held_out
Images: 4000
Image ID SHA256: a9f1c38ff470181bca82180e3f324491ad73dfcce31b21bdea3696f83b4fb95f
Categories with non-crowd valid boxes: 80
Missing categories: []

Split: pilot
Images: 50
Image ID SHA256: efc37384d48a94bfb62f419f2b2ff0df793c6229e793efc2c2f8335f8d66d408
Categories with non-crowd valid boxes: 55
Missing categories: ['bicycle', 'train', 'boat', 'traffic light', 'fire hydrant', 'parking meter', 'sheep', 'elephant', 'tie', 'suitcase', 'frisbee', 'skis', 'knife', 'apple', 'broccoli', 'carrot', 'hot dog', 'potted plant', 'mouse', 'keyboard', 'microwave', 'refrigerator', 'scissors', 'hair drier', 'toothbrush']

Candidate split only; not frozen or saved.
No model evaluation performed. No files modified.


In [ ]:
import json
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/"
    "datasets/coco2017/annotations/instances_val2017.json"
)

with path.open(encoding="utf-8") as stream:
    coco = json.load(stream)

category = next(
    item for item in coco["categories"]
    if item["name"] == "hair drier"
)

annotations = [
    ann for ann in coco["annotations"]
    if ann["category_id"] == category["id"]
    and not ann.get("iscrowd", 0)
    and ann["bbox"][2] > 0
    and ann["bbox"][3] > 0
]

image_ids = sorted({ann["image_id"] for ann in annotations})

print("Category:", category["name"])
print("Category ID:", category["id"])
print("Valid non-crowd annotations:", len(annotations))
print("Images containing this category:", len(image_ids))
print("Image IDs:", image_ids)
print("No split changed. No files modified.")

Category: hair drier
Category ID: 89
Valid non-crowd annotations: 11
Images containing this category: 9
Image IDs: [178028, 239041, 257169, 315219, 350002, 384136, 500464, 501368, 569976]
No split changed. No files modified.


In [ ]:
import json
import random
import hashlib
from pathlib import Path

project = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")
annotation_path = (
    project / "datasets/coco2017/annotations/instances_val2017.json"
)
annotation_bytes = annotation_path.read_bytes()
coco = json.loads(annotation_bytes)

ids = sorted(image["id"] for image in coco["images"])
assert len(ids) == len(set(ids)) == 5000

random.Random(20261108).shuffle(ids)
splits = {
    "development": sorted(ids[:1000]),
    "held_out": sorted(ids[1000:]),
    "pilot": sorted(ids[:50]),
}

expected = {
    "development": "dfe9971cbbbe98ff2e22c5cc52929771c69e9a3b0483716428d0e335ee16d4a1",
    "held_out": "a9f1c38ff470181bca82180e3f324491ad73dfcce31b21bdea3696f83b4fb95f",
    "pilot": "efc37384d48a94bfb62f419f2b2ff0df793c6229e793efc2c2f8335f8d66d408",
}

hashes = {}
for name, selected in splits.items():
    payload = json.dumps(selected, separators=(",", ":")).encode("utf-8")
    hashes[name] = hashlib.sha256(payload).hexdigest()
    if hashes[name] != expected[name]:
        raise RuntimeError(f"Candidate split changed: {name}")

assert not set(splits["development"]) & set(splits["held_out"])
assert set(splits["pilot"]) <= set(splits["development"])

manifest = {
    "schema_version": "1.0",
    "status": "IMAGE_IDS_FROZEN_EXPERIMENT_PROTOCOL_PENDING",
    "dataset": "COCO2017 val2017",
    "seed": 20261108,
    "method": "Python random shuffle of sorted image IDs",
    "annotation_sha256": hashlib.sha256(annotation_bytes).hexdigest(),
    "image_ids": splits,
    "image_id_sha256": hashes,
    "limitations": [
        "All images have prior recorded clean-baseline evaluation.",
        "Development has no valid non-crowd hair drier annotations.",
        "Pilot covers 55 categories and is intended for execution checks.",
        "Historical bus image has no exact decoded match; variants remain unaudited.",
        "Checkpoint pretraining provenance remains to be verified.",
    ],
    "restrictions": [
        "Use development images only for method and parameter selection.",
        "Do not evaluate held-out performance until the method is frozen.",
        "Do not interpret baseline exposure as proof of training exposure.",
    ],
}

output = (
    project / "code/experiments/notebook05/coco2017_split_manifest_v1.json"
)
output.parent.mkdir(parents=True, exist_ok=True)

content = (
    json.dumps(manifest, indent=2, ensure_ascii=False, allow_nan=False) + "\n"
).encode("utf-8")

with output.open("xb") as stream:
    stream.write(content)

assert output.read_bytes() == content
print("Saved:", output)
print("Manifest SHA256:", hashlib.sha256(content).hexdigest())
print("Split sizes:", {name: len(values) for name, values in splits.items()})
print("No images moved. No inference, commit or push performed.")

Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/experiments/notebook05/coco2017_split_manifest_v1.json
Manifest SHA256: 143c3273b2bf0cd6393cfe148a74e48fbafb74e770bd3fbdf3f14222355978ee
Split sizes: {'development': 1000, 'held_out': 4000, 'pilot': 50}
No images moved. No inference, commit or push performed.


In [ ]:
import json
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/"
    "results/fault_injection/int4_extended_layer_sensitivity_plan_v2.json"
)

plan = json.loads(path.read_text(encoding="utf-8"))

print("--- Plan metadata ---")
for key, value in plan.items():
    if key != "scenarios":
        print(f"{key}: {json.dumps(value, ensure_ascii=False)}")

scenarios = plan.get("scenarios", [])
print("\nScenario count:", len(scenarios))

print("\n--- First two scenarios ---")
for scenario in scenarios[:2]:
    print(json.dumps(scenario, indent=2, ensure_ascii=False))

print("\nRead-only inspection. No new fault scenarios created.")

--- Plan metadata ---
schema_version: "2.0"
experiment: "extended_balanced_layer_sensitivity"
model: "YOLOv8s"
seed: 20261105
representation: "logical_signed_INT4"
source_plan: "int4_extended_layer_sensitivity_plan_v1.json"
selected_layers: ["1.conv", "4.cv1.conv", "4.m.1.cv2.conv", "6.m.1.cv1.conv", "9.cv1.conv", "9.cv2.conv", "15.cv2.conv", "18.m.0.cv2.conv", "21.cv2.conv", "22.cv2.2.0.conv", "22.cv3.0.2", "22.cv3.2.1.conv"]
layer_count: 12
scenarios_per_layer: 8
scenario_count: 96
design: "Two scenarios per bit per layer, with at least two scenarios per quantized weight category per layer"
layer_audits: [{"layer": "1.conv", "bit_counts": {"0": 2, "1": 2, "2": 2, "3": 2}, "category_counts": {"positive": 3, "zero": 3, "negative": 2}}, {"layer": "4.cv1.conv", "bit_counts": {"0": 2, "1": 2, "2": 2, "3": 2}, "category_counts": {"zero": 3, "negative": 2, "positive": 3}}, {"layer": "4.m.1.cv2.conv", "bit_counts": {"0": 2, "1": 2, "2": 2, "3": 2}, "category_counts": {"positive": 3, "negativ

In [ ]:
import json

fault_protocol_draft = {
    "status": "DRAFT_NOT_FROZEN",
    "representation": "logical_signed_INT4_in_FP32_inference",
    "quantization": {
        "scope": "target_layer_only",
        "clean_integer_range": [-7, 7],
        "faulted_integer_range": [-8, 7],
        "scale_rule": "pending_explicit_definition_and_validation",
    },
    "fault_model": {
        "target": "one_weight_in_one_selected_layer",
        "operation": "single_bit_XOR",
        "persistence": "fixed_across_all_images_in_each_scenario",
        "reset": "restore_verified_clean_state_between_scenarios",
        "hardware_fault_rate_claim": False,
    },
    "pilot": {
        "images": "development_pilot_only",
        "scenarios": "small_documented_subset_of_historical_plan",
        "purpose": [
            "validate_injection_and_ground_truth_evaluation",
            "measure_runtime",
            "check_clean_vs_faulty_comparisons",
        ],
        "final_layer_ranking": False,
    },
    "main_study": {
        "layer_scope": "12_previously_selected_layers",
        "weight_sampling": "uniform_over_target_layer_weight_indices",
        "bit_sampling": "equal_allocation_over_bits_0_to_3",
        "weight_sign_balancing": False,
        "replicates": "pending_pilot_and_compute_budget",
        "seed": 20261109,
        "network_wide_representativeness_claim": False,
    },
    "comparisons": {
        "quantization_effect": "FP32_clean_vs_target_layer_INT4_clean",
        "fault_effect": "target_layer_INT4_clean_vs_matched_faulty_model",
    },
    "held_out_use": "prohibited_during_method_development",
}

print(json.dumps(fault_protocol_draft, indent=2))
print("\nDraft only. No scenarios sampled. No files modified.")

{
  "status": "DRAFT_NOT_FROZEN",
  "representation": "logical_signed_INT4_in_FP32_inference",
  "quantization": {
    "scope": "target_layer_only",
    "clean_integer_range": [
      -7,
      7
    ],
    "faulted_integer_range": [
      -8,
      7
    ],
    "scale_rule": "pending_explicit_definition_and_validation"
  },
  "fault_model": {
    "target": "one_weight_in_one_selected_layer",
    "operation": "single_bit_XOR",
    "persistence": "fixed_across_all_images_in_each_scenario",
    "reset": "restore_verified_clean_state_between_scenarios",
    "hardware_fault_rate_claim": false
  },
  "pilot": {
    "images": "development_pilot_only",
    "scenarios": "small_documented_subset_of_historical_plan",
    "purpose": [
      "validate_injection_and_ground_truth_evaluation",
      "measure_runtime",
      "check_clean_vs_faulty_comparisons"
    ],
    "final_layer_ranking": false
  },
  "main_study": {
    "layer_scope": "12_previously_selected_layers",
    "weight_sampling": "unif

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps", "ultralytics==8.4.189"
    ],
    check=True,
)

print("Ultralytics installation command completed.")
print("Dependencies were not installed or upgraded by this command.")
print("Import and dependency compatibility have not been checked yet.")

Ultralytics installation command completed.
Dependencies were not installed or upgraded by this command.
Import and dependency compatibility have not been checked yet.


In [ ]:
import subprocess
import sys
from importlib.metadata import version

print("Ultralytics installed version:", version("ultralytics"))
assert version("ultralytics") == "8.4.189"

import torch
import torchvision
import ultralytics
from ultralytics import YOLO
from ultralytics.utils.nms import non_max_suppression
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

print("Required imports: PASS")
print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)

result = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True,
)

print("\n--- Dependency check ---")
print("Exit code:", result.returncode)
print(result.stdout.strip() or result.stderr.strip() or "(no output)")
print("\nNo model loaded. No inference performed.")

Ultralytics installed version: 8.4.189
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Required imports: PASS
PyTorch: 2.14.1+cpu
torchvision: 0.29.1+cpu

--- Dependency check ---
Exit code: 1
ultralytics 8.4.189 requires nvidia-ml-py, which is not installed.
ultralytics 8.4.189 requires ultralytics-platform, which is not installed.
ultralytics 8.4.189 requires ultralytics-thop, which is not installed.
ipython 7.34.0 requires jedi, which is not installed.

No model loaded. No inference performed.


In [ ]:
from importlib.metadata import requires
from packaging.requirements import Requirement
from packaging.utils import canonicalize_name

missing = {
    "nvidia-ml-py",
    "ultralytics-platform",
    "ultralytics-thop",
    "jedi",
}

for package in ("ultralytics", "ipython"):
    print(f"\n--- Requirements declared by {package} ---")
    for entry in requires(package) or []:
        requirement = Requirement(entry)
        if canonicalize_name(requirement.name) in missing:
            print(entry)

print("\nMetadata inspection only. No packages installed.")


--- Requirements declared by ultralytics ---
nvidia-ml-py>=12.0.0
ultralytics-thop>=2.2.3
ultralytics-platform>=0.1.45; python_version >= "3.11"

--- Requirements declared by ipython ---
jedi (>=0.16)

Metadata inspection only. No packages installed.


In [ ]:
import subprocess
import sys

packages = [
    "nvidia-ml-py>=12.0.0",
    "ultralytics-thop>=2.2.3",
    "ultralytics-platform>=0.1.45",
    "jedi>=0.16",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", *packages],
    check=True,
)

result = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True,
)

print("\n--- Dependency check ---")
print("Exit code:", result.returncode)
print(result.stdout.strip() or result.stderr.strip() or "(no output)")
print("\nNo model loaded. No inference performed.")


--- Dependency check ---
Exit code: 0
No broken requirements found.

No model loaded. No inference performed.


In [ ]:
import hashlib
import json
from pathlib import Path

import torch
from ultralytics import YOLO

project = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")
checkpoint = project / "models/yolov8s.pt"
plan_path = (
    project / "code/results/fault_injection/"
    "int4_extended_layer_sensitivity_plan_v2.json"
)

expected_sha256 = (
    "1f47a78bf100391c2a140b7ac73a1caae18c32779be7d310658112f7ac9aa78a"
)

digest = hashlib.sha256()
with checkpoint.open("rb") as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        digest.update(chunk)

if checkpoint.stat().st_size != 22588772:
    raise RuntimeError("Checkpoint size mismatch.")
if digest.hexdigest() != expected_sha256:
    raise RuntimeError("Checkpoint SHA256 mismatch.")

yolo = YOLO(str(checkpoint))
model = yolo.model.cpu().eval()

plan = json.loads(plan_path.read_text(encoding="utf-8"))
modules = dict(model.named_modules())

print("Checkpoint SHA256: PASS")
print("Model class:", type(model).__name__)
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Classes:", len(yolo.names))
print("Device:", next(model.parameters()).device)

if len(yolo.names) != 80:
    raise RuntimeError("Unexpected class count.")

for layer_name in plan["selected_layers"]:
    module = modules.get("model." + layer_name)
    if not isinstance(module, torch.nn.Conv2d):
        raise RuntimeError(f"Expected Conv2d not found: {layer_name}")
    print(f"{layer_name}: {tuple(module.weight.shape)}")

print("Selected layer count:", len(plan["selected_layers"]))
print("No inference performed. Checkpoint file unchanged.")

Checkpoint SHA256: PASS
Model class: DetectionModel
Parameters: 11166560
Classes: 80
Device: cpu
1.conv: (64, 32, 3, 3)
4.cv1.conv: (128, 128, 1, 1)
4.m.1.cv2.conv: (64, 64, 3, 3)
6.m.1.cv1.conv: (128, 128, 3, 3)
9.cv1.conv: (256, 512, 1, 1)
9.cv2.conv: (512, 1024, 1, 1)
15.cv2.conv: (128, 192, 1, 1)
18.m.0.cv2.conv: (128, 128, 3, 3)
21.cv2.conv: (512, 768, 1, 1)
22.cv2.2.0.conv: (64, 512, 3, 3)
22.cv3.0.2: (80, 128, 1, 1)
22.cv3.2.1.conv: (128, 128, 3, 3)
Selected layer count: 12
No inference performed. Checkpoint file unchanged.


In [ ]:
import hashlib
import json
from pathlib import Path

import cv2
import numpy as np
import torch
from ultralytics.data.augment import LetterBox

project = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")
root = project / "datasets/coco2017"
manifest_path = (
    project / "code/experiments/notebook05/coco2017_split_manifest_v1.json"
)

manifest_bytes = manifest_path.read_bytes()
expected_hash = (
    "143c3273b2bf0cd6393cfe148a74e48fbafb74e770bd3fbdf3f14222355978ee"
)
if hashlib.sha256(manifest_bytes).hexdigest() != expected_hash:
    raise RuntimeError("Split manifest changed.")

manifest = json.loads(manifest_bytes)
annotation_bytes = (
    root / "annotations/instances_val2017.json"
).read_bytes()

if hashlib.sha256(annotation_bytes).hexdigest() != manifest["annotation_sha256"]:
    raise RuntimeError("Annotation file changed.")

coco = json.loads(annotation_bytes)
records = {item["id"]: item for item in coco["images"]}

pilot_image_id = manifest["image_ids"]["pilot"][0]
assert pilot_image_id in manifest["image_ids"]["development"]
assert pilot_image_id not in manifest["image_ids"]["held_out"]

pilot_record = records[pilot_image_id]
image_path = root / "images/val2017" / pilot_record["file_name"]
pilot_bgr = cv2.imread(str(image_path))

if pilot_bgr is None:
    raise RuntimeError("Image decoding failed.")
if pilot_bgr.shape[:2] != (pilot_record["height"], pilot_record["width"]):
    raise RuntimeError("Image dimensions disagree with annotations.")

letterbox = LetterBox(
    new_shape=(640, 640),
    auto=False,
    scale_fill=False,
    scaleup=True,
    stride=32,
)
padded_bgr = letterbox(image=pilot_bgr)
rgb_chw = np.ascontiguousarray(padded_bgr[:, :, ::-1].transpose(2, 0, 1))
pilot_tensor = torch.from_numpy(rgb_chw).unsqueeze(0).float() / 255.0

assert tuple(pilot_tensor.shape) == (1, 3, 640, 640)
assert torch.isfinite(pilot_tensor).all()
assert 0 <= pilot_tensor.min() <= pilot_tensor.max() <= 1

print("Pilot image ID:", pilot_image_id)
print("Filename:", pilot_record["file_name"])
print("Original shape:", pilot_bgr.shape)
print("Input shape:", tuple(pilot_tensor.shape))
print("Input dtype:", pilot_tensor.dtype)
print("Input range:", pilot_tensor.min().item(), pilot_tensor.max().item())
print("Preprocessing: BGR -> LetterBox -> RGB CHW -> FP32 / 255")
print("No inference performed. No files modified.")

Pilot image ID: 8844
Filename: 000000008844.jpg
Original shape: (426, 640, 3)
Input shape: (1, 3, 640, 640)
Input dtype: torch.float32
Input range: 0.0 1.0
Preprocessing: BGR -> LetterBox -> RGB CHW -> FP32 / 255
No inference performed. No files modified.


In [ ]:
import time
import torch
from ultralytics.utils.nms import non_max_suppression

model.cpu().eval()

start = time.perf_counter()
with torch.inference_mode():
    output = model(pilot_tensor)
    prediction = output[0] if isinstance(output, (tuple, list)) else output
    fp32_raw = prediction.detach().clone()

forward_seconds = time.perf_counter() - start

assert tuple(fp32_raw.shape) == (1, 84, 8400)
assert torch.isfinite(fp32_raw).all()

start = time.perf_counter()
with torch.inference_mode():
    fp32_detections = non_max_suppression(
        fp32_raw.clone(),
        conf_thres=0.001,
        iou_thres=0.7,
        nc=80,
        max_det=300,
    )[0].detach().cpu().clone()

nms_seconds = time.perf_counter() - start

assert fp32_detections.ndim == 2
assert fp32_detections.shape[1] == 6
assert torch.isfinite(fp32_detections).all()

if len(fp32_detections):
    assert (
        (fp32_detections[:, 4] >= 0)
        & (fp32_detections[:, 4] <= 1)
    ).all()
    assert (
        (fp32_detections[:, 5] >= 0)
        & (fp32_detections[:, 5] < 80)
        & (fp32_detections[:, 5] == fp32_detections[:, 5].round())
    ).all()

print("Image ID:", pilot_image_id)
print("Raw prediction shape:", tuple(fp32_raw.shape))
print("Post-NMS detections:", len(fp32_detections))
print("First forward time (seconds):", round(forward_seconds, 4))
print("NMS time (seconds):", round(nms_seconds, 4))
print("First-run timings are not steady-state benchmarks.")
print("No fault injected. No ground-truth evaluation performed.")

Image ID: 8844
Raw prediction shape: (1, 84, 8400)
Post-NMS detections: 217
First forward time (seconds): 2.0262
NMS time (seconds): 0.03
First-run timings are not steady-state benchmarks.
No fault injected. No ground-truth evaluation performed.


In [ ]:
import torch
from ultralytics.utils.ops import scale_boxes

category_ids = sorted(category["id"] for category in coco["categories"])
category_names = {
    category["id"]: category["name"]
    for category in coco["categories"]
}

if len(category_ids) != 80:
    raise RuntimeError("Expected 80 COCO categories.")

for class_index, category_id in enumerate(category_ids):
    if yolo.names[class_index] != category_names[category_id]:
        raise RuntimeError(
            f"Class mapping mismatch at model class {class_index}."
        )

original_detections = fp32_detections.clone()
original_detections[:, :4] = scale_boxes(
    pilot_tensor.shape[2:],
    original_detections[:, :4],
    pilot_bgr.shape[:2],
)

pilot_coco_predictions = []

for detection in original_detections:
    x1, y1, x2, y2, score, class_index = detection.tolist()
    if x2 <= x1 or y2 <= y1:
        continue

    pilot_coco_predictions.append({
        "image_id": int(pilot_image_id),
        "category_id": int(category_ids[int(class_index)]),
        "bbox": [x1, y1, x2 - x1, y2 - y1],
        "score": score,
    })

width = pilot_record["width"]
height = pilot_record["height"]

for prediction in pilot_coco_predictions:
    x, y, w, h = prediction["bbox"]
    assert 0 <= x < x + w <= width + 1e-4
    assert 0 <= y < y + h <= height + 1e-4

print("Class-name mapping: PASS")
print("Original image size:", (width, height))
print("COCO prediction records:", len(pilot_coco_predictions))
print(
    "Degenerate boxes excluded:",
    len(original_detections) - len(pilot_coco_predictions),
)
print("First three records:")
for prediction in pilot_coco_predictions[:3]:
    print(prediction)

print("No files saved. No COCOeval executed yet.")

Class-name mapping: PASS
Original image size: (640, 426)
COCO prediction records: 217
Degenerate boxes excluded: 0
First three records:
{'image_id': 8844, 'category_id': 1, 'bbox': [355.90521240234375, 55.456634521484375, 228.306884765625, 230.97705078125], 'score': 0.9164350032806396}
{'image_id': 8844, 'category_id': 52, 'bbox': [367.4266357421875, 244.5008544921875, 268.22039794921875, 149.0430908203125], 'score': 0.8889705538749695}
{'image_id': 8844, 'category_id': 1, 'bbox': [338.24005126953125, 187.68890380859375, 33.22607421875, 61.029296875], 'score': 0.6473013162612915}
No files saved. No COCOeval executed yet.


In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

if not pilot_coco_predictions:
    raise RuntimeError("No predictions available for this evaluation check.")

assert all(
    prediction["image_id"] == pilot_image_id
    for prediction in pilot_coco_predictions
)
assert pilot_image_id in manifest["image_ids"]["development"]
assert pilot_image_id not in manifest["image_ids"]["held_out"]

ground_truth = COCO(
    str(root / "annotations/instances_val2017.json")
)
detections = ground_truth.loadRes(pilot_coco_predictions)

pilot_evaluator = COCOeval(ground_truth, detections, iouType="bbox")
pilot_evaluator.params.imgIds = [int(pilot_image_id)]
pilot_evaluator.params.catIds = category_ids
pilot_evaluator.params.maxDets = [1, 10, 100]

pilot_evaluator.evaluate()
pilot_evaluator.accumulate()
pilot_evaluator.summarize()

assert pilot_evaluator.params.imgIds == [int(pilot_image_id)]

print("\nEvaluated image IDs:", pilot_evaluator.params.imgIds)
print("Evaluation pipeline smoke test completed.")
print("Single-image metrics are not dataset-level performance estimates.")
print("Values of -1 indicate unavailable metrics for this evaluation subset.")
print("No fault injected. No files modified.")

loading annotations into memory...
Done (t=3.35s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.23s).
Accumulating evaluation results...
DONE (t=0.24s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.428
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.708
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.430
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.250
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.385
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.700
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.175
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.450
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets

In [ ]:
import torch
from ultralytics.utils.ops import scale_boxes

annotations = [
    ann for ann in coco["annotations"]
    if ann["image_id"] == pilot_image_id
    and not ann.get("iscrowd", 0)
    and ann["bbox"][2] > 0
    and ann["bbox"][3] > 0
]

if not annotations:
    raise RuntimeError("No valid non-crowd boxes for the geometry check.")

original_boxes = torch.tensor(
    [
        [x, y, x + w, y + h]
        for x, y, w, h in (ann["bbox"] for ann in annotations)
    ],
    dtype=torch.float32,
)

original_h, original_w = pilot_bgr.shape[:2]
input_h, input_w = pilot_tensor.shape[2:]

gain = min(input_h / original_h, input_w / original_w)
resized_w = round(original_w * gain)
resized_h = round(original_h * gain)
left = round((input_w - resized_w) / 2 - 0.1)
top = round((input_h - resized_h) / 2 - 0.1)

input_boxes = original_boxes.clone()
input_boxes[:, [0, 2]] = input_boxes[:, [0, 2]] * gain + left
input_boxes[:, [1, 3]] = input_boxes[:, [1, 3]] * gain + top

recovered_boxes = scale_boxes(
    (input_h, input_w),
    input_boxes.clone(),
    (original_h, original_w),
)

expected_boxes = original_boxes.clone()
expected_boxes[:, [0, 2]].clamp_(0, original_w)
expected_boxes[:, [1, 3]].clamp_(0, original_h)

# Advanced indexing returns a copy; assign clipped values explicitly.
expected_boxes[:, [0, 2]] = original_boxes[:, [0, 2]].clamp(0, original_w)
expected_boxes[:, [1, 3]] = original_boxes[:, [1, 3]].clamp(0, original_h)

max_error = (recovered_boxes - expected_boxes).abs().max().item()

print("Ground-truth boxes checked:", len(annotations))
print("Gain:", gain)
print("Left/top padding:", (left, top))
print("Maximum round-trip coordinate error (pixels):", max_error)

if max_error > 1e-3:
    raise RuntimeError("Coordinate round-trip check failed.")

print("Coordinate round-trip check: PASS")
print("This check covers the current image geometry only.")
print("No inference performed. No files modified.")

Ground-truth boxes checked: 7
Gain: 1.0
Left/top padding: (0, 107)
Maximum round-trip coordinate error (pixels): 3.0517578125e-05
Coordinate round-trip check: PASS
This check covers the current image geometry only.
No inference performed. No files modified.


In [ ]:
import copy
import torch

scenario = next(
    item for item in plan["scenarios"]
    if item["scenario_id"] == "EXT2-001"
)

layer_name = "model." + scenario["layer"]
weight_index = tuple(scenario["weight_index"])
bit = int(scenario["bit"])
scale = float(scenario["scale"])

if not 0 <= bit <= 3 or not scale > 0:
    raise RuntimeError("Invalid fault configuration.")

int4_clean_model = copy.deepcopy(model).cpu().eval()
clean_layer = dict(int4_clean_model.named_modules())[layer_name]

with torch.no_grad():
    q_clean = torch.round(
        clean_layer.weight.detach() / scale
    ).clamp(-7, 7).to(torch.int8)

    before = int(q_clean[weight_index].item())
    flipped_bits = (before & 0xF) ^ (1 << bit)
    after = flipped_bits - 16 if flipped_bits >= 8 else flipped_bits

    if before != int(scenario["q_clean"]):
        raise RuntimeError("Clean integer value differs from historical plan.")
    if after != int(scenario["q_faulty"]):
        raise RuntimeError("Faulty integer value differs from historical plan.")

    clean_layer.weight.copy_(q_clean.float() * scale)

    int4_faulty_model = copy.deepcopy(int4_clean_model)
    faulty_layer = dict(int4_faulty_model.named_modules())[layer_name]

    q_faulty = q_clean.clone()
    q_faulty[weight_index] = after
    faulty_layer.weight.copy_(q_faulty.float() * scale)

clean_state = int4_clean_model.state_dict()
faulty_state = int4_faulty_model.state_dict()
changed_entries = {}

for name, clean_tensor in clean_state.items():
    count = int((clean_tensor != faulty_state[name]).sum().item())
    if count:
        changed_entries[name] = count

expected_key = layer_name + ".weight"
if changed_entries != {expected_key: 1}:
    raise RuntimeError(f"Unexpected state changes: {changed_entries}")

print("Scenario:", scenario["scenario_id"])
print("Layer:", layer_name)
print("Weight index:", weight_index)
print("Bit:", bit)
print("Logical INT4 transition:", before, "->", after)
print("Historical scale:", scale)
print("Changed state entries:", changed_entries)
print("Single-weight isolation check: PASS")
print("Historical scale reused for this pilot only.")
print("No inference performed. No checkpoint file modified.")

Scenario: EXT2-001
Layer: model.1.conv
Weight index: (28, 17, 2, 1)
Bit: 0
Logical INT4 transition: 1 -> 0
Historical scale: 0.06072126116071429
Changed state entries: {'model.1.conv.weight': 1}
Single-weight isolation check: PASS
Historical scale reused for this pilot only.
No inference performed. No checkpoint file modified.


In [ ]:
import time
import torch
from ultralytics.utils.nms import non_max_suppression

pilot_outputs = {}

for name, candidate in (
    ("int4_clean", int4_clean_model),
    ("int4_faulty", int4_faulty_model),
):
    start = time.perf_counter()

    with torch.inference_mode():
        output = candidate(pilot_tensor)
        prediction = output[0] if isinstance(output, (tuple, list)) else output
        raw = prediction.detach().clone()

    forward_seconds = time.perf_counter() - start

    assert tuple(raw.shape) == (1, 84, 8400)
    assert torch.isfinite(raw).all()

    start = time.perf_counter()
    with torch.inference_mode():
        detections = non_max_suppression(
            raw.clone(),
            conf_thres=0.001,
            iou_thres=0.7,
            nc=80,
            max_det=300,
        )[0].detach().cpu().clone()

    assert detections.ndim == 2 and detections.shape[1] == 6
    assert torch.isfinite(detections).all()

    pilot_outputs[name] = {
        "raw": raw,
        "detections": detections,
        "forward_seconds": forward_seconds,
        "nms_seconds": time.perf_counter() - start,
    }

    print(
        f"{name}: detections={len(detections)}, "
        f"forward_seconds={forward_seconds:.4f}"
    )

difference = (
    pilot_outputs["int4_clean"]["raw"]
    - pilot_outputs["int4_faulty"]["raw"]
).abs()

print("\nFault-induced differences relative to the INT4 clean baseline:")
print("Box-coordinate MAE (input pixels):", difference[:, :4, :].mean().item())
print("Class-score MAE:", difference[:, 4:, :].mean().item())
print("Maximum box-coordinate difference:", difference[:, :4, :].max().item())
print("Maximum class-score difference:", difference[:, 4:, :].max().item())
print("These diagnostics do not measure detection accuracy.")
print("No files saved. No held-out images evaluated.")

int4_clean: detections=161, forward_seconds=1.4930
int4_faulty: detections=159, forward_seconds=0.9517

Fault-induced differences relative to the INT4 clean baseline:
Box-coordinate MAE (input pixels): 0.13131612539291382
Class-score MAE: 5.284996973387024e-07
Maximum box-coordinate difference: 24.8311767578125
Maximum class-score difference: 0.03652900457382202
These diagnostics do not measure detection accuracy.
No files saved. No held-out images evaluated.


In [ ]:
import copy
import torch
from pycocotools.cocoeval import COCOeval
from ultralytics.utils.ops import scale_boxes

def to_coco_records(detections):
    restored = detections.detach().cpu().clone()
    restored[:, :4] = scale_boxes(
        pilot_tensor.shape[2:],
        restored[:, :4],
        pilot_bgr.shape[:2],
    )

    records = []
    for row in restored:
        x1, y1, x2, y2, score, class_index = row.tolist()
        if x2 <= x1 or y2 <= y1:
            continue
        records.append({
            "image_id": int(pilot_image_id),
            "category_id": int(category_ids[int(class_index)]),
            "bbox": [x1, y1, x2 - x1, y2 - y1],
            "score": score,
        })
    return records

comparison_records = {
    "fp32_clean": copy.deepcopy(pilot_coco_predictions),
    "int4_clean": to_coco_records(
        pilot_outputs["int4_clean"]["detections"]
    ),
    "int4_faulty": to_coco_records(
        pilot_outputs["int4_faulty"]["detections"]
    ),
}

single_image_metrics = {}

for name, records in comparison_records.items():
    if not records:
        raise RuntimeError(f"Empty predictions require explicit handling: {name}")

    assert all(r["image_id"] == pilot_image_id for r in records)

    results = ground_truth.loadRes(copy.deepcopy(records))
    evaluator = COCOeval(ground_truth, results, iouType="bbox")
    evaluator.params.imgIds = [int(pilot_image_id)]
    evaluator.params.catIds = category_ids
    evaluator.params.maxDets = [1, 10, 100]

    print(f"\n=== {name} ===")
    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()

    single_image_metrics[name] = {
        "AP50_95": float(evaluator.stats[0]),
        "AP50": float(evaluator.stats[1]),
        "AP75": float(evaluator.stats[2]),
        "AR100": float(evaluator.stats[8]),
    }

print("\n=== Single-image comparison ===")
for name, metrics in single_image_metrics.items():
    print(name, metrics)

fp32_ap = single_image_metrics["fp32_clean"]["AP50_95"]
clean_ap = single_image_metrics["int4_clean"]["AP50_95"]
faulty_ap = single_image_metrics["int4_faulty"]["AP50_95"]

print("Quantization AP change (INT4 clean - FP32):", clean_ap - fp32_ap)
print("Fault-induced AP change (faulty - INT4 clean):", faulty_ap - clean_ap)
print("Single-image diagnostics only; no layer ranking or generalization claim.")
print("No files saved. No held-out images evaluated.")

Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!

=== fp32_clean ===
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.02s).
Accumulating evaluation results...
DONE (t=0.03s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.428
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.708
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.430
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.250
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.385
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.700
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.175
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.450
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.738
 Average Recall     (AR) @[ IoU=0.50:0.95 | area

In [ ]:
import hashlib
import json
import platform
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

project = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")
output = (
    project / "code/results/notebook05/"
    "EXT2-001_image8844_cpu_smoke_v1.json"
)

if int(pilot_image_id) != 8844 or scenario["scenario_id"] != "EXT2-001":
    raise RuntimeError("Unexpected image or scenario.")

report = {
    "schema_version": "1.0",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "scope": "single_image_pipeline_smoke_test",
    "image_id": int(pilot_image_id),
    "scenario": scenario,
    "checkpoint_sha256": expected_sha256,
    "split_manifest_sha256": hashlib.sha256(
        (
            project / "code/experiments/notebook05/"
            "coco2017_split_manifest_v1.json"
        ).read_bytes()
    ).hexdigest(),
    "environment": {
        "python": platform.python_version(),
        **{
            name: version(name)
            for name in (
                "torch", "torchvision", "ultralytics",
                "numpy", "pycocotools", "opencv-python", "Pillow"
            )
        },
        "device": "cpu",
    },
    "preprocessing": {
        "input_size": [640, 640],
        "letterbox": {
            "auto": False,
            "scale_fill": False,
            "scaleup": True,
            "stride": 32,
        },
        "color": "RGB",
        "dtype": "float32",
        "normalization": "divide_by_255",
    },
    "evaluation": {
        "confidence_threshold": 0.001,
        "nms_iou_threshold": 0.7,
        "inference_max_det": 300,
        "cocoeval_max_dets": [1, 10, 100],
        "ground_truth": "original_COCO_instances_val2017",
    },
    "metrics": single_image_metrics,
    "predictions": comparison_records,
    "limitations": [
        "One development image and one historical fault scenario.",
        "Not a dataset-level performance estimate or layer ranking.",
        "Logical INT4 weights with FP32 inference.",
        "Historical inference has not been independently reproduced.",
        "Timing is not a steady-state benchmark.",
    ],
}

content = (
    json.dumps(report, indent=2, allow_nan=False) + "\n"
).encode("utf-8")

output.parent.mkdir(parents=True, exist_ok=True)
with output.open("xb") as stream:
    stream.write(content)

assert output.read_bytes() == content
print("Saved:", output)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("No existing results overwritten. No commit or push performed.")

Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook05/EXT2-001_image8844_cpu_smoke_v1.json
SHA256: 39ae5967afb8d76306f4dee12ca6b0a21f8e7b45abb6b4a8043f4a7ff8306c9a
No existing results overwritten. No commit or push performed.


In [ ]:
import copy
import hashlib
import json
import time
from pathlib import Path

import cv2
import numpy as np
import torch
from ultralytics.data.augment import LetterBox
from ultralytics.utils.nms import non_max_suppression
from ultralytics.utils.ops import scale_boxes

project = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")
root = project / "datasets/coco2017"
split_path = (
    project / "code/experiments/notebook05/coco2017_split_manifest_v1.json"
)

split_bytes = split_path.read_bytes()
split_sha = hashlib.sha256(split_bytes).hexdigest()
if split_sha != "143c3273b2bf0cd6393cfe148a74e48fbafb74e770bd3fbdf3f14222355978ee":
    raise RuntimeError("Split manifest changed.")

split_manifest = json.loads(split_bytes)
pilot_ids = split_manifest["image_ids"]["pilot"]

assert len(pilot_ids) == len(set(pilot_ids)) == 50
assert set(pilot_ids) <= set(split_manifest["image_ids"]["development"])
assert not set(pilot_ids) & set(split_manifest["image_ids"]["held_out"])

image_records = {item["id"]: item for item in coco["images"]}
category_mapping = list(category_ids)

def run_cpu_pilot(candidate, output_path, label, fault_scenario=None):
    output_path = Path(output_path)
    if output_path.exists():
        raise FileExistsError("Pilot output already exists; inspect before rerunning.")
    if next(candidate.parameters()).device.type != "cpu":
        raise RuntimeError("This pilot runner expects a CPU model.")

    transform = LetterBox(
        new_shape=(640, 640), auto=False,
        scale_fill=False, scaleup=True, stride=32,
    )
    candidate.eval()
    predictions = []
    image_results = []
    started = time.perf_counter()

    with torch.inference_mode():
        for number, image_id in enumerate(pilot_ids, start=1):
            record = image_records[image_id]
            image = cv2.imread(
                str(root / "images/val2017" / record["file_name"])
            )
            if image is None:
                raise RuntimeError(f"Image decoding failed: {image_id}")
            if image.shape[:2] != (record["height"], record["width"]):
                raise RuntimeError(f"Image dimensions mismatch: {image_id}")

            padded = transform(image=image)
            array = np.ascontiguousarray(
                padded[:, :, ::-1].transpose(2, 0, 1)
            )
            tensor = torch.from_numpy(array).unsqueeze(0).float() / 255.0

            forward_start = time.perf_counter()
            output = candidate(tensor)
            raw = output[0] if isinstance(output, (tuple, list)) else output
            forward_seconds = time.perf_counter() - forward_start

            if tuple(raw.shape) != (1, 84, 8400):
                raise RuntimeError(f"Unexpected prediction shape: {image_id}")
            if not torch.isfinite(raw).all():
                raise RuntimeError(f"Non-finite predictions: {image_id}")

            detections = non_max_suppression(
                raw.clone(), conf_thres=0.001, iou_thres=0.7,
                nc=80, max_det=300,
            )[0].detach().cpu().clone()

            if not torch.isfinite(detections).all():
                raise RuntimeError(f"Non-finite detections: {image_id}")

            detections[:, :4] = scale_boxes(
                tensor.shape[2:], detections[:, :4], image.shape[:2]
            )

            count = 0
            for row in detections:
                x1, y1, x2, y2, score, class_index = row.tolist()
                if x2 <= x1 or y2 <= y1:
                    continue
                if not (
                    class_index == int(class_index)
                    and 0 <= int(class_index) < 80
                    and 0 <= score <= 1
                ):
                    raise RuntimeError(f"Invalid detection: {image_id}")

                predictions.append({
                    "image_id": int(image_id),
                    "category_id": int(category_mapping[int(class_index)]),
                    "bbox": [x1, y1, x2 - x1, y2 - y1],
                    "score": score,
                })
                count += 1

            image_results.append({
                "image_id": int(image_id),
                "prediction_count": count,
                "forward_seconds": forward_seconds,
            })

            if number % 10 == 0:
                print(f"{label}: {number}/50 images completed", flush=True)

    result = {
        "schema_version": "1.0",
        "label": label,
        "device": "cpu",
        "split_manifest_sha256": split_sha,
        "image_ids": list(pilot_ids),
        "scenario": copy.deepcopy(fault_scenario),
        "settings": {
            "imgsz": 640, "batch": 1,
            "letterbox_auto": False, "scale_fill": False,
            "scaleup": True, "stride": 32,
            "conf": 0.001, "nms_iou": 0.7, "max_det": 300,
        },
        "predictions": predictions,
        "per_image": image_results,
        "wall_seconds": time.perf_counter() - started,
        "scope": "development_pilot_not_final_evaluation",
    }

    content = (json.dumps(result, indent=2, allow_nan=False) + "\n").encode()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("xb") as stream:
        stream.write(content)

    print("Saved:", output_path)
    print("SHA256:", hashlib.sha256(content).hexdigest())
    print("Prediction count:", len(predictions))
    print("Wall time (seconds):", round(result["wall_seconds"], 2))
    print("Timing includes image loading and processing; not a model benchmark.")
    return result

fp32_pilot = run_cpu_pilot(
    model,
    project / "code/results/notebook05/pilot50_fp32_cpu_predictions_v1.json",
    "fp32_clean",
)

fp32_clean: 10/50 images completed
fp32_clean: 20/50 images completed
fp32_clean: 30/50 images completed
fp32_clean: 40/50 images completed
fp32_clean: 50/50 images completed
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook05/pilot50_fp32_cpu_predictions_v1.json
SHA256: 4bead0c6193ad746bbf7d0f1a46887d11dff077151a975b76f01a88b937d6cf3
Prediction count: 4731
Wall time (seconds): 48.28
Timing includes image loading and processing; not a model benchmark.


In [ ]:
int4_clean_pilot = run_cpu_pilot(
    int4_clean_model,
    project / "code/results/notebook05/pilot50_int4_clean_cpu_predictions_v1.json",
    "int4_clean",
)

int4_clean: 10/50 images completed
int4_clean: 20/50 images completed
int4_clean: 30/50 images completed
int4_clean: 40/50 images completed
int4_clean: 50/50 images completed
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook05/pilot50_int4_clean_cpu_predictions_v1.json
SHA256: 048a137908dc68572a5c572f1d7a55d75e9e30cfa028abeac408c23a12e9e4a3
Prediction count: 5058
Wall time (seconds): 32.6
Timing includes image loading and processing; not a model benchmark.


In [ ]:
int4_faulty_pilot = run_cpu_pilot(
    int4_faulty_model,
    project / "code/results/notebook05/pilot50_EXT2-001_cpu_predictions_v1.json",
    "int4_faulty_EXT2-001",
    fault_scenario=scenario,
)

int4_faulty_EXT2-001: 10/50 images completed
int4_faulty_EXT2-001: 20/50 images completed
int4_faulty_EXT2-001: 30/50 images completed
int4_faulty_EXT2-001: 40/50 images completed
int4_faulty_EXT2-001: 50/50 images completed
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook05/pilot50_EXT2-001_cpu_predictions_v1.json
SHA256: 2ea33fdee01f82e8983ada66eec312717b174b00222cabe2ffe881764b912201
Prediction count: 5047
Wall time (seconds): 34.34
Timing includes image loading and processing; not a model benchmark.


In [ ]:
import copy
import hashlib
import json
from pathlib import Path

import numpy as np
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

project = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")
folder = project / "code/results/notebook05"
output_path = folder / "pilot50_EXT2-001_cpu_cocoeval_v1.json"

if output_path.exists():
    raise FileExistsError("Evaluation report already exists; inspect before rerunning.")

filenames = {
    "fp32_clean": "pilot50_fp32_cpu_predictions_v1.json",
    "int4_clean": "pilot50_int4_clean_cpu_predictions_v1.json",
    "int4_faulty": "pilot50_EXT2-001_cpu_predictions_v1.json",
}

split_path = (
    project / "code/experiments/notebook05/coco2017_split_manifest_v1.json"
)
split_bytes = split_path.read_bytes()
split_hash = hashlib.sha256(split_bytes).hexdigest()
if split_hash != "143c3273b2bf0cd6393cfe148a74e48fbafb74e770bd3fbdf3f14222355978ee":
    raise RuntimeError("Split manifest changed.")

split = json.loads(split_bytes)
evaluation_ids = split["image_ids"]["pilot"]

annotation_path = (
    project / "datasets/coco2017/annotations/instances_val2017.json"
)
if hashlib.sha256(annotation_path.read_bytes()).hexdigest() != split["annotation_sha256"]:
    raise RuntimeError("Ground-truth annotation file changed.")

gt = COCO(str(annotation_path))
runs = {}
source_hashes = {}

for name, filename in filenames.items():
    content = (folder / filename).read_bytes()
    source_hashes[name] = hashlib.sha256(content).hexdigest()
    runs[name] = json.loads(content)
    run = runs[name]

    if run["image_ids"] != evaluation_ids:
        raise RuntimeError(f"Image list mismatch: {name}")
    if run["split_manifest_sha256"] != split_hash:
        raise RuntimeError(f"Split provenance mismatch: {name}")
    if [item["image_id"] for item in run["per_image"]] != evaluation_ids:
        raise RuntimeError(f"Incomplete image execution record: {name}")
    if any(p["image_id"] not in evaluation_ids for p in run["predictions"]):
        raise RuntimeError(f"Unexpected prediction image ID: {name}")

reference_settings = runs["fp32_clean"]["settings"]
if any(run["settings"] != reference_settings for run in runs.values()):
    raise RuntimeError("Inference settings differ across runs.")

metric_names = [
    "AP50_95", "AP50", "AP75", "AP_small", "AP_medium", "AP_large",
    "AR1", "AR10", "AR100", "AR_small", "AR_medium", "AR_large",
]
metrics = {}

for name, run in runs.items():
    if not run["predictions"]:
        raise RuntimeError(f"Empty predictions need explicit handling: {name}")

    dt = gt.loadRes(copy.deepcopy(run["predictions"]))
    evaluator = COCOeval(gt, dt, iouType="bbox")
    evaluator.params.imgIds = list(evaluation_ids)
    evaluator.params.catIds = sorted(gt.getCatIds())
    evaluator.params.maxDets = [1, 10, 100]

    print(f"\n=== {name} ===")
    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()

    if not np.isfinite(evaluator.stats).all():
        raise RuntimeError(f"Non-finite evaluation metrics: {name}")
    metrics[name] = dict(zip(metric_names, map(float, evaluator.stats)))

quantization_delta = (
    metrics["int4_clean"]["AP50_95"] - metrics["fp32_clean"]["AP50_95"]
)
fault_delta = (
    metrics["int4_faulty"]["AP50_95"] - metrics["int4_clean"]["AP50_95"]
)

report = {
    "scope": "50_image_development_pilot_one_fault_scenario",
    "source_prediction_sha256": source_hashes,
    "split_manifest_sha256": split_hash,
    "image_ids": evaluation_ids,
    "settings": reference_settings,
    "cocoeval_max_dets": [1, 10, 100],
    "metrics": metrics,
    "AP50_95_changes": {
        "int4_clean_minus_fp32": quantization_delta,
        "faulty_minus_int4_clean": fault_delta,
    },
    "limitations": [
        "One historical fault scenario; no layer ranking.",
        "Development pilot only; held-out images were not evaluated.",
        "No uncertainty estimate or statistical significance claim.",
        "Subset AP is computed jointly, not averaged from per-image AP.",
    ],
}

content = (json.dumps(report, indent=2, allow_nan=False) + "\n").encode()
with output_path.open("xb") as stream:
    stream.write(content)

print("\n=== Pilot comparison ===")
for name in metrics:
    print(name, {key: metrics[name][key] for key in ("AP50_95", "AP50", "AR100")})
print("Quantization AP change:", quantization_delta)
print("Fault-induced AP change:", fault_delta)
print("Saved:", output_path)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("No new inference, commit or push performed.")

loading annotations into memory...
Done (t=2.33s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.03s)
creating index...
index created!

=== fp32_clean ===
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=2.76s).
Accumulating evaluation results...
DONE (t=0.29s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.607
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.751
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.665
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.372
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.575
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.780
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.491
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.683
 Average Recall     (AR) @[ IoU=0.50:0.95 | a

In [ ]:
import copy
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

control_gt = COCO()
control_gt.dataset = {
    "info": {"description": "Synthetic evaluation control"},
    "images": [{"id": 1, "width": 100, "height": 100}],
    "categories": [{"id": 1, "name": "control_object"}],
    "annotations": [{
        "id": 1,
        "image_id": 1,
        "category_id": 1,
        "bbox": [10, 20, 30, 40],
        "area": 1200,
        "iscrowd": 0,
    }],
}
control_gt.createIndex()

perfect_predictions = [{
    "image_id": 1,
    "category_id": 1,
    "bbox": [10, 20, 30, 40],
    "score": 1.0,
}]

control_dt = control_gt.loadRes(copy.deepcopy(perfect_predictions))
control_eval = COCOeval(control_gt, control_dt, iouType="bbox")
control_eval.params.imgIds = [1]
control_eval.params.catIds = [1]
control_eval.params.maxDets = [1, 10, 100]
control_eval.evaluate()
control_eval.accumulate()
control_eval.summarize()

ap = float(control_eval.stats[0])
if abs(ap - 1.0) > 1e-9:
    raise RuntimeError(f"Perfect-prediction control failed: AP={ap}")

print("Perfect-prediction evaluation control: PASS")
print("This validates the synthetic evaluator control, not the entire pipeline.")
print("No model inference. No files modified.")

creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.00s).
Accumulating evaluation results...
DONE (t=0.02s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 1.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 1.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = -1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 1.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 1.000
 Average Recall     (AR) @[ IoU=0

In [ ]:
import torch

modules = dict(model.named_modules())

print("Layer | historical scale | FP32 tensor scale | changed integer weights")

for layer_name in plan["selected_layers"]:
    layer_scenarios = [
        item for item in plan["scenarios"]
        if item["layer"] == layer_name
    ]
    historical_scales = {
        float(item["scale"]) for item in layer_scenarios
    }
    if len(historical_scales) != 1:
        raise RuntimeError(f"Inconsistent historical scales: {layer_name}")

    historical_scale = historical_scales.pop()
    weight = modules["model." + layer_name].weight.detach().cpu()

    if weight.dtype != torch.float32:
        raise RuntimeError(f"Expected FP32 weights: {layer_name}")

    fp32_scale = weight.abs().max() / 7
    if not torch.isfinite(fp32_scale) or fp32_scale.item() <= 0:
        raise RuntimeError(f"Invalid scale: {layer_name}")

    historical_q = torch.round(
        weight / historical_scale
    ).clamp(-7, 7).to(torch.int8)

    fp32_q = torch.round(
        weight / fp32_scale
    ).clamp(-7, 7).to(torch.int8)

    changed = int((historical_q != fp32_q).sum().item())

    print(
        f"{layer_name} | {historical_scale:.17g} | "
        f"{fp32_scale.item():.17g} | {changed}"
    )

print("\nArithmetic inspection only.")
print("No model weights changed. No inference or files written.")

Layer | historical scale | FP32 tensor scale | changed integer weights
1.conv | 0.060721261160714288 | 0.060721259564161301 | 0
4.cv1.conv | 0.043247767857142856 | 0.0432477667927742 | 0
4.m.1.cv2.conv | 0.022513253348214284 | 0.02251325361430645 | 0
6.m.1.cv1.conv | 0.019705636160714284 | 0.01970563642680645 | 0
9.cv1.conv | 0.030133928571428572 | 0.0301339291036129 | 0
9.cv2.conv | 0.0260009765625 | 0.0260009765625 | 0
15.cv2.conv | 0.027326311383928572 | 0.0273263119161129 | 0
18.m.0.cv2.conv | 0.021902901785714284 | 0.02190290205180645 | 0
21.cv2.conv | 0.018205915178571428 | 0.0182059146463871 | 0
22.cv2.2.0.conv | 0.014569963727678572 | 0.014569963328540325 | 0
22.cv3.0.2 | 0.0325927734375 | 0.0325927734375 | 0
22.cv3.2.1.conv | 0.017490931919642856 | 0.01749093271791935 | 0

Arithmetic inspection only.
No model weights changed. No inference or files written.


In [ ]:
import json
from pathlib import Path

folder = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook05"
)

filenames = [
    "pilot50_fp32_cpu_predictions_v1.json",
    "pilot50_int4_clean_cpu_predictions_v1.json",
    "pilot50_EXT2-001_cpu_predictions_v1.json",
]

seconds_per_image = []
for filename in filenames:
    run = json.loads((folder / filename).read_text(encoding="utf-8"))
    seconds_per_image.append(run["wall_seconds"] / len(run["image_ids"]))

low = min(seconds_per_image)
high = max(seconds_per_image)

print("Observed CPU wall seconds/image:", [round(v, 3) for v in seconds_per_image])
print("\nReplicates per bit | Fault scenarios | Total image forwards | CPU hours")

for replicates_per_bit in (2, 5, 10):
    fault_scenarios = 12 * 4 * replicates_per_bit
    # One FP32 baseline, 12 layer-specific clean baselines, and faulty models.
    image_forwards = (1 + 12 + fault_scenarios) * 1000
    low_hours = image_forwards * low / 3600
    high_hours = image_forwards * high / 3600

    print(
        f"{replicates_per_bit:18d} | {fault_scenarios:15d} | "
        f"{image_forwards:20d} | {low_hours:.1f}–{high_hours:.1f}"
    )

print("\nRough extrapolation, not a confidence interval or runtime guarantee.")
print("Excludes COCOeval, model preparation, saving and interruption overhead.")
print("Replicate count is not selected by this calculation.")
print("No inference performed. No files modified.")

Observed CPU wall seconds/image: [0.966, 0.652, 0.687]

Replicates per bit | Fault scenarios | Total image forwards | CPU hours
                 2 |              96 |               109000 | 19.7–29.2
                 5 |             240 |               253000 | 45.8–67.9
                10 |             480 |               493000 | 89.3–132.2

Rough extrapolation, not a confidence interval or runtime guarantee.
Excludes COCOeval, model preparation, saving and interruption overhead.
Replicate count is not selected by this calculation.
No inference performed. No files modified.


In [ ]:
from pathlib import Path
import hashlib
import json

if not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")

expected = {
    "experiments/notebook05/coco2017_split_manifest_v1.json":
        "143c3273b2bf0cd6393cfe148a74e48fbafb74e770bd3fbdf3f14222355978ee",
    "results/notebook05/pilot50_EXT2-001_cpu_cocoeval_v1.json":
        "6de9c5de39868d1d721be376e77d3d460be498dfcb3304e4d1b8e94f3f322353",
}

for relative_path, expected_hash in expected.items():
    path = repo / relative_path
    content = path.read_bytes()
    actual_hash = hashlib.sha256(content).hexdigest()

    if actual_hash != expected_hash:
        raise RuntimeError(f"File integrity mismatch: {relative_path}")

    json.loads(content)
    print("PASS:", relative_path)

print("Saved split and pilot report integrity verified.")
print("No inference performed. No files modified.")

Mounted at /content/drive
PASS: experiments/notebook05/coco2017_split_manifest_v1.json
PASS: results/notebook05/pilot50_EXT2-001_cpu_cocoeval_v1.json
Saved split and pilot report integrity verified.
No inference performed. No files modified.


In [ ]:
import sys
from importlib.metadata import version, PackageNotFoundError

import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GiB):",
          round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

for package in ("torchvision", "ultralytics", "numpy", "pycocotools"):
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("No packages installed. No inference performed.")

Python: 3.13.18
PyTorch: 2.14.1+cpu
CUDA available: False
torchvision: 0.29.1+cpu
ultralytics: NOT INSTALLED
numpy: 2.4.6
pycocotools: 2.0.11
No packages installed. No inference performed.


In [ ]:
import shutil
import subprocess

command = shutil.which("nvidia-smi")

if command is None:
    print("nvidia-smi: NOT FOUND")
else:
    result = subprocess.run(
        [
            command,
            "--query-gpu=name,driver_version,memory.total",
            "--format=csv,noheader",
        ],
        capture_output=True,
        text=True,
        timeout=30,
    )
    print("Exit code:", result.returncode)
    print(result.stdout.strip() or result.stderr.strip() or "(no output)")

print("No packages installed. No files modified.")

Exit code: 0
Tesla T4, 580.82.07, 15360 MiB
No packages installed. No files modified.


In [ ]:
import sys
from importlib.metadata import version, PackageNotFoundError
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    test = torch.ones(1, device="cuda")
    torch.cuda.synchronize()
    print("CUDA tensor allocation: PASS")
    del test

for package in ("torchvision", "ultralytics", "pycocotools"):
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("No packages installed. No model inference performed.")

Python: 3.13.16
PyTorch: 2.14.1+cu130
PyTorch CUDA build: 13.0
CUDA available: True
GPU: Tesla T4
CUDA tensor allocation: PASS
torchvision: 0.29.1+cu130
ultralytics: NOT INSTALLED
pycocotools: 2.0.11
No packages installed. No model inference performed.


In [ ]:
import subprocess
import sys
from importlib.metadata import version

packages = [
    "ultralytics==8.4.189",
    "nvidia-ml-py>=12.0.0",
    "ultralytics-thop>=2.2.3",
    "ultralytics-platform>=0.1.45",
    "jedi>=0.16",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", *packages],
    check=True,
)

print("--- Installed versions ---")
for package in (
    "ultralytics", "nvidia-ml-py", "ultralytics-thop",
    "ultralytics-platform", "jedi",
):
    print(f"{package}: {version(package)}")

result = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True,
)

print("\n--- Dependency check ---")
print("Exit code:", result.returncode)
print(result.stdout.strip() or result.stderr.strip() or "(no output)")
print("\nNo model loaded. No inference performed.")

--- Installed versions ---
ultralytics: 8.4.189
nvidia-ml-py: 13.615.71
ultralytics-thop: 2.2.5
ultralytics-platform: 0.1.108
jedi: 0.20.1

--- Dependency check ---
Exit code: 0
No broken requirements found.

No model loaded. No inference performed.


In [ ]:
from pathlib import Path
import hashlib
import json

if not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

project = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")

checks = {
    "checkpoint": (
        project / "models/yolov8s.pt",
        "1f47a78bf100391c2a140b7ac73a1caae18c32779be7d310658112f7ac9aa78a",
    ),
    "split_manifest": (
        project / "code/experiments/notebook05/coco2017_split_manifest_v1.json",
        "143c3273b2bf0cd6393cfe148a74e48fbafb74e770bd3fbdf3f14222355978ee",
    ),
}

for label, (path, expected) in checks.items():
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    if digest.hexdigest() != expected:
        raise RuntimeError(f"Integrity mismatch: {label}")
    print(f"{label}: PASS")

split_manifest = json.loads(checks["split_manifest"][0].read_bytes())
annotation_path = (
    project / "datasets/coco2017/annotations/instances_val2017.json"
)
annotation_bytes = annotation_path.read_bytes()

if hashlib.sha256(annotation_bytes).hexdigest() != split_manifest["annotation_sha256"]:
    raise RuntimeError("Annotation integrity mismatch.")

coco = json.loads(annotation_bytes)
pilot_ids = split_manifest["image_ids"]["pilot"]

assert len(pilot_ids) == len(set(pilot_ids)) == 50
assert set(pilot_ids) <= set(split_manifest["image_ids"]["development"])
assert not set(pilot_ids) & set(split_manifest["image_ids"]["held_out"])

print("Annotations: PASS")
print("Pilot image count:", len(pilot_ids))
print("Pilot/development separation from held-out: PASS")
print("No files modified. No model loaded or inference performed.")

Mounted at /content/drive
checkpoint: PASS
split_manifest: PASS
Annotations: PASS
Pilot image count: 50
Pilot/development separation from held-out: PASS
No files modified. No model loaded or inference performed.


In [ ]:
import os
import random
import numpy as np
import torch
from ultralytics import YOLO

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

seed = 20261109
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.use_deterministic_algorithms(True)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable.")

gpu_yolo = YOLO(str(project / "models/yolov8s.pt"))
gpu_model = gpu_yolo.model.float().to("cuda:0").eval()

if len(gpu_yolo.names) != 80:
    raise RuntimeError("Unexpected class count.")
if any(parameter.dtype != torch.float32 for parameter in gpu_model.parameters()):
    raise RuntimeError("Expected FP32 model parameters.")

print("GPU:", torch.cuda.get_device_name(0))
print("Model:", type(gpu_model).__name__)
print("Parameters:", sum(p.numel() for p in gpu_model.parameters()))
print("Device:", next(gpu_model.parameters()).device)
print("Parameter dtype:", next(gpu_model.parameters()).dtype)
print("Deterministic algorithms:", torch.are_deterministic_algorithms_enabled())
print("cuDNN benchmark:", torch.backends.cudnn.benchmark)
print("Matmul TF32:", torch.backends.cuda.matmul.allow_tf32)
print("cuDNN TF32:", torch.backends.cudnn.allow_tf32)
print("No inference performed. Checkpoint file unchanged.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
GPU: Tesla T4
Model: DetectionModel
Parameters: 11166560
Device: cuda:0
Parameter dtype: torch.float32
Deterministic algorithms: True
cuDNN benchmark: False
Matmul TF32: False
cuDNN TF32: False
No inference performed. Checkpoint file unchanged.


In [ ]:
import time
import cv2
import numpy as np
import torch
from ultralytics.data.augment import LetterBox

image_records = {item["id"]: item for item in coco["images"]}
gpu_image_id = pilot_ids[0]
record = image_records[gpu_image_id]

image_path = (
    project / "datasets/coco2017/images/val2017" / record["file_name"]
)
image = cv2.imread(str(image_path))
if image is None:
    raise RuntimeError("Image decoding failed.")
if image.shape[:2] != (record["height"], record["width"]):
    raise RuntimeError("Image dimensions mismatch.")

transform = LetterBox(
    new_shape=(640, 640),
    auto=False,
    scale_fill=False,
    scaleup=True,
    stride=32,
)
padded = transform(image=image)
array = np.ascontiguousarray(padded[:, :, ::-1].transpose(2, 0, 1))
gpu_input_cpu = torch.from_numpy(array).unsqueeze(0).float() / 255.0
gpu_input = gpu_input_cpu.to("cuda:0")

def extract_raw(output):
    prediction = output[0] if isinstance(output, (tuple, list)) else output
    if tuple(prediction.shape) != (1, 84, 8400):
        raise RuntimeError("Unexpected output shape.")
    if not torch.isfinite(prediction).all():
        raise RuntimeError("Non-finite predictions.")
    return prediction.detach().clone()

with torch.inference_mode():
    torch.cuda.synchronize()
    start = time.perf_counter()
    gpu_raw_first = extract_raw(gpu_model(gpu_input))
    torch.cuda.synchronize()
    first_seconds = time.perf_counter() - start

    start = time.perf_counter()
    gpu_raw_second = extract_raw(gpu_model(gpu_input))
    torch.cuda.synchronize()
    second_seconds = time.perf_counter() - start

difference = (gpu_raw_first - gpu_raw_second).abs()

print("Image ID:", gpu_image_id)
print("Input shape:", tuple(gpu_input.shape))
print("Output shape:", tuple(gpu_raw_first.shape))
print("Repeated outputs exactly equal:", torch.equal(gpu_raw_first, gpu_raw_second))
print("Maximum repeated-output difference:", difference.max().item())
print("First forward/check time (seconds):", round(first_seconds, 4))
print("Second forward/check time (seconds):", round(second_seconds, 4))
print("These timings include output checks and are not a benchmark.")
print("No fault injected. No files modified.")

Image ID: 8844
Input shape: (1, 3, 640, 640)
Output shape: (1, 84, 8400)
Repeated outputs exactly equal: True
Maximum repeated-output difference: 0.0
First forward/check time (seconds): 1.5962
Second forward/check time (seconds): 0.0173
These timings include output checks and are not a benchmark.
No fault injected. No files modified.


In [ ]:
import copy
import torch

cpu_reference_model = copy.deepcopy(gpu_model).cpu().float().eval()

with torch.inference_mode():
    cpu_raw_reference = extract_raw(
        cpu_reference_model(gpu_input_cpu)
    ).cpu()

gpu_raw_reference = gpu_raw_first.detach().cpu()
difference = (cpu_raw_reference - gpu_raw_reference).abs()

print("Image ID:", gpu_image_id)
print("Exactly equal:", torch.equal(cpu_raw_reference, gpu_raw_reference))
print("Box-coordinate MAE (input pixels):",
      difference[:, :4, :].mean().item())
print("Maximum box-coordinate difference:",
      difference[:, :4, :].max().item())
print("Class-score MAE:",
      difference[:, 4:, :].mean().item())
print("Maximum class-score difference:",
      difference[:, 4:, :].max().item())
print("Numerical differences only; detection-level agreement is not checked yet.")

del cpu_reference_model
print("No fault injected. No files modified.")

Image ID: 8844
Exactly equal: False
Box-coordinate MAE (input pixels): 2.054092510661576e-05
Maximum box-coordinate difference: 0.0012664794921875
Class-score MAE: 5.5597297599074835e-11
Maximum class-score difference: 9.238719940185547e-07
Numerical differences only; detection-level agreement is not checked yet.
No fault injected. No files modified.


In [ ]:
import torch
from ultralytics.utils.nms import non_max_suppression

def cpu_nms(raw):
    with torch.inference_mode():
        return non_max_suppression(
            raw.detach().cpu().clone(),
            conf_thres=0.001,
            iou_thres=0.7,
            nc=80,
            max_det=300,
        )[0].detach().clone()

cpu_reference_det = cpu_nms(cpu_raw_reference)
gpu_reference_det = cpu_nms(gpu_raw_reference)

print("CPU-output detections:", len(cpu_reference_det))
print("GPU-output detections:", len(gpu_reference_det))

same_shape = cpu_reference_det.shape == gpu_reference_det.shape
print("Detection shapes equal:", same_shape)

if same_shape and len(cpu_reference_det):
    same_classes = torch.equal(
        cpu_reference_det[:, 5], gpu_reference_det[:, 5]
    )
    print("Class sequence equal:", same_classes)
    print(
        "Maximum row-wise box difference (pixels):",
        (cpu_reference_det[:, :4] - gpu_reference_det[:, :4])
        .abs().max().item(),
    )
    print(
        "Maximum row-wise confidence difference:",
        (cpu_reference_det[:, 4] - gpu_reference_det[:, 4])
        .abs().max().item(),
    )
    print("Row-wise comparison is diagnostic, not geometric matching.")
elif same_shape:
    print("Both detection sets are empty.")
else:
    print("Detection counts differ; row-wise comparison was skipped.")

print("NMS ran on CPU for both prediction tensors.")
print("Single-image check only. No fault injected. No files modified.")

CPU-output detections: 217
GPU-output detections: 217
Detection shapes equal: True
Class sequence equal: True
Maximum row-wise box difference (pixels): 0.0009765625
Maximum row-wise confidence difference: 6.407499313354492e-07
Row-wise comparison is diagnostic, not geometric matching.
NMS ran on CPU for both prediction tensors.
Single-image check only. No fault injected. No files modified.


In [ ]:
import hashlib
import json
import time
from importlib.metadata import version
import platform

import cv2
import numpy as np
import torch
from ultralytics.data.augment import LetterBox
from ultralytics.utils.ops import scale_boxes

output_path = (
    project / "code/results/notebook05/"
    "pilot50_fp32_gpu_cpuNMS_predictions_v1.json"
)
if output_path.exists():
    raise FileExistsError("Output already exists. Inspect before rerunning.")

assert len(pilot_ids) == len(set(pilot_ids)) == 50
assert set(pilot_ids) <= set(split_manifest["image_ids"]["development"])
assert not set(pilot_ids) & set(split_manifest["image_ids"]["held_out"])

category_ids = sorted(item["id"] for item in coco["categories"])
category_names = {item["id"]: item["name"] for item in coco["categories"]}
assert all(
    gpu_yolo.names[i] == category_names[cid]
    for i, cid in enumerate(category_ids)
)

transform = LetterBox(
    new_shape=(640, 640), auto=False,
    scale_fill=False, scaleup=True, stride=32,
)
predictions = []
per_image = []
started = time.perf_counter()

with torch.inference_mode():
    for number, image_id in enumerate(pilot_ids, start=1):
        record = image_records[image_id]
        path = (
            project / "datasets/coco2017/images/val2017"
            / record["file_name"]
        )
        image = cv2.imread(str(path))
        if image is None or image.shape[:2] != (
            record["height"], record["width"]
        ):
            raise RuntimeError(f"Image read or dimension failure: {image_id}")

        padded = transform(image=image)
        array = np.ascontiguousarray(
            padded[:, :, ::-1].transpose(2, 0, 1)
        )
        tensor = (
            torch.from_numpy(array).unsqueeze(0)
            .to(device="cuda:0", dtype=torch.float32) / 255.0
        )

        raw = extract_raw(gpu_model(tensor))
        detections = cpu_nms(raw)
        detections[:, :4] = scale_boxes(
            tensor.shape[2:], detections[:, :4], image.shape[:2]
        )
        if not torch.isfinite(detections).all():
            raise RuntimeError(f"Non-finite detections: {image_id}")

        count = 0
        for row in detections:
            x1, y1, x2, y2, score, cls = row.tolist()
            if x2 <= x1 or y2 <= y1:
                continue
            if not (cls == int(cls) and 0 <= cls < 80 and 0 <= score <= 1):
                raise RuntimeError(f"Invalid detection: {image_id}")
            predictions.append({
                "image_id": int(image_id),
                "category_id": int(category_ids[int(cls)]),
                "bbox": [x1, y1, x2 - x1, y2 - y1],
                "score": score,
            })
            count += 1

        per_image.append({
            "image_id": int(image_id), "prediction_count": count
        })
        if number % 10 == 0:
            print(f"GPU FP32: {number}/50 completed", flush=True)

elapsed = time.perf_counter() - started

report = {
    "scope": "development_pilot",
    "label": "fp32_clean_gpu_cpuNMS",
    "image_ids": list(pilot_ids),
    "checkpoint_sha256": (
        "1f47a78bf100391c2a140b7ac73a1caae18c32779be7d310658112f7ac9aa78a"
    ),
    "split_manifest_sha256": (
        "143c3273b2bf0cd6393cfe148a74e48fbafb74e770bd3fbdf3f14222355978ee"
    ),
    "environment": {
        "python": platform.python_version(),
        **{p: version(p) for p in (
            "torch", "torchvision", "ultralytics", "numpy",
            "pycocotools", "opencv-python", "Pillow"
        )},
        "gpu": torch.cuda.get_device_name(0),
        "cuda_build": torch.version.cuda,
    },
    "settings": {
        "imgsz": 640, "batch": 1, "dtype": "float32",
        "inference_device": "cuda:0", "nms_device": "cpu",
        "conf": 0.001, "nms_iou": 0.7, "max_det": 300,
        "letterbox_auto": False, "scale_fill": False,
        "scaleup": True, "stride": 32,
        "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
        "cudnn_benchmark": torch.backends.cudnn.benchmark,
        "matmul_tf32": torch.backends.cuda.matmul.allow_tf32,
        "cudnn_tf32": torch.backends.cudnn.allow_tf32,
    },
    "predictions": predictions,
    "per_image": per_image,
    "wall_seconds": elapsed,
}
content = (json.dumps(report, indent=2, allow_nan=False) + "\n").encode()
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("xb") as stream:
    stream.write(content)

print("Saved:", output_path)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("Predictions:", len(predictions))
print("Wall time (seconds):", round(elapsed, 2))
print("No fault injected. No held-out images evaluated.")

GPU FP32: 10/50 completed
GPU FP32: 20/50 completed
GPU FP32: 30/50 completed
GPU FP32: 40/50 completed
GPU FP32: 50/50 completed
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook05/pilot50_fp32_gpu_cpuNMS_predictions_v1.json
SHA256: 637e7f2642400376c510220b03d68324e32bd9f897aaf0b8e8bf43861452c22e
Predictions: 4731
Wall time (seconds): 19.18
No fault injected. No held-out images evaluated.


In [ ]:
import copy
import hashlib
import json
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

folder = project / "code/results/notebook05"
gpu_path = folder / "pilot50_fp32_gpu_cpuNMS_predictions_v1.json"
cpu_path = folder / "pilot50_EXT2-001_cpu_cocoeval_v1.json"
output = folder / "pilot50_fp32_cpu_gpu_comparison_v1.json"

if output.exists():
    raise FileExistsError("Comparison report already exists.")

def read_verified(path, expected_hash):
    content = path.read_bytes()
    if hashlib.sha256(content).hexdigest() != expected_hash:
        raise RuntimeError(f"Integrity mismatch: {path.name}")
    return json.loads(content)

gpu_run = read_verified(
    gpu_path,
    "637e7f2642400376c510220b03d68324e32bd9f897aaf0b8e8bf43861452c22e",
)
cpu_report = read_verified(
    cpu_path,
    "6de9c5de39868d1d721be376e77d3d460be498dfcb3304e4d1b8e94f3f322353",
)

if gpu_run["image_ids"] != cpu_report["image_ids"]:
    raise RuntimeError("CPU and GPU evaluation image lists differ.")
if gpu_run["image_ids"] != split_manifest["image_ids"]["pilot"]:
    raise RuntimeError("Unexpected pilot image list.")
if any(
    item["image_id"] not in gpu_run["image_ids"]
    for item in gpu_run["predictions"]
):
    raise RuntimeError("Unexpected prediction image ID.")

annotation_path = (
    project / "datasets/coco2017/annotations/instances_val2017.json"
)
if hashlib.sha256(annotation_path.read_bytes()).hexdigest() != (
    split_manifest["annotation_sha256"]
):
    raise RuntimeError("Ground-truth file changed.")

gt = COCO(str(annotation_path))
dt = gt.loadRes(copy.deepcopy(gpu_run["predictions"]))
evaluation = COCOeval(gt, dt, iouType="bbox")
evaluation.params.imgIds = list(gpu_run["image_ids"])
evaluation.params.catIds = sorted(gt.getCatIds())
evaluation.params.maxDets = [1, 10, 100]
evaluation.evaluate()
evaluation.accumulate()
evaluation.summarize()

indices = {"AP50_95": 0, "AP50": 1, "AP75": 2, "AR100": 8}
gpu_metrics = {
    name: float(evaluation.stats[index])
    for name, index in indices.items()
}
cpu_metrics = {
    name: cpu_report["metrics"]["fp32_clean"][name]
    for name in indices
}
deltas = {
    name: gpu_metrics[name] - cpu_metrics[name]
    for name in indices
}

comparison = {
    "scope": "50_image_development_pilot_CPU_vs_GPU_FP32",
    "cpu_report": cpu_path.name,
    "gpu_predictions": gpu_path.name,
    "image_ids": gpu_run["image_ids"],
    "cpu_metrics": cpu_metrics,
    "gpu_metrics": gpu_metrics,
    "gpu_minus_cpu": deltas,
    "limitations": [
        "No equivalence tolerance was predefined.",
        "This comparison does not establish agreement on other images or faulty models.",
        "Both runs use CPU NMS.",
    ],
}
content = (
    json.dumps(comparison, indent=2, allow_nan=False) + "\n"
).encode()
with output.open("xb") as stream:
    stream.write(content)

print("\nCPU:", cpu_metrics)
print("GPU:", gpu_metrics)
print("GPU minus CPU:", deltas)
print("Saved:", output)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("No new inference. No held-out images evaluated.")

loading annotations into memory...
Done (t=0.88s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.31s).
Accumulating evaluation results...
DONE (t=0.25s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.607
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.751
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.665
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.372
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.575
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.780
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.491
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.683
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets

In [ ]:
import copy
import json
import torch

plan_path = (
    project / "code/results/fault_injection/"
    "int4_extended_layer_sensitivity_plan_v2.json"
)
historical_plan = json.loads(plan_path.read_text(encoding="utf-8"))
gpu_scenario = next(
    item for item in historical_plan["scenarios"]
    if item["scenario_id"] == "EXT2-001"
)

layer_key = "model." + gpu_scenario["layer"]
weight_index = tuple(gpu_scenario["weight_index"])
scale = float(gpu_scenario["scale"])
bit = int(gpu_scenario["bit"])

gpu_int4_clean_model = copy.deepcopy(gpu_model).cpu().float().eval()
clean_layer = dict(gpu_int4_clean_model.named_modules())[layer_key]

with torch.no_grad():
    q_clean = torch.round(
        clean_layer.weight.detach() / scale
    ).clamp(-7, 7).to(torch.int8)

    before = int(q_clean[weight_index].item())
    bits_after = (before & 0xF) ^ (1 << bit)
    after = bits_after - 16 if bits_after >= 8 else bits_after

    if before != int(gpu_scenario["q_clean"]):
        raise RuntimeError("Clean integer differs from historical scenario.")
    if after != int(gpu_scenario["q_faulty"]):
        raise RuntimeError("Faulty integer differs from historical scenario.")

    clean_layer.weight.copy_(q_clean.float() * scale)

    gpu_int4_faulty_model = copy.deepcopy(gpu_int4_clean_model)
    faulty_layer = dict(gpu_int4_faulty_model.named_modules())[layer_key]
    q_faulty = q_clean.clone()
    q_faulty[weight_index] = after
    faulty_layer.weight.copy_(q_faulty.float() * scale)

clean_state = gpu_int4_clean_model.state_dict()
faulty_state = gpu_int4_faulty_model.state_dict()
changes = {}

for name, tensor in clean_state.items():
    changed = int((tensor != faulty_state[name]).sum().item())
    if changed:
        changes[name] = changed

if changes != {layer_key + ".weight": 1}:
    raise RuntimeError(f"Unexpected model differences: {changes}")

gpu_int4_clean_model.to("cuda:0")
gpu_int4_faulty_model.to("cuda:0")

# Check the selected weight after transfer without changing either model.
clean_value = dict(
    gpu_int4_clean_model.named_modules()
)[layer_key].weight[weight_index].item()
faulty_value = dict(
    gpu_int4_faulty_model.named_modules()
)[layer_key].weight[weight_index].item()

if clean_value == faulty_value:
    raise RuntimeError("Fault was not preserved after device transfer.")

print("Scenario:", gpu_scenario["scenario_id"])
print("Layer:", layer_key)
print("Logical INT4 transition:", before, "->", after)
print("State differences before transfer:", changes)
print("Clean/faulty weight after transfer:", clean_value, faulty_value)
print("Both models are FP32 on cuda:0.")
print("Historical scale retained for CPU/GPU pilot comparison only.")
print("No inference performed. No files modified.")

Scenario: EXT2-001
Layer: model.1.conv
Logical INT4 transition: 1 -> 0
State differences before transfer: {'model.1.conv.weight': 1}
Clean/faulty weight after transfer: 0.0607212595641613 0.0
Both models are FP32 on cuda:0.
Historical scale retained for CPU/GPU pilot comparison only.
No inference performed. No files modified.


In [ ]:
import copy
import hashlib
import json
import time

import cv2
import numpy as np
import torch
from ultralytics.utils.ops import scale_boxes

folder = project / "code/results/notebook05"
targets = {
    "int4_clean": folder / "pilot50_int4_clean_gpu_cpuNMS_v1.json",
    "int4_faulty": folder / "pilot50_EXT2-001_gpu_cpuNMS_v1.json",
}

if any(path.exists() for path in targets.values()):
    raise FileExistsError("An output already exists. Inspect before rerunning.")

reference_path = folder / "pilot50_fp32_gpu_cpuNMS_predictions_v1.json"
reference_bytes = reference_path.read_bytes()
if hashlib.sha256(reference_bytes).hexdigest() != (
    "637e7f2642400376c510220b03d68324e32bd9f897aaf0b8e8bf43861452c22e"
):
    raise RuntimeError("GPU reference file changed.")

reference = json.loads(reference_bytes)
assert reference["image_ids"] == list(pilot_ids)
assert not set(pilot_ids) & set(split_manifest["image_ids"]["held_out"])

candidates = {
    "int4_clean": gpu_int4_clean_model,
    "int4_faulty": gpu_int4_faulty_model,
}
outputs = {
    name: {"predictions": [], "per_image": []}
    for name in candidates
}
started = time.perf_counter()

with torch.inference_mode():
    for number, image_id in enumerate(pilot_ids, start=1):
        record = image_records[image_id]
        path = (
            project / "datasets/coco2017/images/val2017"
            / record["file_name"]
        )
        image = cv2.imread(str(path))
        if image is None or image.shape[:2] != (
            record["height"], record["width"]
        ):
            raise RuntimeError(f"Image read or dimension failure: {image_id}")

        padded = transform(image=image)
        array = np.ascontiguousarray(
            padded[:, :, ::-1].transpose(2, 0, 1)
        )
        tensor = (
            torch.from_numpy(array).unsqueeze(0)
            .to(device="cuda:0", dtype=torch.float32) / 255.0
        )

        for name, candidate in candidates.items():
            raw = extract_raw(candidate(tensor))
            detections = cpu_nms(raw)
            detections[:, :4] = scale_boxes(
                tensor.shape[2:], detections[:, :4], image.shape[:2]
            )
            if not torch.isfinite(detections).all():
                raise RuntimeError(f"Non-finite detections: {name}, {image_id}")

            records = []
            for row in detections:
                x1, y1, x2, y2, score, cls = row.tolist()
                if x2 <= x1 or y2 <= y1:
                    continue
                if not (cls == int(cls) and 0 <= cls < 80 and 0 <= score <= 1):
                    raise RuntimeError(f"Invalid detection: {name}, {image_id}")
                records.append({
                    "image_id": int(image_id),
                    "category_id": int(category_ids[int(cls)]),
                    "bbox": [x1, y1, x2 - x1, y2 - y1],
                    "score": score,
                })

            outputs[name]["predictions"].extend(records)
            outputs[name]["per_image"].append({
                "image_id": int(image_id),
                "prediction_count": len(records),
            })

        if number % 10 == 0:
            print(f"GPU INT4 pair: {number}/50 completed", flush=True)

elapsed = time.perf_counter() - started

for name, path in targets.items():
    report = {
        key: copy.deepcopy(reference[key])
        for key in (
            "scope", "image_ids", "checkpoint_sha256",
            "split_manifest_sha256", "environment", "settings"
        )
    }
    report.update({
        "label": name,
        "quantization": {
            "layer": gpu_scenario["layer"],
            "scale": float(gpu_scenario["scale"]),
            "scale_source": "historical_EXT2_plan",
            "computed_on": "cpu",
            "clean_range": [-7, 7],
        },
        "scenario": copy.deepcopy(gpu_scenario) if name == "int4_faulty" else None,
        **outputs[name],
        "paired_run_wall_seconds": elapsed,
        "timing_note": "Shared elapsed time for both models, not per-model timing.",
    })

    content = (json.dumps(report, indent=2, allow_nan=False) + "\n").encode()
    with path.open("xb") as stream:
        stream.write(content)

    print("Saved:", path)
    print("SHA256:", hashlib.sha256(content).hexdigest())
    print("Predictions:", len(report["predictions"]))

print("Paired wall time (seconds):", round(elapsed, 2))
print("No held-out images evaluated. No commit or push performed.")

GPU INT4 pair: 10/50 completed
GPU INT4 pair: 20/50 completed
GPU INT4 pair: 30/50 completed
GPU INT4 pair: 40/50 completed
GPU INT4 pair: 50/50 completed
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook05/pilot50_int4_clean_gpu_cpuNMS_v1.json
SHA256: 45363669ce3a0fbf4162a376e8c8126c657a94d3f5229fb4053991d18c1836a5
Predictions: 5058
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook05/pilot50_EXT2-001_gpu_cpuNMS_v1.json
SHA256: 674a5db6c28e623a1f2cf050b7a0b4b7b8d9b794a56025fad171bfc852c6ad57
Predictions: 5047
Paired wall time (seconds): 6.63
No held-out images evaluated. No commit or push performed.


In [ ]:
import copy
import hashlib
import json
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

folder = project / "code/results/notebook05"
output_path = folder / "pilot50_INT4_cpu_gpu_cocoeval_comparison_v1.json"
if output_path.exists():
    raise FileExistsError("Comparison report already exists.")

sources = {
    "int4_clean": (
        "pilot50_int4_clean_gpu_cpuNMS_v1.json",
        "45363669ce3a0fbf4162a376e8c8126c657a94d3f5229fb4053991d18c1836a5",
    ),
    "int4_faulty": (
        "pilot50_EXT2-001_gpu_cpuNMS_v1.json",
        "674a5db6c28e623a1f2cf050b7a0b4b7b8d9b794a56025fad171bfc852c6ad57",
    ),
}

def read_verified(path, expected_hash):
    content = path.read_bytes()
    if hashlib.sha256(content).hexdigest() != expected_hash:
        raise RuntimeError(f"Integrity mismatch: {path.name}")
    return json.loads(content)

cpu_report = read_verified(
    folder / "pilot50_EXT2-001_cpu_cocoeval_v1.json",
    "6de9c5de39868d1d721be376e77d3d460be498dfcb3304e4d1b8e94f3f322353",
)

annotation_path = (
    project / "datasets/coco2017/annotations/instances_val2017.json"
)
if hashlib.sha256(annotation_path.read_bytes()).hexdigest() != (
    split_manifest["annotation_sha256"]
):
    raise RuntimeError("Ground-truth file changed.")

gt = COCO(str(annotation_path))
metric_indices = {"AP50_95": 0, "AP50": 1, "AP75": 2, "AR100": 8}
gpu_metrics = {}
settings = None

for name, (filename, expected_hash) in sources.items():
    run = read_verified(folder / filename, expected_hash)

    if run["image_ids"] != cpu_report["image_ids"]:
        raise RuntimeError(f"Evaluation image mismatch: {name}")
    if run["image_ids"] != split_manifest["image_ids"]["pilot"]:
        raise RuntimeError(f"Unexpected pilot IDs: {name}")
    if run["split_manifest_sha256"] != (
        "143c3273b2bf0cd6393cfe148a74e48fbafb74e770bd3fbdf3f14222355978ee"
    ):
        raise RuntimeError(f"Split provenance mismatch: {name}")
    if any(p["image_id"] not in run["image_ids"] for p in run["predictions"]):
        raise RuntimeError(f"Unexpected prediction image ID: {name}")
    if not run["predictions"]:
        raise RuntimeError(f"Empty predictions require explicit handling: {name}")

    if settings is None:
        settings = run["settings"]
    elif run["settings"] != settings:
        raise RuntimeError("GPU inference settings differ.")

    for key in ("imgsz", "batch", "conf", "nms_iou", "max_det"):
        if run["settings"][key] != cpu_report["settings"][key]:
            raise RuntimeError(f"CPU/GPU setting mismatch: {key}")

    dt = gt.loadRes(copy.deepcopy(run["predictions"]))
    evaluator = COCOeval(gt, dt, iouType="bbox")
    evaluator.params.imgIds = list(run["image_ids"])
    evaluator.params.catIds = sorted(gt.getCatIds())
    evaluator.params.maxDets = [1, 10, 100]

    print(f"\n=== GPU {name} ===")
    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()

    gpu_metrics[name] = {
        key: float(evaluator.stats[index])
        for key, index in metric_indices.items()
    }

deltas = {
    name: {
        key: gpu_metrics[name][key] - cpu_report["metrics"][name][key]
        for key in metric_indices
    }
    for name in sources
}
fault_delta = (
    gpu_metrics["int4_faulty"]["AP50_95"]
    - gpu_metrics["int4_clean"]["AP50_95"]
)

report = {
    "scope": "50_image_development_pilot_one_historical_fault",
    "source_gpu_prediction_sha256": {
        name: value[1] for name, value in sources.items()
    },
    "image_ids": cpu_report["image_ids"],
    "gpu_metrics": gpu_metrics,
    "gpu_minus_cpu": deltas,
    "gpu_fault_induced_AP50_95_change": fault_delta,
    "limitations": [
        "Pilot only; no layer ranking or statistical significance claim.",
        "No equivalence tolerance was predefined.",
        "No held-out images evaluated.",
    ],
}

content = (json.dumps(report, indent=2, allow_nan=False) + "\n").encode()
with output_path.open("xb") as stream:
    stream.write(content)

print("\nGPU metrics:", gpu_metrics)
print("GPU minus CPU:", deltas)
print("GPU fault-induced AP change:", fault_delta)
print("Saved:", output_path)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("No new inference. No commit or push performed.")

loading annotations into memory...
Done (t=0.64s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!

=== GPU int4_clean ===
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.55s).
Accumulating evaluation results...
DONE (t=0.44s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.535
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.652
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.579
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.350
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.544
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.677
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.460
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.619
 Average Recall     (AR) @[ IoU=0.50:0.95

In [ ]:
import json
from pathlib import Path

folder = project / "code/results/notebook05"

fp32_path = folder / "pilot50_fp32_gpu_cpuNMS_predictions_v1.json"
clean_path = folder / "pilot50_int4_clean_gpu_cpuNMS_v1.json"
faulty_path = folder / "pilot50_EXT2-001_gpu_cpuNMS_v1.json"

fp32_run = json.loads(fp32_path.read_bytes())
clean_run = json.loads(clean_path.read_bytes())

# Observed elapsed rates include different caching and execution conditions.
rate_a = fp32_run["wall_seconds"] / 50
rate_b = clean_run["paired_run_wall_seconds"] / (2 * 50)
low_rate, high_rate = sorted((rate_a, rate_b))

bytes_per_image = max(
    path.stat().st_size / 50
    for path in (fp32_path, clean_path, faulty_path)
)

print("Observed GPU pipeline seconds/image-forward:")
print(round(low_rate, 4), "to", round(high_rate, 4))
print("\nReplicates/bit | Fault scenarios | Image forwards | Hours | JSON GiB")

for replicates in (2, 5, 10):
    scenarios = 12 * 4 * replicates
    forwards = (1 + 12 + scenarios) * 1000
    low_hours = forwards * low_rate / 3600
    high_hours = forwards * high_rate / 3600
    storage_gib = forwards * bytes_per_image / 2**30

    print(
        f"{replicates:14d} | {scenarios:15d} | {forwards:14d} | "
        f"{low_hours:.1f}-{high_hours:.1f} | {storage_gib:.2f}"
    )

print("\nRough extrapolation, not guaranteed bounds or statistical intervals.")
print("JSON estimate assumes the observed maximum file bytes per image.")
print("Compression, prediction density and evaluation costs may change estimates.")
print("Replicate count remains undecided.")
print("No inference performed. No files modified.")

Observed GPU pipeline seconds/image-forward:
0.0663 to 0.3836

Replicates/bit | Fault scenarios | Image forwards | Hours | JSON GiB
             2 |              96 |         109000 | 2.0-11.6 | 2.37
             5 |             240 |         253000 | 4.7-27.0 | 5.50
            10 |             480 |         493000 | 9.1-52.5 | 10.71

Rough extrapolation, not guaranteed bounds or statistical intervals.
JSON estimate assumes the observed maximum file bytes per image.
Compression, prediction density and evaluation costs may change estimates.
Replicate count remains undecided.
No inference performed. No files modified.


In [ ]:
import json

main_design = {
    "status": "DRAFT_BEFORE_SCENARIO_GENERATION",
    "layers": list(historical_plan["selected_layers"]),
    "development_image_count": 1000,
    "bits": [0, 1, 2, 3],
    "weights_per_bit_per_layer": 5,
    "scenario_count": 240,
    "sampling": {
        "seed": 20261109,
        "weights": "uniform_without_replacement_within_each_layer_bit",
        "between_bits": "independent_sampling_weight_reuse_allowed",
        "sign_or_zero_balancing": False,
    },
    "quantization": {
        "scope": "target_layer_only",
        "scale": "CPU_FP32_tensor_abs_max_divided_by_7",
        "rounding": "torch.round",
        "clean_range": [-7, 7],
        "fault_range": [-8, 7],
        "inference_dtype": "float32",
    },
    "fault": {
        "operation": "one_logical_weight_bit_XOR",
        "persistence": "fixed_across_all_development_images_per_scenario",
        "reset": "exact_clean_state_restoration_and_verification",
    },
    "execution": {
        "inference_device": "cuda:0",
        "nms_device": "cpu",
        "resume_unit": "one_completed_scenario",
        "prediction_storage": "gzip_compressed_JSON",
    },
    "analysis": {
        "primary": "faulty_AP50_95_minus_layer_specific_clean_AP50_95",
        "report": [
            "quantization_degradation_separately",
            "per_scenario_and_per_bit_results",
            "layer_mean_and_distribution_of_fault_effects",
            "uncertainty_and_sampling_limitations",
        ],
        "scope": "selected_12_layers_not_the_entire_network",
        "replicate_count_is_power_validated": False,
        "additional_sampling": (
            "Any extension requires a versioned plan and disclosure "
            "of the reason; do not stop based on a preferred ranking."
        ),
    },
    "held_out_evaluation": False,
}

assert len(main_design["layers"]) == 12
assert (
    len(main_design["layers"])
    * len(main_design["bits"])
    * main_design["weights_per_bit_per_layer"]
    == main_design["scenario_count"]
)

print(json.dumps(main_design, indent=2))
print("\nDraft only. No scenarios generated. No files modified.")

{
  "status": "DRAFT_BEFORE_SCENARIO_GENERATION",
  "layers": [
    "1.conv",
    "4.cv1.conv",
    "4.m.1.cv2.conv",
    "6.m.1.cv1.conv",
    "9.cv1.conv",
    "9.cv2.conv",
    "15.cv2.conv",
    "18.m.0.cv2.conv",
    "21.cv2.conv",
    "22.cv2.2.0.conv",
    "22.cv3.0.2",
    "22.cv3.2.1.conv"
  ],
  "development_image_count": 1000,
  "bits": [
    0,
    1,
    2,
    3
  ],
  "weights_per_bit_per_layer": 5,
  "scenario_count": 240,
  "sampling": {
    "seed": 20261109,
    "weights": "uniform_without_replacement_within_each_layer_bit",
    "between_bits": "independent_sampling_weight_reuse_allowed",
    "sign_or_zero_balancing": false
  },
  "quantization": {
    "scope": "target_layer_only",
    "scale": "CPU_FP32_tensor_abs_max_divided_by_7",
    "rounding": "torch.round",
    "clean_range": [
      -7,
      7
    ],
    "fault_range": [
      -8,
      7
    ],
    "inference_dtype": "float32"
  },
  "fault": {
    "operation": "one_logical_weight_bit_XOR",
    "persistence"

In [ ]:
import copy
import hashlib
import json
import random
from pathlib import Path

import numpy as np
import torch
from ultralytics import YOLO

project = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")
output = project / "code/experiments/notebook05/int4_main_fault_plan_v1.json"
checkpoint = project / "models/yolov8s.pt"
split_path = (
    project / "code/experiments/notebook05/coco2017_split_manifest_v1.json"
)

if output.exists():
    raise FileExistsError("Fault plan already exists. Inspect before rerunning.")

checkpoint_hash = hashlib.sha256(checkpoint.read_bytes()).hexdigest()
split_hash = hashlib.sha256(split_path.read_bytes()).hexdigest()

if checkpoint_hash != (
    "1f47a78bf100391c2a140b7ac73a1caae18c32779be7d310658112f7ac9aa78a"
):
    raise RuntimeError("Checkpoint identity mismatch.")
if split_hash != (
    "143c3273b2bf0cd6393cfe148a74e48fbafb74e770bd3fbdf3f14222355978ee"
):
    raise RuntimeError("Split manifest identity mismatch.")

split = json.loads(split_path.read_bytes())
if len(split["image_ids"]["development"]) != 1000:
    raise RuntimeError("Unexpected development image count.")

source_yolo = YOLO(str(checkpoint))
source_model = source_yolo.model.cpu().float().eval()
modules = dict(source_model.named_modules())
rng = random.Random(main_design["sampling"]["seed"])

scenarios = []
layer_records = []

for layer_name in main_design["layers"]:
    layer = modules.get("model." + layer_name)
    if not isinstance(layer, torch.nn.Conv2d):
        raise RuntimeError(f"Expected Conv2d: {layer_name}")

    weight = layer.weight.detach()
    scale_tensor = weight.abs().max() / 7
    if not torch.isfinite(scale_tensor) or scale_tensor.item() <= 0:
        raise RuntimeError(f"Invalid scale: {layer_name}")

    q = torch.round(weight / scale_tensor).clamp(-7, 7).to(torch.int8)
    scale = float(scale_tensor.item())

    layer_records.append({
        "layer": layer_name,
        "weight_shape": list(weight.shape),
        "weight_count": weight.numel(),
        "scale": scale,
    })

    for bit in main_design["bits"]:
        flat_indices = rng.sample(
            range(weight.numel()),
            main_design["weights_per_bit_per_layer"],
        )

        for replicate, flat_index in enumerate(flat_indices, start=1):
            index = tuple(
                int(value) for value in np.unravel_index(flat_index, weight.shape)
            )
            before = int(q[index].item())
            flipped_bits = (before & 0xF) ^ (1 << bit)
            after = flipped_bits - 16 if flipped_bits >= 8 else flipped_bits

            assert ((before & 0xF) ^ (after & 0xF)) == (1 << bit)

            clean_fp32 = float((q[index].float() * scale_tensor).item())
            faulty_fp32 = float(
                (torch.tensor(after, dtype=torch.float32) * scale_tensor).item()
            )
            if clean_fp32 == faulty_fp32:
                raise RuntimeError("Injected bit change vanished after dequantization.")

            scenarios.append({
                "scenario_id": f"N05-{len(scenarios) + 1:03d}",
                "layer": layer_name,
                "bit": bit,
                "replicate": replicate,
                "flat_weight_index": flat_index,
                "weight_index": list(index),
                "q_clean": before,
                "q_faulty": after,
                "scale": scale,
                "clean_weight_fp32": clean_fp32,
                "faulty_weight_fp32": faulty_fp32,
            })

if len(scenarios) != 240:
    raise RuntimeError("Unexpected scenario count.")

plan_record = {
    "schema_version": "1.0",
    "status": "FAULT_SCENARIOS_FROZEN_EXECUTION_PROTOCOL_PENDING",
    "checkpoint_sha256": checkpoint_hash,
    "split_manifest_sha256": split_hash,
    "design": copy.deepcopy(main_design),
    "layers": layer_records,
    "scenarios": scenarios,
}
plan_record["design"]["status"] = "SCENARIOS_GENERATED"

content = (
    json.dumps(plan_record, indent=2, allow_nan=False) + "\n"
).encode("utf-8")
output.parent.mkdir(parents=True, exist_ok=True)
with output.open("xb") as stream:
    stream.write(content)

del source_model, source_yolo, modules

print("Saved:", output)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("Layers:", len(layer_records))
print("Scenarios:", len(scenarios))
print("First scenario:", scenarios[0])
print("No inference performed. No checkpoint modified.")

Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/experiments/notebook05/int4_main_fault_plan_v1.json
SHA256: 906af255a7b46654f4736e4a3f77c8c778570ed3d86e5e12077bf9dbcc6f0fbb
Layers: 12
Scenarios: 240
First scenario: {'scenario_id': 'N05-001', 'layer': '1.conv', 'bit': 0, 'replicate': 1, 'flat_weight_index': 6018, 'weight_index': [20, 28, 2, 0], 'q_clean': -2, 'q_faulty': -1, 'scale': 0.0607212595641613, 'clean_weight_fp32': -0.1214425191283226, 'faulty_weight_fp32': -0.0607212595641613}
No inference performed. No checkpoint modified.


In [ ]:
import hashlib
import json
import os
import platform
import subprocess
import sys
from datetime import datetime, timezone
from importlib.metadata import version

import torch

folder = project / "code/experiments/notebook05"
plan_path = folder / "int4_main_fault_plan_v1.json"
output = folder / "int4_main_execution_protocol_v1.json"

if output.exists():
    raise FileExistsError("Execution protocol already exists.")

plan_bytes = plan_path.read_bytes()
plan_sha = hashlib.sha256(plan_bytes).hexdigest()
if plan_sha != (
    "906af255a7b46654f4736e4a3f77c8c778570ed3d86e5e12077bf9dbcc6f0fbb"
):
    raise RuntimeError("Fault plan changed.")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable.")
if not torch.are_deterministic_algorithms_enabled():
    raise RuntimeError("Deterministic algorithms are not enabled.")
if (
    torch.backends.cudnn.benchmark
    or torch.backends.cuda.matmul.allow_tf32
    or torch.backends.cudnn.allow_tf32
):
    raise RuntimeError("Numerical settings differ from the GPU pilot.")

freeze = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    capture_output=True, text=True, check=True,
).stdout.splitlines()

protocol = {
    "schema_version": "1.0",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "fault_plan_sha256": plan_sha,
    "split_manifest_sha256": json.loads(plan_bytes)["split_manifest_sha256"],
    "environment": {
        "python": platform.python_version(),
        "packages": {
            name: version(name)
            for name in (
                "torch", "torchvision", "ultralytics", "numpy",
                "pycocotools", "opencv-python", "Pillow"
            )
        },
        "pip_freeze": freeze,
        "gpu": torch.cuda.get_device_name(0),
        "cuda_build": torch.version.cuda,
        "cudnn_version": torch.backends.cudnn.version(),
    },
    "inference": {
        "dtype": "float32",
        "batch": 1,
        "input_size": [640, 640],
        "device": "cuda:0",
        "letterbox": {
            "auto": False, "scale_fill": False,
            "scaleup": True, "stride": 32,
        },
        "color_order": "RGB",
        "normalization": "divide_by_255",
        "nms_device": "cpu",
        "confidence_threshold": 0.001,
        "nms_iou_threshold": 0.7,
        "max_det": 300,
    },
    "numerical_settings": {
        "deterministic_algorithms": True,
        "cudnn_deterministic": torch.backends.cudnn.deterministic,
        "cudnn_benchmark": False,
        "matmul_tf32": False,
        "cudnn_tf32": False,
        "CUBLAS_WORKSPACE_CONFIG": os.environ.get("CUBLAS_WORKSPACE_CONFIG"),
    },
    "evaluation": {
        "implementation": "pycocotools_COCOeval",
        "iou_type": "bbox",
        "max_dets": [1, 10, 100],
        "images": "all_1000_frozen_development_ids",
        "primary_change": "faulty_AP50_95_minus_layer_clean_AP50_95",
        "aggregate_AP": "joint_COCOeval_not_mean_per_image_AP",
    },
    "storage_requirements": {
        "predictions": "gzip_JSON",
        "completion": "verified_prediction_file_and_evaluation_report",
        "resume": "validate_provenance_before_skipping_completed_runs",
        "partial_files": "never_treat_as_completed",
        "overwrite_existing_results": False,
        "empty_predictions": "evaluate_as_empty_detections_not_skip_images",
    },
    "execution_gate": "validate_one_complete_scenario_before_batch_execution",
    "analysis_status": "uncertainty_estimation_protocol_pending_before_batch",
    "held_out_evaluation": False,
}

content = (
    json.dumps(protocol, indent=2, allow_nan=False) + "\n"
).encode("utf-8")

with output.open("xb") as stream:
    stream.write(content)

print("Saved:", output)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("Development images per run: 1000")
print("First execution gate: one complete scenario")
print("No inference, commit or push performed.")

Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/experiments/notebook05/int4_main_execution_protocol_v1.json
SHA256: 3bf8913bb78ac1cb0b903a5e45caa0ff67ae867eb3a583991d76b4cf606a66ed
Development images per run: 1000
First execution gate: one complete scenario
No inference, commit or push performed.


In [ ]:
import hashlib
import json
from pathlib import Path

project = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")
split_path = (
    project / "code/experiments/notebook05/coco2017_split_manifest_v1.json"
)
split_bytes = split_path.read_bytes()

if hashlib.sha256(split_bytes).hexdigest() != (
    "143c3273b2bf0cd6393cfe148a74e48fbafb74e770bd3fbdf3f14222355978ee"
):
    raise RuntimeError("Split manifest changed.")

split = json.loads(split_bytes)
development_ids = split["image_ids"]["development"]
assert len(development_ids) == len(set(development_ids)) == 1000
assert not set(development_ids) & set(split["image_ids"]["held_out"])

annotation_path = (
    project / "datasets/coco2017/annotations/instances_val2017.json"
)
annotation_bytes = annotation_path.read_bytes()
if hashlib.sha256(annotation_bytes).hexdigest() != split["annotation_sha256"]:
    raise RuntimeError("Annotation file changed.")

records = {
    item["id"]: item for item in json.loads(annotation_bytes)["images"]
}
source_dir = project / "datasets/coco2017/images/val2017"
development_cache = Path("/content/notebook05_cache/development_images")
development_cache.mkdir(parents=True, exist_ok=True)

copied = 0
reused = 0
total_bytes = 0

for number, image_id in enumerate(development_ids, start=1):
    filename = records[image_id]["file_name"]
    if Path(filename).name != filename:
        raise RuntimeError("Unexpected image filename.")

    source = source_dir / filename
    destination = development_cache / filename
    content = source.read_bytes()
    expected = hashlib.sha256(content).digest()

    if destination.exists():
        if hashlib.sha256(destination.read_bytes()).digest() != expected:
            raise RuntimeError(f"Existing cache file differs: {filename}")
        reused += 1
    else:
        with destination.open("xb") as stream:
            stream.write(content)
        if hashlib.sha256(destination.read_bytes()).digest() != expected:
            raise RuntimeError(f"Copy verification failed: {filename}")
        copied += 1

    total_bytes += len(content)
    if number % 200 == 0:
        print(f"Verified {number}/1000 development images", flush=True)

print("Cache:", development_cache)
print("New copies:", copied)
print("Verified existing copies:", reused)
print("Total size (MiB):", round(total_bytes / 2**20, 2))
print("No held-out images copied. No inference performed.")
print("Original Drive files unchanged.")

Verified 200/1000 development images
Verified 400/1000 development images
Verified 600/1000 development images
Verified 800/1000 development images
Verified 1000/1000 development images
Cache: /content/notebook05_cache/development_images
New copies: 1000
Verified existing copies: 0
Total size (MiB): 152.77
No held-out images copied. No inference performed.
Original Drive files unchanged.


In [ ]:
import gzip
import hashlib
import json
import time
from pathlib import Path

import cv2
import numpy as np
import torch
from ultralytics.data.augment import LetterBox
from ultralytics.utils.nms import non_max_suppression
from ultralytics.utils.ops import scale_boxes

experiment_dir = project / "code/experiments/notebook05"
result_dir = project / "code/results/notebook05/main_v1"

def verified_json(path, expected_hash):
    content = Path(path).read_bytes()
    if hashlib.sha256(content).hexdigest() != expected_hash:
        raise RuntimeError(f"File integrity mismatch: {Path(path).name}")
    return json.loads(content)

main_plan = verified_json(
    experiment_dir / "int4_main_fault_plan_v1.json",
    "906af255a7b46654f4736e4a3f77c8c778570ed3d86e5e12077bf9dbcc6f0fbb",
)
execution_protocol = verified_json(
    experiment_dir / "int4_main_execution_protocol_v1.json",
    "3bf8913bb78ac1cb0b903a5e45caa0ff67ae867eb3a583991d76b4cf606a66ed",
)

def model_state_hash(candidate):
    digest = hashlib.sha256()
    for name, tensor in sorted(candidate.state_dict().items()):
        array = tensor.detach().cpu().contiguous().numpy()
        digest.update(name.encode())
        digest.update(str(array.dtype).encode())
        digest.update(str(array.shape).encode())
        digest.update(array.tobytes())
    return digest.hexdigest()

def run_main_predictions(candidate, run_id, metadata):
    if not run_id or any(c not in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-_" for c in run_id):
        raise ValueError("Invalid run ID.")

    result_dir.mkdir(parents=True, exist_ok=True)
    destination = result_dir / f"{run_id}_predictions.json.gz"
    partial = result_dir / f"{run_id}_predictions.json.gz.part"

    if destination.exists() or partial.exists():
        raise FileExistsError("Existing result or partial file requires inspection.")

    if next(candidate.parameters()).device.type != "cuda":
        raise RuntimeError("Expected CUDA model.")
    if any(p.dtype != torch.float32 for p in candidate.parameters()):
        raise RuntimeError("Expected FP32 parameters.")

    from importlib.metadata import version
    import platform

    recorded_env = execution_protocol["environment"]
    if platform.python_version() != recorded_env["python"]:
        raise RuntimeError("Python version differs from execution protocol.")
    for name, expected in recorded_env["packages"].items():
        if version(name) != expected:
            raise RuntimeError(f"Package version differs: {name}")
    if (
        not torch.are_deterministic_algorithms_enabled()
        or torch.backends.cudnn.benchmark
        or torch.backends.cuda.matmul.allow_tf32
        or torch.backends.cudnn.allow_tf32
    ):
        raise RuntimeError("Numerical settings differ from protocol.")

    candidate.eval()
    # Warm up before hashing: inference can initialize model buffers.
    with torch.inference_mode():
        warmup = candidate(torch.zeros(1, 3, 640, 640, device="cuda"))
    del warmup
    torch.cuda.synchronize()
    state_before = model_state_hash(candidate)

    transform = LetterBox(
        new_shape=(640, 640), auto=False,
        scale_fill=False, scaleup=True, stride=32,
    )
    predictions, per_image = [], []
    started = time.perf_counter()

    with torch.inference_mode():
        for number, image_id in enumerate(development_ids, start=1):
            record = records[image_id]
            image = cv2.imread(str(development_cache / record["file_name"]))
            if image is None or image.shape[:2] != (
                record["height"], record["width"]
            ):
                raise RuntimeError(f"Image validation failed: {image_id}")

            padded = transform(image=image)
            array = np.ascontiguousarray(
                padded[:, :, ::-1].transpose(2, 0, 1)
            )
            tensor = (
                torch.from_numpy(array).unsqueeze(0)
                .to(device="cuda:0", dtype=torch.float32) / 255.0
            )
            output = candidate(tensor)
            raw = output[0] if isinstance(output, (tuple, list)) else output
            if tuple(raw.shape) != (1, 84, 8400) or not torch.isfinite(raw).all():
                raise RuntimeError(f"Invalid raw output: {image_id}")

            detections = non_max_suppression(
                raw.detach().cpu().clone(),
                conf_thres=0.001, iou_thres=0.7, nc=80, max_det=300,
            )[0].clone()
            detections[:, :4] = scale_boxes(
                tensor.shape[2:], detections[:, :4], image.shape[:2]
            )
            if not torch.isfinite(detections).all():
                raise RuntimeError(f"Invalid detections: {image_id}")

            count = 0
            for row in detections:
                x1, y1, x2, y2, score, cls = row.tolist()
                if x2 <= x1 or y2 <= y1:
                    continue
                if not (cls == int(cls) and 0 <= cls < 80 and 0 <= score <= 1):
                    raise RuntimeError("Invalid class or confidence.")
                predictions.append({
                    "image_id": int(image_id),
                    "category_id": int(category_ids[int(cls)]),
                    "bbox": [x1, y1, x2 - x1, y2 - y1],
                    "score": score,
                })
                count += 1

            per_image.append({
                "image_id": int(image_id), "prediction_count": count
            })
            if number % 100 == 0:
                print(f"{run_id}: {number}/1000", flush=True)

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    state_after = model_state_hash(candidate)
    if state_after != state_before:
        raise RuntimeError("Model state changed during evaluation.")

    report = {
        "run_id": run_id,
        "status": "PREDICTIONS_COMPLETE_EVALUATION_PENDING",
        "image_ids": list(development_ids),
        "fault_plan_sha256": execution_protocol["fault_plan_sha256"],
        "execution_protocol_sha256": (
            "3bf8913bb78ac1cb0b903a5e45caa0ff67ae867eb3a583991d76b4cf606a66ed"
        ),
        "metadata": metadata,
        "model_state_sha256": state_before,
        "predictions": predictions,
        "per_image": per_image,
        "wall_seconds": elapsed,
    }
    payload = json.dumps(report, allow_nan=False, separators=(",", ":")).encode()
    with partial.open("xb") as stream:
        with gzip.GzipFile(fileobj=stream, mode="wb", mtime=0) as compressed:
            compressed.write(payload)

    with gzip.open(partial, "rb") as stream:
        if stream.read() != payload:
            raise RuntimeError("Compressed output verification failed.")

    if destination.exists():
        raise FileExistsError("Destination appeared during execution.")
    partial.rename(destination)

    print("Saved:", destination)
    print("SHA256:", hashlib.sha256(destination.read_bytes()).hexdigest())
    print("Predictions:", len(predictions))
    print("Wall time (seconds):", round(elapsed, 2))
    print("Model state unchanged: PASS")
    return destination

print("Main prediction runner defined. No inference performed by this cell.")
print("Existing results were not modified.")

Main prediction runner defined. No inference performed by this cell.
Existing results were not modified.


In [ ]:
import hashlib
import json
from ultralytics import YOLO

split = verified_json(
    experiment_dir / "coco2017_split_manifest_v1.json",
    "143c3273b2bf0cd6393cfe148a74e48fbafb74e770bd3fbdf3f14222355978ee",
)
development_ids = list(split["image_ids"]["development"])

if len(development_ids) != len(set(development_ids)) != 1000:
    raise RuntimeError("Invalid development image IDs.")
if len(development_ids) != 1000:
    raise RuntimeError("Expected 1000 development images.")
if set(development_ids) & set(split["image_ids"]["held_out"]):
    raise RuntimeError("Development and held-out IDs overlap.")

checkpoint = project / "models/yolov8s.pt"
checkpoint_sha = hashlib.sha256(checkpoint.read_bytes()).hexdigest()
if checkpoint_sha != main_plan["checkpoint_sha256"]:
    raise RuntimeError("Checkpoint identity mismatch.")

baseline_yolo = YOLO(str(checkpoint))
baseline_gpu_model = baseline_yolo.model.float().to("cuda:0").eval()

category_ids = sorted(item["id"] for item in coco["categories"])
category_names = {item["id"]: item["name"] for item in coco["categories"]}
if len(category_ids) != 80 or not all(
    baseline_yolo.names[i] == category_names[cid]
    for i, cid in enumerate(category_ids)
):
    raise RuntimeError("COCO category mapping mismatch.")

baseline_prediction_path = run_main_predictions(
    baseline_gpu_model,
    "FP32-clean",
    {
        "model": "YOLOv8s",
        "checkpoint_sha256": checkpoint_sha,
        "quantization": None,
        "fault": None,
    },
)

FP32-clean: 100/1000
FP32-clean: 200/1000
FP32-clean: 300/1000
FP32-clean: 400/1000
FP32-clean: 500/1000
FP32-clean: 600/1000
FP32-clean: 700/1000
FP32-clean: 800/1000
FP32-clean: 900/1000
FP32-clean: 1000/1000
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook05/main_v1/FP32-clean_predictions.json.gz
SHA256: dc6f83e2e9382f6f8e6c8843a19e5966835275985120ce47e5225c6a922c24fc
Predictions: 109657
Wall time (seconds): 40.7
Model state unchanged: PASS


In [ ]:
import copy
import gzip
import hashlib
import json

import numpy as np
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

prediction_path = result_dir / "FP32-clean_predictions.json.gz"
evaluation_path = result_dir / "FP32-clean_evaluation_v1.json"

if evaluation_path.exists():
    raise FileExistsError("Evaluation report already exists.")

compressed = prediction_path.read_bytes()
prediction_sha = hashlib.sha256(compressed).hexdigest()
if prediction_sha != (
    "dc6f83e2e9382f6f8e6c8843a19e5966835275985120ce47e5225c6a922c24fc"
):
    raise RuntimeError("Prediction file integrity mismatch.")

run = json.loads(gzip.decompress(compressed))
split = verified_json(
    experiment_dir / "coco2017_split_manifest_v1.json",
    "143c3273b2bf0cd6393cfe148a74e48fbafb74e770bd3fbdf3f14222355978ee",
)
expected_ids = split["image_ids"]["development"]

if len(expected_ids) != 1000 or len(set(expected_ids)) != 1000:
    raise RuntimeError("Expected 1000 unique development image IDs.")
if run["image_ids"] != expected_ids:
    raise RuntimeError("Evaluation image list mismatch.")
if [item["image_id"] for item in run["per_image"]] != expected_ids:
    raise RuntimeError("Incomplete per-image execution record.")
if any(item["image_id"] not in expected_ids for item in run["predictions"]):
    raise RuntimeError("Unexpected prediction image ID.")
if sum(item["prediction_count"] for item in run["per_image"]) != len(
    run["predictions"]
):
    raise RuntimeError("Prediction count mismatch.")

annotation_path = (
    project / "datasets/coco2017/annotations/instances_val2017.json"
)
if hashlib.sha256(annotation_path.read_bytes()).hexdigest() != (
    split["annotation_sha256"]
):
    raise RuntimeError("Ground-truth integrity mismatch.")

gt_main = COCO(str(annotation_path))

if run["predictions"]:
    dt_main = gt_main.loadRes(copy.deepcopy(run["predictions"]))
else:
    dt_main = COCO()
    dt_main.dataset = {
        "images": copy.deepcopy(gt_main.dataset["images"]),
        "categories": copy.deepcopy(gt_main.dataset["categories"]),
        "annotations": [],
    }
    dt_main.createIndex()

main_evaluator = COCOeval(gt_main, dt_main, iouType="bbox")
main_evaluator.params.imgIds = list(expected_ids)
main_evaluator.params.catIds = sorted(gt_main.getCatIds())
main_evaluator.params.maxDets = [1, 10, 100]
main_evaluator.evaluate()
main_evaluator.accumulate()
main_evaluator.summarize()

if not np.isfinite(main_evaluator.stats).all():
    raise RuntimeError("Non-finite evaluation metrics.")

metric_names = [
    "AP50_95", "AP50", "AP75", "AP_small", "AP_medium", "AP_large",
    "AR1", "AR10", "AR100", "AR_small", "AR_medium", "AR_large",
]
report = {
    "run_id": run["run_id"],
    "status": "PREDICTIONS_AND_EVALUATION_COMPLETE",
    "prediction_file_sha256": prediction_sha,
    "fault_plan_sha256": run["fault_plan_sha256"],
    "execution_protocol_sha256": run["execution_protocol_sha256"],
    "image_ids": expected_ids,
    "metrics": dict(zip(metric_names, map(float, main_evaluator.stats))),
    "cocoeval_max_dets": [1, 10, 100],
    "scope": "development_only",
    "limitations": [
        "Development subset, not held-out final performance.",
        "Classes without eligible ground truth are excluded from COCO AP averaging.",
        "Subset AP must not be directly interpreted as improvement over historical full-set AP.",
    ],
}

content = (json.dumps(report, indent=2, allow_nan=False) + "\n").encode()
with evaluation_path.open("xb") as stream:
    stream.write(content)

print("\nSaved:", evaluation_path)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("No new inference. No held-out images evaluated.")

loading annotations into memory...
Done (t=0.97s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.19s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=9.13s).
Accumulating evaluation results...
DONE (t=1.65s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.461
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.620
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.502
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.257
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.513
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.638
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.356
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.569
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets

In [ ]:
import torch
from ultralytics import YOLO

layer_record = main_plan["layers"][0]
layer_name = layer_record["layer"]
layer_key = "model." + layer_name
scale = float(layer_record["scale"])

layer_yolo = YOLO(str(project / "models/yolov8s.pt"))
layer_clean_model = layer_yolo.model.cpu().float().eval()
target = dict(layer_clean_model.named_modules())[layer_key]

if not isinstance(target, torch.nn.Conv2d):
    raise RuntimeError("Expected Conv2d layer.")
if list(target.weight.shape) != layer_record["weight_shape"]:
    raise RuntimeError("Layer shape mismatch.")

reference_state = {
    name: tensor.detach().clone()
    for name, tensor in layer_clean_model.state_dict().items()
}

with torch.no_grad():
    scale_tensor = target.weight.detach().abs().max() / 7
    if float(scale_tensor.item()) != scale:
        raise RuntimeError("Scale differs from the frozen fault plan.")

    layer_q_clean = torch.round(
        target.weight.detach() / scale_tensor
    ).clamp(-7, 7).to(torch.int8)

    target.weight.copy_(layer_q_clean.float() * scale_tensor)

for name, tensor in layer_clean_model.state_dict().items():
    if name != layer_key + ".weight":
        if not torch.equal(tensor, reference_state[name]):
            raise RuntimeError(f"Unexpected state change: {name}")

if not torch.equal(target.weight, layer_q_clean.float() * scale_tensor):
    raise RuntimeError("Clean quantized weights do not match.")

del reference_state
layer_clean_model.to("cuda:0").eval()

print("Target layer:", layer_name)
print("Scale:", scale)
print("Target-layer-only quantization check: PASS")

layer_clean_prediction_path = run_main_predictions(
    layer_clean_model,
    "INT4-clean-L01",
    {
        "checkpoint_sha256": main_plan["checkpoint_sha256"],
        "quantization": layer_record,
        "fault": None,
    },
)

Target layer: 1.conv
Scale: 0.0607212595641613
Target-layer-only quantization check: PASS
INT4-clean-L01: 100/1000
INT4-clean-L01: 200/1000
INT4-clean-L01: 300/1000
INT4-clean-L01: 400/1000
INT4-clean-L01: 500/1000
INT4-clean-L01: 600/1000
INT4-clean-L01: 700/1000
INT4-clean-L01: 800/1000
INT4-clean-L01: 900/1000
INT4-clean-L01: 1000/1000
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook05/main_v1/INT4-clean-L01_predictions.json.gz
SHA256: 219f08143e4b3fdf297c0cc2563e43ef5c23ee8cef20dbfecc4002535dceaecd
Predictions: 114946
Wall time (seconds): 39.26
Model state unchanged: PASS


In [ ]:
import copy
import gzip
import hashlib
import json

import numpy as np
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

def evaluate_main_run(run_id, expected_prediction_sha):
    prediction_path = result_dir / f"{run_id}_predictions.json.gz"
    evaluation_path = result_dir / f"{run_id}_evaluation_v1.json"

    if evaluation_path.exists():
        raise FileExistsError("Evaluation already exists; inspect before rerunning.")

    compressed = prediction_path.read_bytes()
    prediction_sha = hashlib.sha256(compressed).hexdigest()
    if prediction_sha != expected_prediction_sha:
        raise RuntimeError("Prediction integrity mismatch.")

    run = json.loads(gzip.decompress(compressed))
    split = verified_json(
        experiment_dir / "coco2017_split_manifest_v1.json",
        "143c3273b2bf0cd6393cfe148a74e48fbafb74e770bd3fbdf3f14222355978ee",
    )
    ids = split["image_ids"]["development"]

    if run["run_id"] != run_id or run["image_ids"] != ids:
        raise RuntimeError("Run identity or image list mismatch.")
    if len(ids) != 1000 or len(set(ids)) != 1000:
        raise RuntimeError("Invalid development image list.")
    if [item["image_id"] for item in run["per_image"]] != ids:
        raise RuntimeError("Incomplete image execution record.")
    if sum(item["prediction_count"] for item in run["per_image"]) != len(
        run["predictions"]
    ):
        raise RuntimeError("Prediction count mismatch.")
    if any(p["image_id"] not in set(ids) for p in run["predictions"]):
        raise RuntimeError("Unexpected prediction image ID.")
    if run["fault_plan_sha256"] != (
        "906af255a7b46654f4736e4a3f77c8c778570ed3d86e5e12077bf9dbcc6f0fbb"
    ):
        raise RuntimeError("Fault plan mismatch.")
    if run["execution_protocol_sha256"] != (
        "3bf8913bb78ac1cb0b903a5e45caa0ff67ae867eb3a583991d76b4cf606a66ed"
    ):
        raise RuntimeError("Execution protocol mismatch.")

    annotation_path = (
        project / "datasets/coco2017/annotations/instances_val2017.json"
    )
    if hashlib.sha256(annotation_path.read_bytes()).hexdigest() != (
        split["annotation_sha256"]
    ):
        raise RuntimeError("Ground-truth integrity mismatch.")

    gt = COCO(str(annotation_path))
    if run["predictions"]:
        dt = gt.loadRes(copy.deepcopy(run["predictions"]))
    else:
        dt = COCO()
        dt.dataset = {
            "images": copy.deepcopy(gt.dataset["images"]),
            "categories": copy.deepcopy(gt.dataset["categories"]),
            "annotations": [],
        }
        dt.createIndex()

    evaluator = COCOeval(gt, dt, iouType="bbox")
    evaluator.params.imgIds = list(ids)
    evaluator.params.catIds = sorted(gt.getCatIds())
    evaluator.params.maxDets = [1, 10, 100]
    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()

    if not np.isfinite(evaluator.stats).all():
        raise RuntimeError("Non-finite metrics.")

    names = [
        "AP50_95", "AP50", "AP75", "AP_small", "AP_medium", "AP_large",
        "AR1", "AR10", "AR100", "AR_small", "AR_medium", "AR_large",
    ]
    report = {
        "run_id": run_id,
        "status": "PREDICTIONS_AND_EVALUATION_COMPLETE",
        "prediction_file_sha256": prediction_sha,
        "fault_plan_sha256": run["fault_plan_sha256"],
        "execution_protocol_sha256": run["execution_protocol_sha256"],
        "model_state_sha256": run["model_state_sha256"],
        "metadata": run["metadata"],
        "image_ids": ids,
        "metrics": dict(zip(names, map(float, evaluator.stats))),
        "cocoeval_max_dets": [1, 10, 100],
        "scope": "development_only_not_final_test",
    }
    content = (json.dumps(report, indent=2, allow_nan=False) + "\n").encode()
    with evaluation_path.open("xb") as stream:
        stream.write(content)

    print("Saved:", evaluation_path)
    print("SHA256:", hashlib.sha256(content).hexdigest())
    return report

layer_clean_evaluation = evaluate_main_run(
    "INT4-clean-L01",
    "219f08143e4b3fdf297c0cc2563e43ef5c23ee8cef20dbfecc4002535dceaecd",
)

fp32_evaluation = verified_json(
    result_dir / "FP32-clean_evaluation_v1.json",
    "d4b7a578dd31a2aa756083552f1a4a2f2472a7e570c412fedada81a9f86f3bff",
)

print(
    "\nQuantization AP change (INT4 clean - FP32):",
    layer_clean_evaluation["metrics"]["AP50_95"]
    - fp32_evaluation["metrics"]["AP50_95"],
)
print("No fault injected. No new inference performed.")

loading annotations into memory...
Done (t=0.49s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.13s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=8.75s).
Accumulating evaluation results...
DONE (t=2.99s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.400
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.545
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.436
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.192
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.466
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.557
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.322
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.520
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets

In [ ]:
import copy
import torch

first_scenario = main_plan["scenarios"][0]
if first_scenario["scenario_id"] != "N05-001":
    raise RuntimeError("Unexpected first scenario.")
if first_scenario["layer"] != layer_record["layer"]:
    raise RuntimeError("Scenario and clean baseline layers differ.")

clean_state_before = model_state_hash(layer_clean_model)
if clean_state_before != layer_clean_evaluation["model_state_sha256"]:
    raise RuntimeError("Clean baseline state differs from its evaluated state.")

first_faulty_model = copy.deepcopy(layer_clean_model).eval()
target_key = "model." + first_scenario["layer"]
faulty_target = dict(first_faulty_model.named_modules())[target_key]
index = tuple(first_scenario["weight_index"])

if faulty_target.weight[index].item() != first_scenario["clean_weight_fp32"]:
    raise RuntimeError("Clean weight differs from the frozen scenario.")

with torch.no_grad():
    faulty_target.weight[index] = first_scenario["faulty_weight_fp32"]

clean_state = layer_clean_model.state_dict()
faulty_state = first_faulty_model.state_dict()
changes = {}

for name, clean_tensor in clean_state.items():
    count = int((clean_tensor != faulty_state[name]).sum().item())
    if count:
        changes[name] = count

if changes != {target_key + ".weight": 1}:
    raise RuntimeError(f"Unexpected model changes: {changes}")

if faulty_target.weight[index].item() != first_scenario["faulty_weight_fp32"]:
    raise RuntimeError("Faulty weight differs from the frozen scenario.")

print("Scenario:", first_scenario["scenario_id"])
print("Layer:", first_scenario["layer"])
print("Weight index:", index)
print("Bit:", first_scenario["bit"])
print("INT4:", first_scenario["q_clean"], "->", first_scenario["q_faulty"])
print("Single-weight isolation check: PASS")

first_fault_prediction_path = run_main_predictions(
    first_faulty_model,
    first_scenario["scenario_id"],
    {
        "checkpoint_sha256": main_plan["checkpoint_sha256"],
        "quantization": layer_record,
        "fault": first_scenario,
        "clean_baseline_run_id": "INT4-clean-L01",
        "clean_baseline_state_sha256": clean_state_before,
    },
)

if model_state_hash(layer_clean_model) != clean_state_before:
    raise RuntimeError("Original clean baseline was modified.")

print("Original clean baseline preserved: PASS")

Scenario: N05-001
Layer: 1.conv
Weight index: (20, 28, 2, 0)
Bit: 0
INT4: -2 -> -1
Single-weight isolation check: PASS
N05-001: 100/1000
N05-001: 200/1000
N05-001: 300/1000
N05-001: 400/1000
N05-001: 500/1000
N05-001: 600/1000
N05-001: 700/1000
N05-001: 800/1000
N05-001: 900/1000
N05-001: 1000/1000
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook05/main_v1/N05-001_predictions.json.gz
SHA256: 7169d2175c1aa7397af06cb84fcd99bbc8b9f4b8c1fca63643eac9db88b57a50
Predictions: 114808
Wall time (seconds): 39.48
Model state unchanged: PASS
Original clean baseline preserved: PASS


In [ ]:
import hashlib
import json

comparison_path = result_dir / "N05-001_vs_INT4-clean-L01_v1.json"
if comparison_path.exists():
    raise FileExistsError("Comparison already exists; inspect before rerunning.")

clean_evaluation = verified_json(
    result_dir / "INT4-clean-L01_evaluation_v1.json",
    "2806756643d31d3a9713a27399266b75b90606f36e413e602ce6026ccfe48e90",
)

fault_evaluation = evaluate_main_run(
    "N05-001",
    "7169d2175c1aa7397af06cb84fcd99bbc8b9f4b8c1fca63643eac9db88b57a50",
)

if fault_evaluation["image_ids"] != clean_evaluation["image_ids"]:
    raise RuntimeError("Clean and faulty evaluation images differ.")

metadata = fault_evaluation["metadata"]
if metadata["clean_baseline_state_sha256"] != (
    clean_evaluation["model_state_sha256"]
):
    raise RuntimeError("Clean baseline state provenance mismatch.")
if metadata["quantization"] != clean_evaluation["metadata"]["quantization"]:
    raise RuntimeError("Quantization configurations differ.")

metric_keys = ("AP50_95", "AP50", "AP75", "AR100")
changes = {
    key: (
        fault_evaluation["metrics"][key]
        - clean_evaluation["metrics"][key]
    )
    for key in metric_keys
}

comparison = {
    "scenario_id": "N05-001",
    "clean_baseline_run_id": "INT4-clean-L01",
    "image_count": len(clean_evaluation["image_ids"]),
    "clean_metrics": {
        key: clean_evaluation["metrics"][key] for key in metric_keys
    },
    "faulty_metrics": {
        key: fault_evaluation["metrics"][key] for key in metric_keys
    },
    "faulty_minus_clean": changes,
    "faulty_evaluation_sha256": hashlib.sha256(
        (result_dir / "N05-001_evaluation_v1.json").read_bytes()
    ).hexdigest(),
    "clean_evaluation_sha256": (
        "2806756643d31d3a9713a27399266b75b90606f36e413e602ce6026ccfe48e90"
    ),
    "scope": "one_fault_scenario_on_1000_development_images",
    "limitations": [
        "No layer ranking or statistical significance claim.",
        "Held-out final evaluation has not been performed.",
    ],
}

content = (json.dumps(comparison, indent=2, allow_nan=False) + "\n").encode()
with comparison_path.open("xb") as stream:
    stream.write(content)

print("\nClean:", comparison["clean_metrics"])
print("Faulty:", comparison["faulty_metrics"])
print("Faulty minus clean:", changes)
print("Saved:", comparison_path)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("No new inference. No commit or push performed.")

loading annotations into memory...
Done (t=0.49s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.13s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=14.01s).
Accumulating evaluation results...
DONE (t=1.68s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.401
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.544
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.436
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.193
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.467
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.557
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.322
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.520
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDet

In [ ]:
import hashlib
import json
from datetime import datetime, timezone

output = experiment_dir / "int4_main_analysis_protocol_v1.json"
if output.exists():
    raise FileExistsError("Analysis protocol already exists.")

analysis_protocol = {
    "schema_version": "1.0",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "fault_plan_sha256": (
        "906af255a7b46654f4736e4a3f77c8c778570ed3d86e5e12077bf9dbcc6f0fbb"
    ),
    "timing_disclosure": (
        "Recorded after observing the historical pilot and N05-001, "
        "before evaluating the remaining 239 planned scenarios."
    ),
    "primary_effect": "faulty_AP50_95_minus_layer_specific_clean_AP50_95",
    "inclusion": {
        "retain_positive_negative_and_zero_effects": True,
        "do_not_drop_failed_runs": True,
        "failed_runs": "report_separately_with_reason_and_completion_status",
        "do_not_change_sampling_based_on_observed_ranking": True,
    },
    "layer_summary": {
        "scenario_count": 20,
        "scenarios_per_bit": 5,
        "primary_estimate": "equal_weight_mean_of_four_bit_specific_mean_effects",
        "additional_summaries": [
            "all_individual_effects",
            "per_bit_mean_and_range",
            "layer_median_minimum_maximum",
            "counts_of_positive_negative_and_exact_zero_effects",
            "quantization_degradation_separately",
        ],
    },
    "fault_sampling_uncertainty": {
        "method": "stratified_percentile_bootstrap",
        "resampling_unit": "sampled_weight_fault_scenario",
        "strata": "bit_position_within_each_layer",
        "resamples": 2000,
        "seed": 20261110,
        "interval_percentiles": [2.5, 97.5],
        "fixed": "development_images_and_layer_clean_baseline",
        "limitations": [
            "Only five sampled weights per bit; interval coverage may be poor.",
            "Exploratory intervals are not proof of statistical significance.",
            "Intervals exclude image-sampling and layer-selection uncertainty.",
            "Pointwise intervals are not simultaneous comparisons across layers.",
        ],
    },
    "ranking": {
        "descriptive_only": True,
        "criterion": "ascending_mean_fault_effect_more_negative_first",
        "scope": "12_selected_layers_under_the_defined_fault_distribution",
        "no_network_wide_or_definitive_rank_claim": True,
    },
    "extensions": (
        "Any additional sampling requires a versioned amendment, "
        "a stated reason and disclosure of results already observed."
    ),
    "held_out_evaluation": False,
}

content = (
    json.dumps(analysis_protocol, indent=2, allow_nan=False) + "\n"
).encode("utf-8")
with output.open("xb") as stream:
    stream.write(content)

print("Saved:", output)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("Observation timing disclosed: pilot and N05-001 already observed.")
print("No inference, commit or push performed.")

Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/experiments/notebook05/int4_main_analysis_protocol_v1.json
SHA256: 7bc74fb60ac14ca0afaf4a8fb74646cae4fa57dc98259e99a72b1895305b1ab5
Observation timing disclosed: pilot and N05-001 already observed.
No inference, commit or push performed.


In [ ]:
import copy
import gzip
import hashlib
import json
import torch

verified_json(
    experiment_dir / "int4_main_analysis_protocol_v1.json",
    "7bc74fb60ac14ca0afaf4a8fb74646cae4fa57dc98259e99a72b1895305b1ab5",
)

clean_report = verified_json(
    result_dir / "INT4-clean-L01_evaluation_v1.json",
    "2806756643d31d3a9713a27399266b75b90606f36e413e602ce6026ccfe48e90",
)
clean_hash = model_state_hash(layer_clean_model)
if clean_hash != clean_report["model_state_sha256"]:
    raise RuntimeError("Clean baseline state changed.")

selected = [
    s for s in main_plan["scenarios"]
    if s["scenario_id"] in ("N05-002", "N05-003")
]
if len(selected) != 2:
    raise RuntimeError("Expected two scenarios.")

for s in selected:
    sid = s["scenario_id"]
    if s["layer"] != layer_record["layer"]:
        raise RuntimeError("Scenario layer differs from the clean baseline.")

    prediction_path = result_dir / f"{sid}_predictions.json.gz"
    evaluation_path = result_dir / f"{sid}_evaluation_v1.json"

    expected_metadata = {
        "checkpoint_sha256": main_plan["checkpoint_sha256"],
        "quantization": layer_record,
        "fault": s,
        "clean_baseline_run_id": "INT4-clean-L01",
        "clean_baseline_state_sha256": clean_hash,
    }

    if not prediction_path.exists():
        if evaluation_path.exists():
            raise RuntimeError("Evaluation exists without its predictions.")

        faulty_model = copy.deepcopy(layer_clean_model).eval()
        key = "model." + s["layer"]
        target = dict(faulty_model.named_modules())[key]
        index = tuple(s["weight_index"])

        if target.weight[index].item() != s["clean_weight_fp32"]:
            raise RuntimeError("Clean weight differs from scenario.")

        before = int(s["q_clean"])
        bits = (before & 0xF) ^ (1 << int(s["bit"]))
        after = bits - 16 if bits >= 8 else bits
        if after != s["q_faulty"]:
            raise RuntimeError("Scenario bit transition is invalid.")

        with torch.no_grad():
            target.weight[index] = s["faulty_weight_fp32"]

        clean_state = layer_clean_model.state_dict()
        faulty_state = faulty_model.state_dict()
        changes = {
            name: int((tensor != faulty_state[name]).sum().item())
            for name, tensor in clean_state.items()
            if not torch.equal(tensor, faulty_state[name])
        }
        if changes != {key + ".weight": 1}:
            raise RuntimeError(f"Unexpected state differences: {changes}")

        print(f"\n=== Executing {sid} ===", flush=True)
        try:
            run_main_predictions(faulty_model, sid, expected_metadata)
        finally:
            del target, faulty_state, faulty_model
            if model_state_hash(layer_clean_model) != clean_hash:
                raise RuntimeError("Clean baseline was modified.")

    compressed = prediction_path.read_bytes()
    prediction_sha = hashlib.sha256(compressed).hexdigest()
    stored = json.loads(gzip.decompress(compressed))

    if stored["metadata"] != expected_metadata:
        raise RuntimeError(f"Stored prediction metadata mismatch: {sid}")
    if stored["image_ids"] != clean_report["image_ids"]:
        raise RuntimeError(f"Stored image list mismatch: {sid}")
    if stored["run_id"] != sid:
        raise RuntimeError("Stored run ID mismatch.")

    if evaluation_path.exists():
        evaluation = json.loads(evaluation_path.read_bytes())
        if (
            evaluation["prediction_file_sha256"] != prediction_sha
            or evaluation["metadata"] != expected_metadata
            or evaluation["image_ids"] != clean_report["image_ids"]
            or evaluation["model_state_sha256"] != stored["model_state_sha256"]
            or evaluation["status"] != "PREDICTIONS_AND_EVALUATION_COMPLETE"
        ):
            raise RuntimeError(f"Existing evaluation mismatch: {sid}")
        print("Verified existing evaluation:", sid)
    else:
        evaluation = evaluate_main_run(sid, prediction_sha)

    delta = (
        evaluation["metrics"]["AP50_95"]
        - clean_report["metrics"]["AP50_95"]
    )
    print(f"{sid}: faulty-minus-clean AP50:95 = {delta:+.9f}")

print("\nTwo-scenario batch completed.")
print("No held-out images evaluated. No commit or push performed.")


=== Executing N05-002 ===
N05-002: 100/1000
N05-002: 200/1000
N05-002: 300/1000
N05-002: 400/1000
N05-002: 500/1000
N05-002: 600/1000
N05-002: 700/1000
N05-002: 800/1000
N05-002: 900/1000
N05-002: 1000/1000
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook05/main_v1/N05-002_predictions.json.gz
SHA256: e36e1d4cc11108eb5b0b68f092e62469af3b8dd271870c9a51d617253fbcb6d4
Predictions: 114890
Wall time (seconds): 40.57
Model state unchanged: PASS
loading annotations into memory...
Done (t=0.47s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.22s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=10.19s).
Accumulating evaluation results...
DONE (t=1.69s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.401
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.545
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0

In [ ]:
import copy
import gzip
import hashlib
import json
import torch

verified_json(
    experiment_dir / "int4_main_analysis_protocol_v1.json",
    "7bc74fb60ac14ca0afaf4a8fb74646cae4fa57dc98259e99a72b1895305b1ab5",
)

clean_report = verified_json(
    result_dir / "INT4-clean-L01_evaluation_v1.json",
    "2806756643d31d3a9713a27399266b75b90606f36e413e602ce6026ccfe48e90",
)

clean_hash = model_state_hash(layer_clean_model)
if clean_hash != clean_report["model_state_sha256"]:
    raise RuntimeError("Clean baseline state changed.")

selected_ids = {f"N05-{number:03d}" for number in range(4, 9)}
selected = [
    s for s in main_plan["scenarios"]
    if s["scenario_id"] in selected_ids
]
if len(selected) != 5:
    raise RuntimeError("Expected five scenarios.")

for s in selected:
    sid = s["scenario_id"]
    if s["layer"] != layer_record["layer"]:
        raise RuntimeError("Scenario layer differs from the clean baseline.")

    prediction_path = result_dir / f"{sid}_predictions.json.gz"
    evaluation_path = result_dir / f"{sid}_evaluation_v1.json"

    expected_metadata = {
        "checkpoint_sha256": main_plan["checkpoint_sha256"],
        "quantization": layer_record,
        "fault": s,
        "clean_baseline_run_id": "INT4-clean-L01",
        "clean_baseline_state_sha256": clean_hash,
    }

    if not prediction_path.exists():
        if evaluation_path.exists():
            raise RuntimeError("Evaluation exists without its predictions.")

        faulty_model = copy.deepcopy(layer_clean_model).eval()
        key = "model." + s["layer"]
        target = dict(faulty_model.named_modules())[key]
        index = tuple(s["weight_index"])

        if target.weight[index].item() != s["clean_weight_fp32"]:
            raise RuntimeError("Clean weight differs from scenario.")

        before = int(s["q_clean"])
        bits = (before & 0xF) ^ (1 << int(s["bit"]))
        after = bits - 16 if bits >= 8 else bits
        if after != s["q_faulty"]:
            raise RuntimeError("Scenario bit transition is invalid.")

        with torch.no_grad():
            target.weight[index] = s["faulty_weight_fp32"]

        clean_state = layer_clean_model.state_dict()
        faulty_state = faulty_model.state_dict()
        changes = {
            name: int((tensor != faulty_state[name]).sum().item())
            for name, tensor in clean_state.items()
            if not torch.equal(tensor, faulty_state[name])
        }
        if changes != {key + ".weight": 1}:
            raise RuntimeError(f"Unexpected state differences: {changes}")

        print(f"\n=== Executing {sid} ===", flush=True)
        try:
            run_main_predictions(faulty_model, sid, expected_metadata)
        finally:
            del target, faulty_state, faulty_model
            if model_state_hash(layer_clean_model) != clean_hash:
                raise RuntimeError("Clean baseline was modified.")

    compressed = prediction_path.read_bytes()
    prediction_sha = hashlib.sha256(compressed).hexdigest()
    stored = json.loads(gzip.decompress(compressed))

    if stored["metadata"] != expected_metadata:
        raise RuntimeError(f"Stored prediction metadata mismatch: {sid}")
    if stored["image_ids"] != clean_report["image_ids"]:
        raise RuntimeError(f"Stored image list mismatch: {sid}")
    if stored["run_id"] != sid:
        raise RuntimeError("Stored run ID mismatch.")

    if evaluation_path.exists():
        evaluation = json.loads(evaluation_path.read_bytes())
        if (
            evaluation["prediction_file_sha256"] != prediction_sha
            or evaluation["metadata"] != expected_metadata
            or evaluation["image_ids"] != clean_report["image_ids"]
            or evaluation["model_state_sha256"] != stored["model_state_sha256"]
            or evaluation["status"] != "PREDICTIONS_AND_EVALUATION_COMPLETE"
        ):
            raise RuntimeError(f"Existing evaluation mismatch: {sid}")
        print("Verified existing evaluation:", sid)
    else:
        evaluation = evaluate_main_run(sid, prediction_sha)

    delta = (
        evaluation["metrics"]["AP50_95"]
        - clean_report["metrics"]["AP50_95"]
    )
    print(f"{sid}: faulty-minus-clean AP50:95 = {delta:+.9f}")

print("\nFive-scenario batch completed.")
print("No held-out images evaluated. No commit or push performed.")


=== Executing N05-004 ===
N05-004: 100/1000
N05-004: 200/1000
N05-004: 300/1000
N05-004: 400/1000
N05-004: 500/1000
N05-004: 600/1000
N05-004: 700/1000
N05-004: 800/1000
N05-004: 900/1000
N05-004: 1000/1000
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook05/main_v1/N05-004_predictions.json.gz
SHA256: b9b35d90ec1101106b0489061d3ba08e053cc9beccf49ed510eea0c2f200f0cd
Predictions: 114642
Wall time (seconds): 39.16
Model state unchanged: PASS
loading annotations into memory...
Done (t=0.50s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.13s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=11.32s).
Accumulating evaluation results...
DONE (t=1.77s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.400
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.545
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0

In [ ]:
import copy
import gzip
import hashlib
import io
import json
from contextlib import redirect_stdout

import torch

def run_first_layer_batch(scenario_ids):
    verified_json(
        experiment_dir / "int4_main_analysis_protocol_v1.json",
        "7bc74fb60ac14ca0afaf4a8fb74646cae4fa57dc98259e99a72b1895305b1ab5",
    )
    frozen_plan = verified_json(
        experiment_dir / "int4_main_fault_plan_v1.json",
        "906af255a7b46654f4736e4a3f77c8c778570ed3d86e5e12077bf9dbcc6f0fbb",
    )
    clean_report = verified_json(
        result_dir / "INT4-clean-L01_evaluation_v1.json",
        "2806756643d31d3a9713a27399266b75b90606f36e413e602ce6026ccfe48e90",
    )
    clean_hash = model_state_hash(layer_clean_model)
    if clean_hash != clean_report["model_state_sha256"]:
        raise RuntimeError("Clean baseline state changed.")
    if development_ids != clean_report["image_ids"]:
        raise RuntimeError("Development image list changed.")

    selected = [
        s for s in frozen_plan["scenarios"]
        if s["scenario_id"] in set(scenario_ids)
    ]
    if len(selected) != len(set(scenario_ids)):
        raise RuntimeError("Scenario selection mismatch.")

    for number, s in enumerate(selected, start=1):
        sid = s["scenario_id"]
        if s["layer"] != clean_report["metadata"]["quantization"]["layer"]:
            raise RuntimeError("This runner supports the first layer only.")

        prediction_path = result_dir / f"{sid}_predictions.json.gz"
        evaluation_path = result_dir / f"{sid}_evaluation_v1.json"
        metadata = {
            "checkpoint_sha256": frozen_plan["checkpoint_sha256"],
            "quantization": clean_report["metadata"]["quantization"],
            "fault": s,
            "clean_baseline_run_id": "INT4-clean-L01",
            "clean_baseline_state_sha256": clean_hash,
        }
        print(f"[{number}/{len(selected)}] Starting {sid}", flush=True)
        log = io.StringIO()

        try:
            with redirect_stdout(log):
                if not prediction_path.exists():
                    if evaluation_path.exists():
                        raise RuntimeError("Evaluation exists without predictions.")

                    faulty_model = copy.deepcopy(layer_clean_model).eval()
                    try:
                        key = "model." + s["layer"]
                        target = dict(faulty_model.named_modules())[key]
                        index = tuple(s["weight_index"])
                        if target.weight[index].item() != s["clean_weight_fp32"]:
                            raise RuntimeError("Clean weight mismatch.")

                        bits = (int(s["q_clean"]) & 0xF) ^ (1 << int(s["bit"]))
                        after = bits - 16 if bits >= 8 else bits
                        if after != s["q_faulty"]:
                            raise RuntimeError("Invalid bit transition.")

                        with torch.no_grad():
                            target.weight[index] = s["faulty_weight_fp32"]

                        a = layer_clean_model.state_dict()
                        b = faulty_model.state_dict()
                        changes = {
                            name: int((tensor != b[name]).sum().item())
                            for name, tensor in a.items()
                            if not torch.equal(tensor, b[name])
                        }
                        if changes != {key + ".weight": 1}:
                            raise RuntimeError("Single-weight isolation failed.")

                        run_main_predictions(faulty_model, sid, metadata)
                    finally:
                        del faulty_model
                        if model_state_hash(layer_clean_model) != clean_hash:
                            raise RuntimeError("Clean baseline was modified.")

                compressed = prediction_path.read_bytes()
                prediction_sha = hashlib.sha256(compressed).hexdigest()
                stored = json.loads(gzip.decompress(compressed))

                if (
                    stored["run_id"] != sid
                    or stored["metadata"] != metadata
                    or stored["image_ids"] != clean_report["image_ids"]
                    or stored["fault_plan_sha256"] != execution_protocol["fault_plan_sha256"]
                    or stored["execution_protocol_sha256"] != (
                        "3bf8913bb78ac1cb0b903a5e45caa0ff67ae867eb3a583991d76b4cf606a66ed"
                    )
                ):
                    raise RuntimeError("Stored prediction provenance mismatch.")

                if evaluation_path.exists():
                    evaluation = json.loads(evaluation_path.read_bytes())
                    if (
                        evaluation["prediction_file_sha256"] != prediction_sha
                        or evaluation["metadata"] != metadata
                        or evaluation["image_ids"] != stored["image_ids"]
                        or evaluation["model_state_sha256"] != stored["model_state_sha256"]
                        or evaluation["run_id"] != sid
                        or evaluation["status"] != "PREDICTIONS_AND_EVALUATION_COMPLETE"
                    ):
                        raise RuntimeError("Stored evaluation mismatch.")
                else:
                    evaluation = evaluate_main_run(sid, prediction_sha)

            delta = (
                evaluation["metrics"]["AP50_95"]
                - clean_report["metrics"]["AP50_95"]
            )
            print(f"{sid}: COMPLETE | AP change={delta:+.9f}", flush=True)
        except Exception:
            print(log.getvalue()[-4000:])
            print(f"STOPPED at {sid}; completed files were preserved.")
            raise

    print("Batch complete. No held-out images evaluated. No commit or push.")

run_first_layer_batch([
    f"N05-{number:03d}" for number in range(9, 21)
])

[1/12] Starting N05-009
N05-009: COMPLETE | AP change=+0.000042457
[2/12] Starting N05-010
N05-010: COMPLETE | AP change=+0.000106628
[3/12] Starting N05-011
N05-011: COMPLETE | AP change=+0.007232614
[4/12] Starting N05-012
N05-012: COMPLETE | AP change=+0.000090489
[5/12] Starting N05-013
N05-013: COMPLETE | AP change=-0.000432501
[6/12] Starting N05-014
N05-014: COMPLETE | AP change=-0.000430254
[7/12] Starting N05-015
N05-015: COMPLETE | AP change=+0.000301582
[8/12] Starting N05-016
N05-016: COMPLETE | AP change=+0.000217488
[9/12] Starting N05-017
N05-017: COMPLETE | AP change=+0.003602109
[10/12] Starting N05-018
N05-018: COMPLETE | AP change=+0.001837356
[11/12] Starting N05-019
N05-019: COMPLETE | AP change=+0.000329558
[12/12] Starting N05-020
N05-020: COMPLETE | AP change=-0.010882609
Batch complete. No held-out images evaluated. No commit or push.


In [ ]:
import gzip
import hashlib
import json
import numpy as np

output = result_dir / "L01_sensitivity_summary_v1.json"
if output.exists():
    raise FileExistsError("Layer summary already exists.")

analysis = verified_json(
    experiment_dir / "int4_main_analysis_protocol_v1.json",
    "7bc74fb60ac14ca0afaf4a8fb74646cae4fa57dc98259e99a72b1895305b1ab5",
)
frozen_plan = verified_json(
    experiment_dir / "int4_main_fault_plan_v1.json",
    "906af255a7b46654f4736e4a3f77c8c778570ed3d86e5e12077bf9dbcc6f0fbb",
)
clean = verified_json(
    result_dir / "INT4-clean-L01_evaluation_v1.json",
    "2806756643d31d3a9713a27399266b75b90606f36e413e602ce6026ccfe48e90",
)

layer_name = clean["metadata"]["quantization"]["layer"]
selected = [s for s in frozen_plan["scenarios"] if s["layer"] == layer_name]
if len(selected) != 20:
    raise RuntimeError("Expected 20 planned scenarios.")

rows = []
for s in selected:
    sid = s["scenario_id"]
    prediction_path = result_dir / f"{sid}_predictions.json.gz"
    evaluation_path = result_dir / f"{sid}_evaluation_v1.json"

    compressed = prediction_path.read_bytes()
    prediction_sha = hashlib.sha256(compressed).hexdigest()
    stored = json.loads(gzip.decompress(compressed))
    evaluation_bytes = evaluation_path.read_bytes()
    evaluation = json.loads(evaluation_bytes)

    for item in (stored, evaluation):
        if (
            item["run_id"] != sid
            or item["image_ids"] != clean["image_ids"]
            or item["metadata"]["fault"] != s
            or item["metadata"]["quantization"] != clean["metadata"]["quantization"]
            or item["metadata"]["clean_baseline_state_sha256"]
                != clean["model_state_sha256"]
            or item["fault_plan_sha256"] != clean["fault_plan_sha256"]
            or item["execution_protocol_sha256"]
                != clean["execution_protocol_sha256"]
        ):
            raise RuntimeError(f"Provenance mismatch: {sid}")

    if (
        evaluation["prediction_file_sha256"] != prediction_sha
        or evaluation["model_state_sha256"] != stored["model_state_sha256"]
        or evaluation["status"] != "PREDICTIONS_AND_EVALUATION_COMPLETE"
    ):
        raise RuntimeError(f"Completion/integrity mismatch: {sid}")

    if [r["image_id"] for r in stored["per_image"]] != clean["image_ids"]:
        raise RuntimeError(f"Incomplete image record: {sid}")

    rows.append({
        "scenario_id": sid,
        "bit": s["bit"],
        "AP50_95_change": (
            evaluation["metrics"]["AP50_95"] - clean["metrics"]["AP50_95"]
        ),
        "prediction_sha256": prediction_sha,
        "evaluation_sha256": hashlib.sha256(evaluation_bytes).hexdigest(),
    })

groups = {
    bit: np.array([r["AP50_95_change"] for r in rows if r["bit"] == bit])
    for bit in range(4)
}
if any(len(values) != 5 for values in groups.values()):
    raise RuntimeError("Expected five scenarios per bit.")

rng = np.random.default_rng(analysis["fault_sampling_uncertainty"]["seed"])
bootstrap = np.mean([
    rng.choice(values, size=(2000, 5), replace=True).mean(axis=1)
    for values in groups.values()
], axis=0)

effects = np.array([r["AP50_95_change"] for r in rows])
summary = {
    "layer": layer_name,
    "scenario_count": len(rows),
    "development_image_count": len(clean["image_ids"]),
    "clean_AP50_95": clean["metrics"]["AP50_95"],
    "mean_AP50_95_change": float(effects.mean()),
    "median_AP50_95_change": float(np.median(effects)),
    "minimum_AP50_95_change": float(effects.min()),
    "maximum_AP50_95_change": float(effects.max()),
    "effect_counts": {
        "positive": int((effects > 0).sum()),
        "negative": int((effects < 0).sum()),
        "exact_zero": int((effects == 0).sum()),
    },
    "per_bit": {
        str(bit): {
            "count": len(values),
            "mean": float(values.mean()),
            "minimum": float(values.min()),
            "maximum": float(values.max()),
        }
        for bit, values in groups.items()
    },
    "exploratory_bootstrap_percentile_interval": (
        np.percentile(bootstrap, [2.5, 97.5]).tolist()
    ),
    "analysis_protocol_sha256": (
        "7bc74fb60ac14ca0afaf4a8fb74646cae4fa57dc98259e99a72b1895305b1ab5"
    ),
    "scenarios": rows,
    "limitations": analysis["fault_sampling_uncertainty"]["limitations"],
}

content = (json.dumps(summary, indent=2, allow_nan=False) + "\n").encode()
with output.open("xb") as stream:
    stream.write(content)

for key in (
    "layer", "scenario_count", "mean_AP50_95_change",
    "median_AP50_95_change", "minimum_AP50_95_change",
    "maximum_AP50_95_change", "effect_counts",
    "exploratory_bootstrap_percentile_interval",
):
    print(key, ":", summary[key])
print("Per bit:", summary["per_bit"])
print("Saved:", output)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("No new inference. No held-out evaluation. No commit or push.")

layer : 1.conv
scenario_count : 20
mean_AP50_95_change : 1.542638191432133e-05
median_AP50_95_change : 0.000179052982409722
minimum_AP50_95_change : -0.010882609379680108
maximum_AP50_95_change : 0.007232614099263235
effect_counts : {'positive': 14, 'negative': 6, 'exact_zero': 0}
exploratory_bootstrap_percentile_interval : [-0.001505224457078491, 0.0012777916122809273]
Per bit: {'0': {'count': 5, 'mean': 7.817562231976494e-05, 'minimum': -0.0007846703147473266, 'maximum': 0.00041585327909721137}, '1': {'count': 5, 'mean': -0.0003896362674363441, 'minimum': -0.002226191567530289, 'maximum': 0.00020533113977999617}, '2': {'count': 5, 'mean': 0.001352385989049365, 'minimum': -0.00043250141256617747, 'maximum': 0.007232614099263235}, '3': {'count': 5, 'mean': -0.0009792198162755007, 'minimum': -0.010882609379680108, 'maximum': 0.0036021089964096697}}
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook05/main_v1/L01_sensitivity_summary_v1.json
SHA256: c851bc9eae00f7b

In [ ]:
import hashlib
import torch
from ultralytics import YOLO

frozen_plan = verified_json(
    experiment_dir / "int4_main_fault_plan_v1.json",
    "906af255a7b46654f4736e4a3f77c8c778570ed3d86e5e12077bf9dbcc6f0fbb",
)
layer2_record = frozen_plan["layers"][1]
if layer2_record["layer"] != "4.cv1.conv":
    raise RuntimeError("Unexpected second layer.")

checkpoint = project / "models/yolov8s.pt"
if hashlib.sha256(checkpoint.read_bytes()).hexdigest() != (
    frozen_plan["checkpoint_sha256"]
):
    raise RuntimeError("Checkpoint identity mismatch.")

layer2_yolo = YOLO(str(checkpoint))
layer2_clean_model = layer2_yolo.model.cpu().float().eval()
key = "model." + layer2_record["layer"]
target = dict(layer2_clean_model.named_modules())[key]

if not isinstance(target, torch.nn.Conv2d):
    raise RuntimeError("Expected Conv2d.")
if list(target.weight.shape) != layer2_record["weight_shape"]:
    raise RuntimeError("Layer shape mismatch.")

original_state = {
    name: tensor.detach().clone()
    for name, tensor in layer2_clean_model.state_dict().items()
}

with torch.no_grad():
    scale_tensor = target.weight.detach().abs().max() / 7
    if scale_tensor.item() != layer2_record["scale"]:
        raise RuntimeError("Scale differs from the frozen plan.")

    q = torch.round(
        target.weight.detach() / scale_tensor
    ).clamp(-7, 7).to(torch.int8)
    target.weight.copy_(q.float() * scale_tensor)

for name, tensor in layer2_clean_model.state_dict().items():
    if name != key + ".weight" and not torch.equal(
        tensor, original_state[name]
    ):
        raise RuntimeError(f"Unexpected state change: {name}")

if not torch.equal(target.weight, q.float() * scale_tensor):
    raise RuntimeError("Quantized weight verification failed.")

del original_state
layer2_clean_model.to("cuda:0").eval()

print("Target layer:", layer2_record["layer"])
print("Target-layer-only quantization: PASS")

layer2_prediction_path = run_main_predictions(
    layer2_clean_model,
    "INT4-clean-L02",
    {
        "checkpoint_sha256": frozen_plan["checkpoint_sha256"],
        "quantization": layer2_record,
        "fault": None,
    },
)

Target layer: 4.cv1.conv
Target-layer-only quantization: PASS
INT4-clean-L02: 100/1000
INT4-clean-L02: 200/1000
INT4-clean-L02: 300/1000
INT4-clean-L02: 400/1000
INT4-clean-L02: 500/1000
INT4-clean-L02: 600/1000
INT4-clean-L02: 700/1000
INT4-clean-L02: 800/1000
INT4-clean-L02: 900/1000
INT4-clean-L02: 1000/1000
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook05/main_v1/INT4-clean-L02_predictions.json.gz
SHA256: e4b37dbab72b09c8615781ce6991dba094095f2d1ba6f3f524292d1e6c9449fe
Predictions: 112925
Wall time (seconds): 43.29
Model state unchanged: PASS


In [ ]:
layer2_clean_evaluation = evaluate_main_run(
    "INT4-clean-L02",
    "e4b37dbab72b09c8615781ce6991dba094095f2d1ba6f3f524292d1e6c9449fe",
)

fp32_evaluation = verified_json(
    result_dir / "FP32-clean_evaluation_v1.json",
    "d4b7a578dd31a2aa756083552f1a4a2f2472a7e570c412fedada81a9f86f3bff",
)

print(
    "\nQuantization AP change (L02 INT4 clean - FP32):",
    layer2_clean_evaluation["metrics"]["AP50_95"]
    - fp32_evaluation["metrics"]["AP50_95"],
)
print("No fault injected. No new inference performed.")

loading annotations into memory...
Done (t=0.82s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.13s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=10.63s).
Accumulating evaluation results...
DONE (t=1.78s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.438
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.592
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.479
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.232
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.493
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.610
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.340
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.547
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDet

In [ ]:
import copy
import gzip
import hashlib
import io
import json
from contextlib import redirect_stdout

import torch

def run_layer_batch(clean_model, baseline_id, baseline_eval_sha, scenario_ids):
    verified_json(
        experiment_dir / "int4_main_analysis_protocol_v1.json",
        "7bc74fb60ac14ca0afaf4a8fb74646cae4fa57dc98259e99a72b1895305b1ab5",
    )
    frozen_plan = verified_json(
        experiment_dir / "int4_main_fault_plan_v1.json",
        "906af255a7b46654f4736e4a3f77c8c778570ed3d86e5e12077bf9dbcc6f0fbb",
    )
    clean_report = verified_json(
        result_dir / f"{baseline_id}_evaluation_v1.json",
        baseline_eval_sha,
    )
    clean_hash = model_state_hash(clean_model)
    if clean_hash != clean_report["model_state_sha256"]:
        raise RuntimeError("Clean baseline state changed.")
    if development_ids != clean_report["image_ids"]:
        raise RuntimeError("Development image list changed.")

    wanted = set(scenario_ids)
    selected = [
        s for s in frozen_plan["scenarios"] if s["scenario_id"] in wanted
    ]
    if len(selected) != len(wanted):
        raise RuntimeError("Scenario selection mismatch.")

    for number, s in enumerate(selected, start=1):
        sid = s["scenario_id"]
        if s["layer"] != clean_report["metadata"]["quantization"]["layer"]:
            raise RuntimeError("Scenario and baseline layers differ.")

        prediction_path = result_dir / f"{sid}_predictions.json.gz"
        evaluation_path = result_dir / f"{sid}_evaluation_v1.json"
        metadata = {
            "checkpoint_sha256": frozen_plan["checkpoint_sha256"],
            "quantization": clean_report["metadata"]["quantization"],
            "fault": s,
            "clean_baseline_run_id": baseline_id,
            "clean_baseline_state_sha256": clean_hash,
        }

        print(f"[{number}/{len(selected)}] Starting {sid}", flush=True)
        log = io.StringIO()

        try:
            with redirect_stdout(log):
                if not prediction_path.exists():
                    if evaluation_path.exists():
                        raise RuntimeError("Evaluation exists without predictions.")

                    faulty_model = copy.deepcopy(clean_model).eval()
                    try:
                        key = "model." + s["layer"]
                        target = dict(faulty_model.named_modules())[key]
                        index = tuple(s["weight_index"])

                        if target.weight[index].item() != s["clean_weight_fp32"]:
                            raise RuntimeError("Clean weight mismatch.")

                        bits = (int(s["q_clean"]) & 0xF) ^ (1 << int(s["bit"]))
                        after = bits - 16 if bits >= 8 else bits
                        if after != s["q_faulty"]:
                            raise RuntimeError("Invalid bit transition.")

                        with torch.no_grad():
                            target.weight[index] = s["faulty_weight_fp32"]

                        a = clean_model.state_dict()
                        b = faulty_model.state_dict()
                        changes = {
                            name: int((tensor != b[name]).sum().item())
                            for name, tensor in a.items()
                            if not torch.equal(tensor, b[name])
                        }
                        if changes != {key + ".weight": 1}:
                            raise RuntimeError("Single-weight isolation failed.")

                        run_main_predictions(faulty_model, sid, metadata)
                    finally:
                        del faulty_model
                        if model_state_hash(clean_model) != clean_hash:
                            raise RuntimeError("Clean baseline was modified.")

                compressed = prediction_path.read_bytes()
                prediction_sha = hashlib.sha256(compressed).hexdigest()
                stored = json.loads(gzip.decompress(compressed))

                if (
                    stored["run_id"] != sid
                    or stored["metadata"] != metadata
                    or stored["image_ids"] != clean_report["image_ids"]
                    or stored["fault_plan_sha256"] != clean_report["fault_plan_sha256"]
                    or stored["execution_protocol_sha256"]
                        != clean_report["execution_protocol_sha256"]
                ):
                    raise RuntimeError("Stored prediction provenance mismatch.")

                if evaluation_path.exists():
                    evaluation = json.loads(evaluation_path.read_bytes())
                    if (
                        evaluation["prediction_file_sha256"] != prediction_sha
                        or evaluation["metadata"] != metadata
                        or evaluation["image_ids"] != stored["image_ids"]
                        or evaluation["model_state_sha256"] != stored["model_state_sha256"]
                        or evaluation["run_id"] != sid
                        or evaluation["status"] != "PREDICTIONS_AND_EVALUATION_COMPLETE"
                        or evaluation["fault_plan_sha256"] != stored["fault_plan_sha256"]
                        or evaluation["execution_protocol_sha256"]
                            != stored["execution_protocol_sha256"]
                    ):
                        raise RuntimeError("Stored evaluation mismatch.")
                else:
                    evaluation = evaluate_main_run(sid, prediction_sha)

            delta = (
                evaluation["metrics"]["AP50_95"]
                - clean_report["metrics"]["AP50_95"]
            )
            print(f"{sid}: COMPLETE | AP change={delta:+.9f}", flush=True)

        except Exception:
            print(log.getvalue()[-4000:])
            print(f"STOPPED at {sid}; completed files were preserved.")
            raise

    print("Batch complete. No held-out images evaluated. No commit or push.")

run_layer_batch(
    layer2_clean_model,
    "INT4-clean-L02",
    "e664c86cc65f39b0ceab4ace926fe861bccc39aeda899b11cc9c61cf630b7806",
    [
        s["scenario_id"] for s in main_plan["scenarios"]
        if s["layer"] == "4.cv1.conv"
    ],
)

[1/20] Starting N05-021
N05-021: COMPLETE | AP change=+0.000052005
[2/20] Starting N05-022
N05-022: COMPLETE | AP change=-0.000031404
[3/20] Starting N05-023
N05-023: COMPLETE | AP change=-0.000446147
[4/20] Starting N05-024
N05-024: COMPLETE | AP change=-0.000007088
[5/20] Starting N05-025
N05-025: COMPLETE | AP change=+0.000124111
[6/20] Starting N05-026
N05-026: COMPLETE | AP change=-0.000190022
[7/20] Starting N05-027
N05-027: COMPLETE | AP change=+0.000358437
[8/20] Starting N05-028
N05-028: COMPLETE | AP change=-0.000070473
[9/20] Starting N05-029
N05-029: COMPLETE | AP change=-0.000301771
[10/20] Starting N05-030
N05-030: COMPLETE | AP change=+0.000153086
[11/20] Starting N05-031
N05-031: COMPLETE | AP change=-0.000223720
[12/20] Starting N05-032
N05-032: COMPLETE | AP change=-0.000206105
[13/20] Starting N05-033
N05-033: COMPLETE | AP change=+0.000367664
[14/20] Starting N05-034
N05-034: COMPLETE | AP change=+0.000068310
[15/20] Starting N05-035
N05-035: COMPLETE | AP change=-0

In [ ]:
import gzip
import hashlib
import json
import numpy as np

def summarize_main_layer(baseline_id, baseline_sha, output_name):
    output = result_dir / output_name
    if output.exists():
        raise FileExistsError("Layer summary already exists.")

    analysis = verified_json(
        experiment_dir / "int4_main_analysis_protocol_v1.json",
        "7bc74fb60ac14ca0afaf4a8fb74646cae4fa57dc98259e99a72b1895305b1ab5",
    )
    frozen_plan = verified_json(
        experiment_dir / "int4_main_fault_plan_v1.json",
        "906af255a7b46654f4736e4a3f77c8c778570ed3d86e5e12077bf9dbcc6f0fbb",
    )
    clean = verified_json(
        result_dir / f"{baseline_id}_evaluation_v1.json", baseline_sha
    )
    layer_name = clean["metadata"]["quantization"]["layer"]
    selected = [
        s for s in frozen_plan["scenarios"] if s["layer"] == layer_name
    ]
    if len(selected) != 20:
        raise RuntimeError("Expected 20 planned scenarios.")

    rows = []
    for s in selected:
        sid = s["scenario_id"]
        compressed = (
            result_dir / f"{sid}_predictions.json.gz"
        ).read_bytes()
        prediction_sha = hashlib.sha256(compressed).hexdigest()
        stored = json.loads(gzip.decompress(compressed))

        evaluation_bytes = (
            result_dir / f"{sid}_evaluation_v1.json"
        ).read_bytes()
        evaluation = json.loads(evaluation_bytes)

        for item in (stored, evaluation):
            if (
                item["run_id"] != sid
                or item["image_ids"] != clean["image_ids"]
                or item["metadata"]["fault"] != s
                or item["metadata"]["quantization"] != clean["metadata"]["quantization"]
                or item["metadata"]["clean_baseline_run_id"] != baseline_id
                or item["metadata"]["clean_baseline_state_sha256"]
                    != clean["model_state_sha256"]
                or item["fault_plan_sha256"] != clean["fault_plan_sha256"]
                or item["execution_protocol_sha256"]
                    != clean["execution_protocol_sha256"]
            ):
                raise RuntimeError(f"Provenance mismatch: {sid}")

        if (
            evaluation["prediction_file_sha256"] != prediction_sha
            or evaluation["model_state_sha256"] != stored["model_state_sha256"]
            or evaluation["status"] != "PREDICTIONS_AND_EVALUATION_COMPLETE"
        ):
            raise RuntimeError(f"Completion/integrity mismatch: {sid}")

        if (
            [r["image_id"] for r in stored["per_image"]] != clean["image_ids"]
            or sum(r["prediction_count"] for r in stored["per_image"])
                != len(stored["predictions"])
        ):
            raise RuntimeError(f"Incomplete execution record: {sid}")

        rows.append({
            "scenario_id": sid,
            "bit": s["bit"],
            "AP50_95_change": (
                evaluation["metrics"]["AP50_95"]
                - clean["metrics"]["AP50_95"]
            ),
            "prediction_sha256": prediction_sha,
            "evaluation_sha256": hashlib.sha256(evaluation_bytes).hexdigest(),
        })

    groups = {
        bit: np.array([
            r["AP50_95_change"] for r in rows if r["bit"] == bit
        ])
        for bit in range(4)
    }
    if any(len(values) != 5 for values in groups.values()):
        raise RuntimeError("Expected five scenarios per bit.")

    uncertainty = analysis["fault_sampling_uncertainty"]
    rng = np.random.default_rng(uncertainty["seed"])
    bootstrap = np.mean([
        rng.choice(
            values, size=(uncertainty["resamples"], 5), replace=True
        ).mean(axis=1)
        for values in groups.values()
    ], axis=0)

    effects = np.array([r["AP50_95_change"] for r in rows])
    if not np.isfinite(effects).all():
        raise RuntimeError("Non-finite effects.")

    summary = {
        "layer": layer_name,
        "scenario_count": len(rows),
        "development_image_count": len(clean["image_ids"]),
        "clean_AP50_95": clean["metrics"]["AP50_95"],
        "mean_AP50_95_change": float(effects.mean()),
        "median_AP50_95_change": float(np.median(effects)),
        "minimum_AP50_95_change": float(effects.min()),
        "maximum_AP50_95_change": float(effects.max()),
        "effect_counts": {
            "positive": int((effects > 0).sum()),
            "negative": int((effects < 0).sum()),
            "exact_zero": int((effects == 0).sum()),
        },
        "per_bit": {
            str(bit): {
                "count": len(values),
                "mean": float(values.mean()),
                "minimum": float(values.min()),
                "maximum": float(values.max()),
            }
            for bit, values in groups.items()
        },
        "exploratory_bootstrap_percentile_interval": np.percentile(
            bootstrap, uncertainty["interval_percentiles"]
        ).tolist(),
        "clean_evaluation_sha256": baseline_sha,
        "analysis_protocol_sha256": (
            "7bc74fb60ac14ca0afaf4a8fb74646cae4fa57dc98259e99a72b1895305b1ab5"
        ),
        "scenarios": rows,
        "limitations": uncertainty["limitations"],
    }

    content = (json.dumps(summary, indent=2, allow_nan=False) + "\n").encode()
    with output.open("xb") as stream:
        stream.write(content)

    for key in (
        "layer", "scenario_count", "mean_AP50_95_change",
        "median_AP50_95_change", "minimum_AP50_95_change",
        "maximum_AP50_95_change", "effect_counts",
        "exploratory_bootstrap_percentile_interval",
    ):
        print(key, ":", summary[key])
    print("Per bit:", summary["per_bit"])
    print("Saved:", output)
    print("SHA256:", hashlib.sha256(content).hexdigest())
    return summary

layer2_summary = summarize_main_layer(
    "INT4-clean-L02",
    "e664c86cc65f39b0ceab4ace926fe861bccc39aeda899b11cc9c61cf630b7806",
    "L02_sensitivity_summary_v1.json",
)

print("No new inference. No held-out evaluation. No commit or push.")

layer : 4.cv1.conv
scenario_count : 20
mean_AP50_95_change : -0.00020361754481948435
median_AP50_95_change : -5.093868552699665e-05
minimum_AP50_95_change : -0.002322965134629984
maximum_AP50_95_change : 0.0004605894243230324
effect_counts : {'positive': 8, 'negative': 12, 'exact_zero': 0}
exploratory_bootstrap_percentile_interval : [-0.0004514817772363855, 2.9043917610548543e-05]
Per bit: {'0': {'count': 5, 'mean': -6.170467293090676e-05, 'minimum': -0.00044614676461252056, 'maximum': 0.00012411067015932753}, '1': {'count': 5, 'mean': -1.0148711292934376e-05, 'minimum': -0.0003017714444751296, 'maximum': 0.00035843685046099694}, '2': {'count': 5, 'mean': -2.5465443832362045e-05, 'minimum': -0.00022372015645466536, 'maximum': 0.0003676640836937084}, '3': {'count': 5, 'mean': -0.0007171513512217343, 'minimum': -0.002322965134629984, 'maximum': 0.0004605894243230324}}
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook05/main_v1/L02_sensitivity_summary_v1.json
SHA2

In [ ]:
import hashlib
import torch
from ultralytics import YOLO

frozen_plan = verified_json(
    experiment_dir / "int4_main_fault_plan_v1.json",
    "906af255a7b46654f4736e4a3f77c8c778570ed3d86e5e12077bf9dbcc6f0fbb",
)
layer3_record = frozen_plan["layers"][2]
if layer3_record["layer"] != "4.m.1.cv2.conv":
    raise RuntimeError("Unexpected third layer.")

checkpoint = project / "models/yolov8s.pt"
if hashlib.sha256(checkpoint.read_bytes()).hexdigest() != (
    frozen_plan["checkpoint_sha256"]
):
    raise RuntimeError("Checkpoint identity mismatch.")

layer3_yolo = YOLO(str(checkpoint))
layer3_clean_model = layer3_yolo.model.cpu().float().eval()
key = "model." + layer3_record["layer"]
target = dict(layer3_clean_model.named_modules())[key]

if not isinstance(target, torch.nn.Conv2d):
    raise RuntimeError("Expected Conv2d.")
if list(target.weight.shape) != layer3_record["weight_shape"]:
    raise RuntimeError("Layer shape mismatch.")

original_state = {
    name: tensor.detach().clone()
    for name, tensor in layer3_clean_model.state_dict().items()
}

with torch.no_grad():
    scale_tensor = target.weight.detach().abs().max() / 7
    if scale_tensor.item() != layer3_record["scale"]:
        raise RuntimeError("Scale differs from the frozen plan.")

    q = torch.round(
        target.weight.detach() / scale_tensor
    ).clamp(-7, 7).to(torch.int8)
    target.weight.copy_(q.float() * scale_tensor)

for name, tensor in layer3_clean_model.state_dict().items():
    if name != key + ".weight" and not torch.equal(
        tensor, original_state[name]
    ):
        raise RuntimeError(f"Unexpected state change: {name}")

if not torch.equal(target.weight, q.float() * scale_tensor):
    raise RuntimeError("Quantized weight verification failed.")

del original_state
layer3_clean_model.to("cuda:0").eval()

print("Target layer:", layer3_record["layer"])
print("Target-layer-only quantization: PASS")

layer3_prediction_path = run_main_predictions(
    layer3_clean_model,
    "INT4-clean-L03",
    {
        "checkpoint_sha256": frozen_plan["checkpoint_sha256"],
        "quantization": layer3_record,
        "fault": None,
    },
)

Target layer: 4.m.1.cv2.conv
Target-layer-only quantization: PASS
INT4-clean-L03: 100/1000
INT4-clean-L03: 200/1000
INT4-clean-L03: 300/1000
INT4-clean-L03: 400/1000
INT4-clean-L03: 500/1000
INT4-clean-L03: 600/1000
INT4-clean-L03: 700/1000
INT4-clean-L03: 800/1000
INT4-clean-L03: 900/1000
INT4-clean-L03: 1000/1000
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook05/main_v1/INT4-clean-L03_predictions.json.gz
SHA256: 7770b255f7c9e70962cb44bdf2037222d2c05902c95d566373f979ed257c22ec
Predictions: 108741
Wall time (seconds): 41.1
Model state unchanged: PASS


In [ ]:
layer3_clean_evaluation = evaluate_main_run(
    "INT4-clean-L03",
    "7770b255f7c9e70962cb44bdf2037222d2c05902c95d566373f979ed257c22ec",
)

fp32_evaluation = verified_json(
    result_dir / "FP32-clean_evaluation_v1.json",
    "d4b7a578dd31a2aa756083552f1a4a2f2472a7e570c412fedada81a9f86f3bff",
)

print(
    "\nQuantization AP change (L03 INT4 clean - FP32):",
    layer3_clean_evaluation["metrics"]["AP50_95"]
    - fp32_evaluation["metrics"]["AP50_95"],
)
print("No fault injected. No new inference performed.")

loading annotations into memory...
Done (t=1.54s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.15s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=7.80s).
Accumulating evaluation results...
DONE (t=2.42s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.457
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.615
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.499
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.254
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.507
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.637
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.355
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.566
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets

In [ ]:
run_layer_batch(
    layer3_clean_model,
    "INT4-clean-L03",
    "ac404c35185bd6b20129e97cbff817a822e1fa9a3291d01a73dcf495ebddd2de",
    [
        s["scenario_id"] for s in main_plan["scenarios"]
        if s["layer"] == "4.m.1.cv2.conv"
    ],
)

[1/20] Starting N05-041
N05-041: COMPLETE | AP change=+0.000225910
[2/20] Starting N05-042
N05-042: COMPLETE | AP change=+0.000123598
[3/20] Starting N05-043
N05-043: COMPLETE | AP change=-0.000099230
[4/20] Starting N05-044
N05-044: COMPLETE | AP change=+0.000077275
[5/20] Starting N05-045
N05-045: COMPLETE | AP change=+0.000110198
[6/20] Starting N05-046
N05-046: COMPLETE | AP change=-0.000034098
[7/20] Starting N05-047
N05-047: COMPLETE | AP change=-0.000278105
[8/20] Starting N05-048
N05-048: COMPLETE | AP change=-0.000122451
[9/20] Starting N05-049
N05-049: COMPLETE | AP change=+0.000182738
[10/20] Starting N05-050
N05-050: COMPLETE | AP change=+0.000040365
[11/20] Starting N05-051
N05-051: COMPLETE | AP change=-0.000105151
[12/20] Starting N05-052
N05-052: COMPLETE | AP change=+0.000135386
[13/20] Starting N05-053
N05-053: COMPLETE | AP change=+0.000064576
[14/20] Starting N05-054
N05-054: COMPLETE | AP change=+0.000175707
[15/20] Starting N05-055
N05-055: COMPLETE | AP change=+0

In [ ]:
layer3_summary = summarize_main_layer(
    "INT4-clean-L03",
    "ac404c35185bd6b20129e97cbff817a822e1fa9a3291d01a73dcf495ebddd2de",
    "L03_sensitivity_summary_v1.json",
)

print("No new inference. No held-out evaluation.")
print("Further layer execution is paused pending experimental-design review.")

layer : 4.m.1.cv2.conv
scenario_count : 20
mean_AP50_95_change : -1.3954578367508797e-05
median_AP50_95_change : 7.09257475886016e-05
minimum_AP50_95_change : -0.0008272835064918604
maximum_AP50_95_change : 0.0003050642491685407
effect_counts : {'positive': 12, 'negative': 8, 'exact_zero': 0}
exploratory_bootstrap_percentile_interval : [-0.00012063401635111229, 8.050982766144098e-05]
Per bit: {'0': {'count': 5, 'mean': 8.755015254594545e-05, 'minimum': -9.923037511117005e-05, 'maximum': 0.00022591013128309134}, '1': {'count': 5, 'mean': -4.2310212349905285e-05, 'minimum': -0.000278104740052576, 'maximum': 0.00018273782347244527}, '2': {'count': 5, 'mean': 7.18595362500074e-05, 'minimum': -0.00010515061177518437, 'maximum': 0.0001757066071348401}, '3': {'count': 5, 'mean': -0.00017291778991608276, 'minimum': -0.0008272835064918604, 'maximum': 0.0003050642491685407}}
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook05/main_v1/L03_sensitivity_summary_v1.json
SHA25

In [2]:
from pathlib import Path
import hashlib
import json

if not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

project = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")
result_dir = project / "code/results/notebook05/main_v1"

expected = {
    "L01_sensitivity_summary_v1.json":
        "c851bc9eae00f7bdfa2516c2cab8e5a56c477801fdaec2c0b0e3b8fb5926239b",
    "L02_sensitivity_summary_v1.json":
        "28e81f185dbdaf53dbeb8f2355188ac0bd5a71d0c278014edefc06c28373969a",
    "L03_sensitivity_summary_v1.json":
        "cc65fdabc9a7c8c9999ee12fdad526474f778a15e26de4a346f873fd23142136",
}

for filename, expected_hash in expected.items():
    content = (result_dir / filename).read_bytes()
    if hashlib.sha256(content).hexdigest() != expected_hash:
        raise RuntimeError(f"Integrity mismatch: {filename}")

    summary = json.loads(content)
    if summary["scenario_count"] != 20:
        raise RuntimeError(f"Unexpected scenario count: {filename}")

    print(
        f"PASS: {summary['layer']} | "
        f"scenarios={summary['scenario_count']} | "
        f"mean AP change={summary['mean_AP50_95_change']:+.9f}"
    )

print("Saved summaries verified. No inference or files modified.")

Mounted at /content/drive
PASS: 1.conv | scenarios=20 | mean AP change=+0.000015426
PASS: 4.cv1.conv | scenarios=20 | mean AP change=-0.000203618
PASS: 4.m.1.cv2.conv | scenarios=20 | mean AP change=-0.000013955
Saved summaries verified. No inference or files modified.


In [3]:
import json
from pathlib import Path

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
folder = repo / "notebooks"

terms = (
    "selected_layers",
    "layer_sensitivity_plan",
)

matches = 0

for path in sorted(folder.glob("04*.ipynb")):
    notebook = json.loads(path.read_text(encoding="utf-8"))

    for number, cell in enumerate(notebook.get("cells", []), start=1):
        source = cell.get("source", [])
        source = source if isinstance(source, str) else "".join(source)

        if any(term in source for term in terms):
            matches += 1
            print(f"\n--- {path.name} | cell {number} ---")
            if len(source) <= 12000:
                print(source)
            else:
                lines = source.splitlines()
                indices = [
                    i for i, line in enumerate(lines)
                    if any(term in line for term in terms)
                ]
                shown = set()
                for index in indices:
                    for i in range(max(0, index - 15), min(len(lines), index + 25)):
                        if i not in shown:
                            print(f"{i + 1}: {lines[i]}")
                            shown.add(i)
                print("[Long cell: matching sections displayed only.]")

print("\nMatching cells:", matches)
print("Source inspection only. No inference or files modified.")


--- 04b_reproducibility_audit.ipynb | cell 26 ---
import json
from pathlib import Path
import torch

plan_path = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/"
    "results/fault_injection/int4_extended_layer_sensitivity_plan_v2.json"
)
plan = json.loads(plan_path.read_text(encoding="utf-8"))

mismatches = []

for scenario in plan["scenarios"]:
    q_clean = int(scenario["q_clean"])
    scale = float(scenario["scale"])

    baseline_value = (
        torch.tensor(q_clean, dtype=torch.float32) * scale
    ).item()

    restored_value = torch.tensor(
        float(q_clean) * scale, dtype=torch.float32
    ).item()

    if baseline_value != restored_value:
        mismatches.append({
            "scenario": scenario["scenario_id"],
            "baseline_value": baseline_value,
            "restored_value": restored_value,
            "absolute_difference": abs(baseline_value - restored_value),
        })

print("Scenarios checked:", len(plan["scenarios"]))
print("Restora

In [4]:
import hashlib
import json
from pathlib import Path

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")

paths = {
    "historical_v1": repo / (
        "results/fault_injection/int4_extended_layer_sensitivity_plan_v1.json"
    ),
    "historical_v2": repo / (
        "results/fault_injection/int4_extended_layer_sensitivity_plan_v2.json"
    ),
    "notebook05": repo / (
        "experiments/notebook05/int4_main_fault_plan_v1.json"
    ),
}

plans = {}
for name, path in paths.items():
    content = path.read_bytes()
    plans[name] = json.loads(content)
    print(f"\n--- {name} ---")
    print("SHA256:", hashlib.sha256(content).hexdigest())

expected_main_sha = (
    "906af255a7b46654f4736e4a3f77c8c778570ed3d86e5e12077bf9dbcc6f0fbb"
)
if hashlib.sha256(paths["notebook05"].read_bytes()).hexdigest() != expected_main_sha:
    raise RuntimeError("Notebook 05 fault plan changed.")

v1 = plans["historical_v1"]
v2 = plans["historical_v2"]
main = plans["notebook05"]

layers_v1 = v1["selected_layers"]
layers_v2 = v2["selected_layers"]
layers_main = [item["layer"] for item in main["layers"]]

print("\nHistorical v1 selection rule:", v1.get("selection", "NOT RECORDED"))
print("Historical v1 seed:", v1.get("seed", "NOT RECORDED"))
print("Historical v2 source:", v2.get("source_plan", "NOT RECORDED"))
print("v1 and v2 layers identical:", layers_v1 == layers_v2)
print("v2 and Notebook 05 layers identical:", layers_v2 == layers_main)

print("\nNotebook 05 layers:")
for number, layer in enumerate(layers_main, start=1):
    print(f"{number:02d}: {layer}")

print("\nStored-plan comparison only; random selection was not rerun.")
print("No inference performed. No files modified.")


--- historical_v1 ---
SHA256: 936a374a4ecfbbe4380b4b436d2461857a90748e33fa1914d1b3c7b9f3e3d51f

--- historical_v2 ---
SHA256: 308b8940a4da3a867983cd0a28eee9cf0da6b72d6771b28a2363b4205cd0e1c3

--- notebook05 ---
SHA256: 906af255a7b46654f4736e4a3f77c8c778570ed3d86e5e12077bf9dbcc6f0fbb

Historical v1 selection rule: One randomly selected Conv2d per ordered-layer stratum; eight uniformly sampled unique weight-bit pairs per layer
Historical v1 seed: 20261104
Historical v2 source: int4_extended_layer_sensitivity_plan_v1.json
v1 and v2 layers identical: True
v2 and Notebook 05 layers identical: True

Notebook 05 layers:
01: 1.conv
02: 4.cv1.conv
03: 4.m.1.cv2.conv
04: 6.m.1.cv1.conv
05: 9.cv1.conv
06: 9.cv2.conv
07: 15.cv2.conv
08: 18.m.0.cv2.conv
09: 21.cv2.conv
10: 22.cv2.2.0.conv
11: 22.cv3.0.2
12: 22.cv3.2.1.conv

Stored-plan comparison only; random selection was not rerun.
No inference performed. No files modified.


In [5]:
import json
from pathlib import Path

path = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/"
    "results/fault_injection/weight_fault_scope_v1.json"
)

if not path.is_file():
    raise FileNotFoundError("Weight-fault scope file was not found.")

scope = json.loads(path.read_text(encoding="utf-8"))
print(json.dumps(scope, indent=2, ensure_ascii=False))
print("\nRead-only inspection. No inference or files modified.")

{
  "schema_version": "1.0",
  "model": "YOLOv8s",
  "total_parameters": 11166560,
  "all_conv2d_layers": 64,
  "all_conv2d_weights": 11146096,
  "eligible_conv2d_layers": 63,
  "eligible_conv2d_weights": 11146080,
  "excluded_layer": {
    "name": "model.22.dfl.conv",
    "weight_shape": [
      1,
      16,
      1,
      1
    ],
    "weight_count": 16,
    "reason": "Fixed DFL projection coefficients excluded by the experimental weight-fault scope"
  },
  "scope_note": "Eligible weights define the population of candidate weight-fault locations, not the number of locations actually sampled in an experiment.",
  "limitations": [
    "Activation faults are outside this weight-only scope",
    "Exclusion does not imply physical immunity to faults",
    "Conv2d biases and non-Conv2d parameters are not included"
  ]
}

Read-only inspection. No inference or files modified.


In [6]:
from importlib.metadata import version, PackageNotFoundError

for package in ("torch", "ultralytics"):
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("No packages installed. No model loaded.")

torch: 2.14.1+cpu
ultralytics: NOT INSTALLED
No packages installed. No model loaded.


In [1]:
import subprocess
import sys

packages = [
    "ultralytics==8.4.189",
    "nvidia-ml-py==13.615.71",
    "ultralytics-thop==2.2.5",
    "ultralytics-platform==0.1.108",
    "jedi==0.20.1",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", *packages],
    check=True,
)

result = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True,
)

print("Dependency check exit code:", result.returncode)
print(result.stdout.strip() or result.stderr.strip() or "(no output)")
print("No model loaded. No inference performed.")

Dependency check exit code: 0
No broken requirements found.
No model loaded. No inference performed.


In [4]:
import hashlib
import json
import random
from pathlib import Path

import torch
from ultralytics import YOLO

project = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")
repo = project / "code"
checkpoint = project / "models/yolov8s.pt"

expected_checkpoint_sha = (
    "1f47a78bf100391c2a140b7ac73a1caae18c32779be7d310658112f7ac9aa78a"
)
if hashlib.sha256(checkpoint.read_bytes()).hexdigest() != expected_checkpoint_sha:
    raise RuntimeError("Checkpoint identity mismatch.")

plan_path = (
    repo / "results/fault_injection/"
    "int4_extended_layer_sensitivity_plan_v1.json"
)
plan_bytes = plan_path.read_bytes()
if hashlib.sha256(plan_bytes).hexdigest() != (
    "936a374a4ecfbbe4380b4b436d2461857a90748e33fa1914d1b3c7b9f3e3d51f"
):
    raise RuntimeError("Historical plan changed.")

historical = json.loads(plan_bytes)
audit_yolo = YOLO(str(checkpoint))
audit_model = audit_yolo.model.cpu().eval()

candidates = [
    name.removeprefix("model.")
    for name, module in audit_model.named_modules()
    if isinstance(module, torch.nn.Conv2d)
    and name != "model.22.dfl.conv"
]

if len(candidates) != 63:
    raise RuntimeError(f"Expected 63 eligible layers, found {len(candidates)}.")

rng = random.Random(20261104)
reconstructed = []

print("Stratum | Candidate count | Reconstructed selection")
for i in range(12):
    group = candidates[i * len(candidates) // 12:
                       (i + 1) * len(candidates) // 12]
    selected = rng.choice(group)
    reconstructed.append(selected)
    print(f"{i + 1:02d} | {len(group)} | {selected}")

matches = reconstructed == historical["selected_layers"]
print("\nReconstructed selection matches historical plan:", matches)

if not matches:
    raise RuntimeError("Layer selection reproduction failed.")

print("Historical layer-selection reproduction: PASS")
print("This verifies selection reproducibility, not statistical sufficiency.")
print("No inference performed. No files modified.")

Stratum | Candidate count | Reconstructed selection
01 | 5 | 1.conv
02 | 5 | 4.cv1.conv
03 | 5 | 4.m.1.cv2.conv
04 | 6 | 6.m.1.cv1.conv
05 | 5 | 9.cv1.conv
06 | 5 | 9.cv2.conv
07 | 5 | 15.cv2.conv
08 | 6 | 18.m.0.cv2.conv
09 | 5 | 21.cv2.conv
10 | 5 | 22.cv2.2.0.conv
11 | 5 | 22.cv3.0.2
12 | 6 | 22.cv3.2.1.conv

Reconstructed selection matches historical plan: True
Historical layer-selection reproduction: PASS
This verifies selection reproducibility, not statistical sufficiency.
No inference performed. No files modified.


In [3]:
from pathlib import Path

drive_root = Path("/content/drive/MyDrive")

if not drive_root.is_dir():
    from google.colab import drive
    drive.mount("/content/drive")

checkpoint = (
    drive_root / "YOLOv8_Fault_Tolerance/models/yolov8s.pt"
)

print("MyDrive accessible:", drive_root.is_dir())
print("Checkpoint exists:", checkpoint.is_file())

if checkpoint.is_file():
    print("Checkpoint size (bytes):", checkpoint.stat().st_size)
else:
    print("Stop: verify the mounted account and project path.")

Mounted at /content/drive
MyDrive accessible: True
Checkpoint exists: True
Checkpoint size (bytes): 22588772


In [5]:
import torch

selected_layers = set(reconstructed)

print("=== Top-level model blocks ===")
for index, block in enumerate(audit_model.model):
    print(
        f"Block {index:02d} | type={type(block).__name__} | "
        f"from={getattr(block, 'f', 'NOT AVAILABLE')}"
    )

print("\n=== Eligible Conv2d layers in traversal order ===")
position = 0
selected_weight_count = 0
total_weight_count = 0

for name, module in audit_model.named_modules():
    if not isinstance(module, torch.nn.Conv2d):
        continue
    if name == "model.22.dfl.conv":
        continue

    short_name = name.removeprefix("model.")
    count = module.weight.numel()
    chosen = short_name in selected_layers

    total_weight_count += count
    if chosen:
        selected_weight_count += count

    print(
        f"{position:02d} | {'SELECTED' if chosen else '--------'} | "
        f"{short_name} | shape={tuple(module.weight.shape)} | weights={count}"
    )
    position += 1

print("\nEligible layer count:", position)
print("Selected layer count:", len(selected_layers))
print("Selected layer weights:", selected_weight_count)
print("Total eligible weights:", total_weight_count)
print(
    "Weight fraction in selected layers:",
    selected_weight_count / total_weight_count,
)
print("Weight fraction is not a measure of sensitivity or statistical sufficiency.")
print("No inference performed. No files modified.")

=== Top-level model blocks ===
Block 00 | type=Conv | from=-1
Block 01 | type=Conv | from=-1
Block 02 | type=C2f | from=-1
Block 03 | type=Conv | from=-1
Block 04 | type=C2f | from=-1
Block 05 | type=Conv | from=-1
Block 06 | type=C2f | from=-1
Block 07 | type=Conv | from=-1
Block 08 | type=C2f | from=-1
Block 09 | type=SPPF | from=-1
Block 10 | type=Upsample | from=-1
Block 11 | type=Concat | from=[-1, 6]
Block 12 | type=C2f | from=-1
Block 13 | type=Upsample | from=-1
Block 14 | type=Concat | from=[-1, 4]
Block 15 | type=C2f | from=-1
Block 16 | type=Conv | from=-1
Block 17 | type=Concat | from=[-1, 12]
Block 18 | type=C2f | from=-1
Block 19 | type=Conv | from=-1
Block 20 | type=Concat | from=[-1, 9]
Block 21 | type=C2f | from=-1
Block 22 | type=Detect | from=[15, 18, 21]

=== Eligible Conv2d layers in traversal order ===
00 | -------- | 0.conv | shape=(32, 3, 3, 3) | weights=864
01 | SELECTED | 1.conv | shape=(64, 32, 3, 3) | weights=18432
02 | -------- | 2.cv1.conv | shape=(64, 64,

In [6]:
import hashlib
import json
from pathlib import Path

import torch

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
output = repo / "docs/research_reports/Notebook05_Layer_Coverage_Audit_v1.json"

if output.exists():
    raise FileExistsError("Coverage audit already exists.")

selected_set = set(reconstructed)
inventory = []

for name, module in audit_model.named_modules():
    if not isinstance(module, torch.nn.Conv2d):
        continue
    if name == "model.22.dfl.conv":
        continue

    short_name = name.removeprefix("model.")
    block = int(short_name.split(".")[0])
    region = (
        "backbone" if block <= 9
        else "detect" if block == 22
        else "neck"
    )
    inventory.append({
        "name": short_name,
        "block": block,
        "region": region,
        "weight_shape": list(module.weight.shape),
        "weight_count": module.weight.numel(),
        "selected": short_name in selected_set,
    })

if len(inventory) != 63 or sum(r["selected"] for r in inventory) != 12:
    raise RuntimeError("Unexpected layer inventory.")

coverage = {}
for region in ("backbone", "neck", "detect"):
    items = [r for r in inventory if r["region"] == region]
    coverage[region] = {
        "eligible_layers": len(items),
        "selected_layers": sum(r["selected"] for r in items),
    }

report = {
    "schema_version": "1.0",
    "checkpoint_sha256": (
        "1f47a78bf100391c2a140b7ac73a1caae18c32779be7d310658112f7ac9aa78a"
    ),
    "historical_selection_reproduced": True,
    "selection": "one_random_layer_per_ordered_list_stratum",
    "selection_seed": 20261104,
    "ordered_layer_count": 63,
    "stratum_count": 12,
    "selected_layers": reconstructed,
    "coverage": coverage,
    "inventory": inventory,
    "scope": "exploratory_study_of_selected_layers",
    "limitations": [
        "Ordered-list strata are not architecture-defined strata.",
        "Selection reproducibility does not establish statistical sufficiency.",
        "Selected layer rankings do not rank all 63 eligible layers.",
        "Target-layer-only INT4 baselines differ between layers.",
        "Twenty sampled faults per layer have no established precision guarantee.",
        "Image-sampling uncertainty is not covered by the existing fault bootstrap.",
    ],
    "review_status": {
        "completed_fault_scenarios_reported_in_session": 60,
        "further_execution": "paused_pending_design_review",
        "network_wide_screening_design": "pending",
        "fault_sample_size_justification": "pending",
    },
}

content = (json.dumps(report, indent=2, allow_nan=False) + "\n").encode()
output.parent.mkdir(parents=True, exist_ok=True)
with output.open("xb") as stream:
    stream.write(content)

print("Coverage:", coverage)
print("Saved:", output)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("No inference. Existing plans and results unchanged. No commit or push.")

Coverage: {'backbone': {'eligible_layers': 27, 'selected_layers': 6}, 'neck': {'eligible_layers': 18, 'selected_layers': 3}, 'detect': {'eligible_layers': 18, 'selected_layers': 3}}
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/docs/research_reports/Notebook05_Layer_Coverage_Audit_v1.json
SHA256: 85c49108074be1d57df4f93618632a9a80876d2abb757bd50991b888094b2456
No inference. Existing plans and results unchanged. No commit or push.


In [7]:
import hashlib
import json
from pathlib import Path
import numpy as np

folder = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/"
    "results/notebook05/main_v1"
)

expected = {
    "L01_sensitivity_summary_v1.json":
        "c851bc9eae00f7bdfa2516c2cab8e5a56c477801fdaec2c0b0e3b8fb5926239b",
    "L02_sensitivity_summary_v1.json":
        "28e81f185dbdaf53dbeb8f2355188ac0bd5a71d0c278014edefc06c28373969a",
    "L03_sensitivity_summary_v1.json":
        "cc65fdabc9a7c8c9999ee12fdad526474f778a15e26de4a346f873fd23142136",
}

print("Layer | Bit | n | Mean AP change | Sample SD | Min | Max")

for filename, expected_hash in expected.items():
    content = (folder / filename).read_bytes()
    if hashlib.sha256(content).hexdigest() != expected_hash:
        raise RuntimeError(f"Integrity mismatch: {filename}")

    summary = json.loads(content)
    for bit in range(4):
        values = np.array([
            item["AP50_95_change"]
            for item in summary["scenarios"]
            if item["bit"] == bit
        ], dtype=float)

        if len(values) != 5 or not np.isfinite(values).all():
            raise RuntimeError("Expected five finite effects per bit.")

        print(
            f"{summary['layer']} | {bit} | {len(values)} | "
            f"{values.mean():+.8f} | {values.std(ddof=1):.8f} | "
            f"{values.min():+.8f} | {values.max():+.8f}"
        )

print("\nDescriptive dispersion only; not a sample-size or power calculation.")
print("Only three previously studied layers and fixed development images.")
print("Rare damaging faults may be absent from these small samples.")
print("No inference performed. No files modified.")

Layer | Bit | n | Mean AP change | Sample SD | Min | Max
1.conv | 0 | 5 | +0.00007818 | 0.00049148 | -0.00078467 | +0.00041585
1.conv | 1 | 5 | -0.00038964 | 0.00103175 | -0.00222619 | +0.00020533
1.conv | 2 | 5 | +0.00135239 | 0.00330293 | -0.00043250 | +0.00723261
1.conv | 3 | 5 | -0.00097922 | 0.00570403 | -0.01088261 | +0.00360211
4.cv1.conv | 0 | 5 | -0.00006170 | 0.00022313 | -0.00044615 | +0.00012411
4.cv1.conv | 1 | 5 | -0.00001015 | 0.00026624 | -0.00030177 | +0.00035844
4.cv1.conv | 2 | 5 | -0.00002547 | 0.00024847 | -0.00022372 | +0.00036766
4.cv1.conv | 3 | 5 | -0.00071715 | 0.00113509 | -0.00232297 | +0.00046059
4.m.1.cv2.conv | 0 | 5 | +0.00008755 | 0.00011827 | -0.00009923 | +0.00022591
4.m.1.cv2.conv | 1 | 5 | -0.00004231 | 0.00017290 | -0.00027810 | +0.00018274
4.m.1.cv2.conv | 2 | 5 | +0.00007186 | 0.00010781 | -0.00010515 | +0.00017571
4.m.1.cv2.conv | 3 | 5 | -0.00017292 | 0.00046473 | -0.00082728 | +0.00030506

Descriptive dispersion only; not a sample-size or powe

In [8]:
import hashlib
import json
import math
from pathlib import Path

import numpy as np

folder = Path(
    "/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/"
    "results/notebook05/main_v1"
)

expected = {
    "L01_sensitivity_summary_v1.json":
        "c851bc9eae00f7bdfa2516c2cab8e5a56c477801fdaec2c0b0e3b8fb5926239b",
    "L02_sensitivity_summary_v1.json":
        "28e81f185dbdaf53dbeb8f2355188ac0bd5a71d0c278014edefc06c28373969a",
    "L03_sensitivity_summary_v1.json":
        "cc65fdabc9a7c8c9999ee12fdad526474f778a15e26de4a346f873fd23142136",
}

print("Illustrative planning for the equally weighted four-bit layer mean")
print("Layer | Target half-width | Approx. weights/bit | Total faults/layer")

for filename, expected_hash in expected.items():
    content = (folder / filename).read_bytes()
    if hashlib.sha256(content).hexdigest() != expected_hash:
        raise RuntimeError(f"Integrity mismatch: {filename}")
    summary = json.loads(content)

    variances = []
    for bit in range(4):
        values = np.array([
            item["AP50_95_change"]
            for item in summary["scenarios"]
            if item["bit"] == bit
        ], dtype=float)
        if len(values) != 5 or not np.isfinite(values).all():
            raise RuntimeError("Expected five finite effects per bit.")
        variances.append(float(values.var(ddof=1)))

    # For equal n per bit:
    # Var(layer mean) = sum(bit-specific variances) / (16 * n).
    for half_width in (0.0005, 0.001, 0.002):
        approximate_n = math.ceil(
            1.96**2 * sum(variances) / (16 * half_width**2)
        )
        planned_n = max(5, approximate_n)

        print(
            f"{summary['layer']} | {half_width:.4f} | "
            f"{planned_n} | {4 * planned_n}"
        )

print("\nPlanning approximation only; targets are not frozen.")
print("Variance estimates use only five observations per bit.")
print("No guarantee of interval coverage, rare-fault detection or stable ranking.")
print("No image-sampling uncertainty or simultaneous-comparison adjustment included.")
print("Do not extrapolate these sample sizes to unstudied layers.")
print("No inference performed. No files modified.")

Illustrative planning for the equally weighted four-bit layer mean
Layer | Target half-width | Approx. weights/bit | Total faults/layer
1.conv | 0.0005 | 43 | 172
1.conv | 0.0010 | 11 | 44
1.conv | 0.0020 | 5 | 20
4.cv1.conv | 0.0005 | 5 | 20
4.cv1.conv | 0.0010 | 5 | 20
4.cv1.conv | 0.0020 | 5 | 20
4.m.1.cv2.conv | 0.0005 | 5 | 20
4.m.1.cv2.conv | 0.0010 | 5 | 20
4.m.1.cv2.conv | 0.0020 | 5 | 20

Planning approximation only; targets are not frozen.
Variance estimates use only five observations per bit.
No guarantee of interval coverage, rare-fault detection or stable ranking.
No image-sampling uncertainty or simultaneous-comparison adjustment included.
Do not extrapolate these sample sizes to unstudied layers.
No inference performed. No files modified.


In [9]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

project = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")
repo = project / "code"
results = repo / "results/notebook05/main_v1"
reports = repo / "docs/research_reports"
output = reports / "Notebook05_Design_Review_Decision_v1.json"

if output.exists():
    raise FileExistsError("Decision record already exists. Inspect before rerunning.")

sources = {
    results / "L01_sensitivity_summary_v1.json":
        "c851bc9eae00f7bdfa2516c2cab8e5a56c477801fdaec2c0b0e3b8fb5926239b",
    results / "L02_sensitivity_summary_v1.json":
        "28e81f185dbdaf53dbeb8f2355188ac0bd5a71d0c278014edefc06c28373969a",
    results / "L03_sensitivity_summary_v1.json":
        "cc65fdabc9a7c8c9999ee12fdad526474f778a15e26de4a346f873fd23142136",
    reports / "Notebook05_Layer_Coverage_Audit_v1.json":
        "85c49108074be1d57df4f93618632a9a80876d2abb757bd50991b888094b2456",
}

evidence = {}
for path, expected_sha in sources.items():
    content = path.read_bytes()
    if hashlib.sha256(content).hexdigest() != expected_sha:
        raise RuntimeError(f"Integrity mismatch: {path.name}")
    evidence[path.name] = {
        "sha256": expected_sha,
        "data": json.loads(content),
    }

completed = sum(
    evidence[name]["data"]["scenario_count"]
    for name in (
        "L01_sensitivity_summary_v1.json",
        "L02_sensitivity_summary_v1.json",
        "L03_sensitivity_summary_v1.json",
    )
)
if completed != 60:
    raise RuntimeError("Unexpected summarized scenario count.")

decision = {
    "schema_version": "1.0",
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "INITIAL_CAMPAIGN_PAUSED_PENDING_REDESIGN",
    "initial_plan": {
        "selected_layers": 12,
        "planned_fault_scenarios": 240,
        "completed_summarized_scenarios": completed,
        "remaining_planned_scenarios": 180,
        "development_images_per_scenario": 1000,
    },
    "observation_disclosure": (
        "Decision made after observing the pilot and the first 60 "
        "scenarios from the first three layers in the frozen execution order."
    ),
    "reasons": [
        "Selected layers do not cover all eligible architectural locations.",
        "Five sampled weights per bit have no validated precision guarantee.",
        "Mean signed AP change can conceal damaging individual scenarios.",
        "Target-layer-only INT4 models differ across layer comparisons.",
        "Final protection selection requires a justified broader design.",
    ],
    "interpretation": [
        "Preserve all observed positive, negative and zero effects.",
        "Treat existing results as exploratory evidence for the studied layers.",
        "Do not present the truncated campaign as the completed 240-scenario study.",
        "Do not claim network-wide layer rankings or hardware reliability.",
        "No guarantee of statistical precision follows from the planning calculation.",
    ],
    "next_design_tasks": [
        "Define the intended deployment representation and clean baseline.",
        "Define screening coverage across 63 eligible Conv2d layers.",
        "Specify primary estimands, sampling budget and precision criteria.",
        "Define independent follow-up fault samples and ranking-stability checks.",
        "Address image-sampling uncertainty and practical degradation thresholds.",
        "Record a versioned protocol before new screening inference.",
    ],
    "preservation": {
        "overwrite_existing_plans_or_results": False,
        "held_out_images_used_in_reported_runs": False,
        "new_screening_protocol_finalized": False,
    },
    "source_sha256": {
        name: item["sha256"] for name, item in evidence.items()
    },
}

content = (
    json.dumps(decision, indent=2, allow_nan=False) + "\n"
).encode("utf-8")

reports.mkdir(parents=True, exist_ok=True)
with output.open("xb") as stream:
    stream.write(content)

print("Saved:", output)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("Completed summarized scenarios:", completed)
print("Initial campaign paused; new screening design is not yet finalized.")
print("No inference. Existing files unchanged. No commit or push.")

Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/docs/research_reports/Notebook05_Design_Review_Decision_v1.json
SHA256: 2a1cf33f85bbbddb87a7ac7910488ec280b66d02dbdaafee13b8f7b5908255d4
Completed summarized scenarios: 60
Initial campaign paused; new screening design is not yet finalized.
No inference. Existing files unchanged. No commit or push.


In [10]:
import hashlib
from pathlib import Path

import torch
from ultralytics import YOLO

project = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")
checkpoint = project / "models/yolov8s.pt"

expected_sha = (
    "1f47a78bf100391c2a140b7ac73a1caae18c32779be7d310658112f7ac9aa78a"
)
if hashlib.sha256(checkpoint.read_bytes()).hexdigest() != expected_sha:
    raise RuntimeError("Checkpoint identity mismatch.")

common_int4_yolo = YOLO(str(checkpoint))
common_int4_model = common_int4_yolo.model.cpu().float().eval()

eligible = [
    (name, module)
    for name, module in common_int4_model.named_modules()
    if isinstance(module, torch.nn.Conv2d)
    and name != "model.22.dfl.conv"
]
if len(eligible) != 63:
    raise RuntimeError("Expected 63 eligible Conv2d layers.")

target_keys = {name + ".weight" for name, _ in eligible}
preserved_state = {
    name: tensor.detach().clone()
    for name, tensor in common_int4_model.state_dict().items()
    if name not in target_keys
}

common_int4_inventory = []
with torch.no_grad():
    for name, module in eligible:
        weight = module.weight.detach()
        scale = weight.abs().max() / 7

        if not torch.isfinite(scale).item() or scale.item() <= 0:
            raise RuntimeError(f"Invalid scale: {name}")

        q = torch.round(weight / scale).clamp(-7, 7).to(torch.int8)
        dequantized = q.float() * scale
        module.weight.copy_(dequantized)

        if not torch.equal(module.weight, dequantized):
            raise RuntimeError(f"Quantization verification failed: {name}")

        common_int4_inventory.append({
            "layer": name.removeprefix("model."),
            "shape": list(q.shape),
            "scale": float(scale.item()),
            "weight_count": q.numel(),
            "zero_count": int((q == 0).sum().item()),
        })

for name, original in preserved_state.items():
    if not torch.equal(common_int4_model.state_dict()[name], original):
        raise RuntimeError(f"Unexpected non-target state change: {name}")

total = sum(item["weight_count"] for item in common_int4_inventory)
zeros = sum(item["zero_count"] for item in common_int4_inventory)

print("Candidate: shared clean logical INT4 weight model")
print("Quantized Conv2d layers:", len(common_int4_inventory))
print("Quantized weights:", total)
print("Zero-valued quantized weight fraction:", zeros / total)
print("Biases, non-target parameters and buffers preserved: PASS")
print("Quantization: symmetric per-tensor, CPU FP32 scale, range [-7, 7]")
print("Inference representation: dequantized FP32 weights")
print("Candidate accuracy has not been evaluated.")
print("No inference performed. No checkpoint or result files modified.")

Candidate: shared clean logical INT4 weight model
Quantized Conv2d layers: 63
Quantized weights: 11146080
Zero-valued quantized weight fraction: 0.8232428800080387
Biases, non-target parameters and buffers preserved: PASS
Quantization: symmetric per-tensor, CPU FP32 scale, range [-7, 7]
Inference representation: dequantized FP32 weights
Candidate accuracy has not been evaluated.
No inference performed. No checkpoint or result files modified.


In [11]:
import copy
import gzip
import hashlib
import json
import platform
import time
from importlib.metadata import version

import cv2
import numpy as np
import torch
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import YOLO
from ultralytics.data.augment import LetterBox
from ultralytics.utils.nms import non_max_suppression
from ultralytics.utils.ops import scale_boxes

repo = project / "code"
output = repo / "results/notebook05/shared_INT4_candidate_pilot50_cpu_v1.json.gz"
partial = output.with_name(output.name + ".part")
if output.exists() or partial.exists():
    raise FileExistsError("Candidate report or partial file already exists.")

split_path = repo / "experiments/notebook05/coco2017_split_manifest_v1.json"
split_bytes = split_path.read_bytes()
if hashlib.sha256(split_bytes).hexdigest() != (
    "143c3273b2bf0cd6393cfe148a74e48fbafb74e770bd3fbdf3f14222355978ee"
):
    raise RuntimeError("Split manifest changed.")
split = json.loads(split_bytes)
ids = split["image_ids"]["pilot"]
assert len(ids) == len(set(ids)) == 50
assert set(ids) <= set(split["image_ids"]["development"])
assert not set(ids) & set(split["image_ids"]["held_out"])

root = project / "datasets/coco2017"
annotation_path = root / "annotations/instances_val2017.json"
annotation_bytes = annotation_path.read_bytes()
if hashlib.sha256(annotation_bytes).hexdigest() != split["annotation_sha256"]:
    raise RuntimeError("Annotation file changed.")

annotations = json.loads(annotation_bytes)
images = {item["id"]: item for item in annotations["images"]}
categories = sorted(annotations["categories"], key=lambda item: item["id"])
category_ids = [item["id"] for item in categories]

if hashlib.sha256(checkpoint.read_bytes()).hexdigest() != expected_sha:
    raise RuntimeError("Checkpoint identity mismatch.")

reference_yolo = YOLO(str(checkpoint))
reference_model = reference_yolo.model.cpu().float().eval()
if not all(
    reference_yolo.names[i] == item["name"]
    and common_int4_yolo.names[i] == item["name"]
    for i, item in enumerate(categories)
):
    raise RuntimeError("Class mapping mismatch.")

torch.manual_seed(20261111)
torch.use_deterministic_algorithms(True)
transform = LetterBox(
    new_shape=(640, 640), auto=False,
    scale_fill=False, scaleup=True, stride=32,
)
candidates = {
    "fp32_clean": reference_model,
    "shared_int4_clean": common_int4_model.cpu().float().eval(),
}
predictions = {name: [] for name in candidates}
started = time.perf_counter()

with torch.inference_mode():
    for number, image_id in enumerate(ids, start=1):
        record = images[image_id]
        image = cv2.imread(str(root / "images/val2017" / record["file_name"]))
        if image is None or image.shape[:2] != (
            record["height"], record["width"]
        ):
            raise RuntimeError(f"Image validation failed: {image_id}")

        padded = transform(image=image)
        array = np.ascontiguousarray(padded[:, :, ::-1].transpose(2, 0, 1))
        tensor = torch.from_numpy(array).unsqueeze(0).float() / 255.0

        for name, candidate in candidates.items():
            result = candidate(tensor)
            raw = result[0] if isinstance(result, (tuple, list)) else result
            if tuple(raw.shape) != (1, 84, 8400) or not torch.isfinite(raw).all():
                raise RuntimeError(f"Invalid predictions: {name}, {image_id}")

            detections = non_max_suppression(
                raw.clone(), conf_thres=0.001, iou_thres=0.7,
                nc=80, max_det=300,
            )[0].clone()
            detections[:, :4] = scale_boxes(
                tensor.shape[2:], detections[:, :4], image.shape[:2]
            )
            if not torch.isfinite(detections).all():
                raise RuntimeError("Non-finite detections.")

            for row in detections:
                x1, y1, x2, y2, score, cls = row.tolist()
                if x2 <= x1 or y2 <= y1:
                    continue
                if not (cls == int(cls) and 0 <= cls < 80 and 0 <= score <= 1):
                    raise RuntimeError("Invalid class or confidence.")
                predictions[name].append({
                    "image_id": int(image_id),
                    "category_id": int(category_ids[int(cls)]),
                    "bbox": [x1, y1, x2 - x1, y2 - y1],
                    "score": score,
                })

        if number % 10 == 0:
            print(f"Paired candidate check: {number}/50", flush=True)

wall_seconds = time.perf_counter() - started
gt = COCO(str(annotation_path))
metrics = {}

for name, records in predictions.items():
    if records:
        dt = gt.loadRes(copy.deepcopy(records))
    else:
        dt = COCO()
        dt.dataset = {
            "images": copy.deepcopy(gt.dataset["images"]),
            "categories": copy.deepcopy(gt.dataset["categories"]),
            "annotations": [],
        }
        dt.createIndex()

    evaluator = COCOeval(gt, dt, iouType="bbox")
    evaluator.params.imgIds = list(ids)
    evaluator.params.catIds = category_ids
    evaluator.params.maxDets = [1, 10, 100]
    print(f"\n=== {name} ===")
    evaluator.evaluate()
    evaluator.accumulate()
    evaluator.summarize()
    metrics[name] = {
        key: float(evaluator.stats[index])
        for key, index in {"AP50_95": 0, "AP50": 1, "AP75": 2, "AR100": 8}.items()
    }

report = {
    "scope": "clean_candidate_quality_check_on_development_pilot",
    "image_ids": ids,
    "checkpoint_sha256": expected_sha,
    "split_manifest_sha256": hashlib.sha256(split_bytes).hexdigest(),
    "annotation_sha256": split["annotation_sha256"],
    "environment": {
        "python": platform.python_version(),
        **{p: version(p) for p in (
            "torch", "torchvision", "ultralytics", "numpy",
            "pycocotools", "opencv-python", "Pillow"
        )},
        "device": "cpu",
    },
    "settings": {
        "imgsz": 640, "batch": 1, "conf": 0.001,
        "nms_iou": 0.7, "max_det": 300,
        "cocoeval_max_dets": [1, 10, 100],
        "letterbox_auto": False, "scale_fill": False,
        "scaleup": True, "stride": 32,
        "dtype": "float32", "color": "RGB", "normalization": "divide_by_255",
    },
    "quantization_inventory": common_int4_inventory,
    "metrics": metrics,
    "predictions": predictions,
    "paired_inference_pipeline_seconds": wall_seconds,
    "limitations": [
        "Development pilot only; not final evaluation.",
        "No candidate acceptance threshold was predefined.",
        "Logical INT4 weights with FP32 computation; no hardware performance claim.",
    ],
}

payload = json.dumps(report, allow_nan=False, separators=(",", ":")).encode()
output.parent.mkdir(parents=True, exist_ok=True)
with partial.open("xb") as stream:
    with gzip.GzipFile(fileobj=stream, mode="wb", mtime=0) as compressed:
        compressed.write(payload)
with gzip.open(partial, "rb") as stream:
    if stream.read() != payload:
        raise RuntimeError("Compressed report verification failed.")
if output.exists():
    raise FileExistsError("Destination appeared during execution.")
partial.rename(output)

print("\nMetrics:", metrics)
print(
    "INT4 minus FP32 AP50:95:",
    metrics["shared_int4_clean"]["AP50_95"] - metrics["fp32_clean"]["AP50_95"],
)
print("Saved:", output)
print("SHA256:", hashlib.sha256(output.read_bytes()).hexdigest())
print("No fault injected. No held-out images evaluated. No commit or push.")

Paired candidate check: 10/50
Paired candidate check: 20/50
Paired candidate check: 30/50
Paired candidate check: 40/50
Paired candidate check: 50/50
loading annotations into memory...
Done (t=0.90s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!

=== fp32_clean ===
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.63s).
Accumulating evaluation results...
DONE (t=0.49s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.607
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.751
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.665
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.372
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.575
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.780
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all

In [12]:
import hashlib
import torch
from ultralytics import YOLO

checkpoint = project / "models/yolov8s.pt"
if hashlib.sha256(checkpoint.read_bytes()).hexdigest() != (
    "1f47a78bf100391c2a140b7ac73a1caae18c32779be7d310658112f7ac9aa78a"
):
    raise RuntimeError("Checkpoint identity mismatch.")

channel_int4_yolo = YOLO(str(checkpoint))
channel_int4_model = channel_int4_yolo.model.cpu().float().eval()

eligible = [
    (name, module)
    for name, module in channel_int4_model.named_modules()
    if isinstance(module, torch.nn.Conv2d)
    and name != "model.22.dfl.conv"
]
if len(eligible) != 63:
    raise RuntimeError("Expected 63 eligible Conv2d layers.")

target_keys = {name + ".weight" for name, _ in eligible}
preserved = {
    name: tensor.detach().clone()
    for name, tensor in channel_int4_model.state_dict().items()
    if name not in target_keys
}

channel_int4_inventory = []

with torch.no_grad():
    for name, module in eligible:
        weight = module.weight.detach()
        channel_max = weight.abs().amax(dim=(1, 2, 3), keepdim=True)

        # Zero-only channels use scale 1 and remain exactly zero.
        scale = torch.where(
            channel_max > 0,
            channel_max / 7,
            torch.ones_like(channel_max),
        )
        if not torch.isfinite(scale).all() or not (scale > 0).all():
            raise RuntimeError(f"Invalid channel scales: {name}")

        q = torch.round(weight / scale).clamp(-7, 7).to(torch.int8)
        reconstructed_weight = q.float() * scale
        module.weight.copy_(reconstructed_weight)

        if not torch.equal(module.weight, reconstructed_weight):
            raise RuntimeError(f"Quantization verification failed: {name}")

        channel_int4_inventory.append({
            "layer": name.removeprefix("model."),
            "shape": list(weight.shape),
            "scale_axis": 0,
            "scales": scale.flatten().tolist(),
            "zero_only_channels": int((channel_max == 0).sum().item()),
            "weight_count": q.numel(),
            "zero_count": int((q == 0).sum().item()),
        })

state = channel_int4_model.state_dict()
for name, original in preserved.items():
    if not torch.equal(state[name], original):
        raise RuntimeError(f"Unexpected non-target state change: {name}")

total = sum(item["weight_count"] for item in channel_int4_inventory)
zeros = sum(item["zero_count"] for item in channel_int4_inventory)

print("Candidate: logical INT4 with per-output-channel scales")
print("Quantized layers:", len(channel_int4_inventory))
print("Quantized weights:", total)
print("Zero-valued quantized weight fraction:", zeros / total)
print("Non-target parameters and buffers preserved: PASS")
print("Accuracy has not been evaluated.")
print("No inference performed. Existing candidates and files unchanged.")

Candidate: logical INT4 with per-output-channel scales
Quantized layers: 63
Quantized weights: 11146080
Zero-valued quantized weight fraction: 0.4293725686519386
Non-target parameters and buffers preserved: PASS
Accuracy has not been evaluated.
No inference performed. Existing candidates and files unchanged.


In [13]:
import copy
import gzip
import hashlib
import json
import platform
import time
from importlib.metadata import version

import cv2
import numpy as np
import torch
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics.data.augment import LetterBox
from ultralytics.utils.nms import non_max_suppression
from ultralytics.utils.ops import scale_boxes

folder = project / "code/results/notebook05"
reference_path = folder / "shared_INT4_candidate_pilot50_cpu_v1.json.gz"
output = folder / "shared_INT4_perchannel_pilot50_cpu_v1.json.gz"
partial = output.with_name(output.name + ".part")

if output.exists() or partial.exists():
    raise FileExistsError("Output or partial file already exists.")

reference_bytes = reference_path.read_bytes()
reference_sha = hashlib.sha256(reference_bytes).hexdigest()
if reference_sha != (
    "d435a9b2989b5f9d13a7d0fa1dc1c875541bc3dc069d1e276015e286f4c84b19"
):
    raise RuntimeError("Reference report changed.")

reference = json.loads(gzip.decompress(reference_bytes))
environment = reference["environment"]
if platform.python_version() != environment["python"]:
    raise RuntimeError("Python version differs from the reference run.")
for name in (
    "torch", "torchvision", "ultralytics", "numpy",
    "pycocotools", "opencv-python", "Pillow"
):
    if version(name) != environment[name]:
        raise RuntimeError(f"Package version differs: {name}")

split_path = (
    project / "code/experiments/notebook05/coco2017_split_manifest_v1.json"
)
split_bytes = split_path.read_bytes()
if hashlib.sha256(split_bytes).hexdigest() != reference["split_manifest_sha256"]:
    raise RuntimeError("Split manifest mismatch.")
split = json.loads(split_bytes)
ids = split["image_ids"]["pilot"]
assert ids == reference["image_ids"]
assert set(ids) <= set(split["image_ids"]["development"])
assert not set(ids) & set(split["image_ids"]["held_out"])

root = project / "datasets/coco2017"
annotation_path = root / "annotations/instances_val2017.json"
annotation_bytes = annotation_path.read_bytes()
if hashlib.sha256(annotation_bytes).hexdigest() != reference["annotation_sha256"]:
    raise RuntimeError("Annotation file changed.")

coco_data = json.loads(annotation_bytes)
images = {item["id"]: item for item in coco_data["images"]}
categories = sorted(coco_data["categories"], key=lambda item: item["id"])
category_ids = [item["id"] for item in categories]
assert all(
    channel_int4_yolo.names[i] == item["name"]
    for i, item in enumerate(categories)
)

torch.manual_seed(20261111)
torch.use_deterministic_algorithms(True)
channel_int4_model.cpu().float().eval()
transform = LetterBox(
    new_shape=(640, 640), auto=False,
    scale_fill=False, scaleup=True, stride=32,
)
predictions = []
started = time.perf_counter()

with torch.inference_mode():
    for number, image_id in enumerate(ids, start=1):
        record = images[image_id]
        image = cv2.imread(str(root / "images/val2017" / record["file_name"]))
        if image is None or image.shape[:2] != (
            record["height"], record["width"]
        ):
            raise RuntimeError(f"Image validation failed: {image_id}")

        padded = transform(image=image)
        array = np.ascontiguousarray(padded[:, :, ::-1].transpose(2, 0, 1))
        tensor = torch.from_numpy(array).unsqueeze(0).float() / 255.0

        result = channel_int4_model(tensor)
        raw = result[0] if isinstance(result, (tuple, list)) else result
        if tuple(raw.shape) != (1, 84, 8400) or not torch.isfinite(raw).all():
            raise RuntimeError(f"Invalid raw output: {image_id}")

        detections = non_max_suppression(
            raw.clone(), conf_thres=0.001, iou_thres=0.7,
            nc=80, max_det=300,
        )[0].clone()
        detections[:, :4] = scale_boxes(
            tensor.shape[2:], detections[:, :4], image.shape[:2]
        )
        if not torch.isfinite(detections).all():
            raise RuntimeError("Non-finite detections.")

        for row in detections:
            x1, y1, x2, y2, score, cls = row.tolist()
            if x2 <= x1 or y2 <= y1:
                continue
            if not (cls == int(cls) and 0 <= cls < 80 and 0 <= score <= 1):
                raise RuntimeError("Invalid class or confidence.")
            predictions.append({
                "image_id": int(image_id),
                "category_id": int(category_ids[int(cls)]),
                "bbox": [x1, y1, x2 - x1, y2 - y1],
                "score": score,
            })

        if number % 10 == 0:
            print(f"Per-channel INT4: {number}/50", flush=True)

elapsed = time.perf_counter() - started
gt = COCO(str(annotation_path))
if predictions:
    dt = gt.loadRes(copy.deepcopy(predictions))
else:
    dt = COCO()
    dt.dataset = {
        "images": copy.deepcopy(gt.dataset["images"]),
        "categories": copy.deepcopy(gt.dataset["categories"]),
        "annotations": [],
    }
    dt.createIndex()

evaluator = COCOeval(gt, dt, iouType="bbox")
evaluator.params.imgIds = list(ids)
evaluator.params.catIds = category_ids
evaluator.params.maxDets = [1, 10, 100]
evaluator.evaluate()
evaluator.accumulate()
evaluator.summarize()

metrics = {
    key: float(evaluator.stats[index])
    for key, index in {"AP50_95": 0, "AP50": 1, "AP75": 2, "AR100": 8}.items()
}
if not all(np.isfinite(value) for value in metrics.values()):
    raise RuntimeError("Non-finite metrics.")

report = {
    "scope": "development_pilot_clean_candidate_quality_check",
    "candidate": "shared_logical_INT4_per_output_channel",
    "reference_report_sha256": reference_sha,
    "checkpoint_sha256": reference["checkpoint_sha256"],
    "split_manifest_sha256": reference["split_manifest_sha256"],
    "annotation_sha256": reference["annotation_sha256"],
    "environment": environment,
    "settings": reference["settings"],
    "image_ids": ids,
    "quantization_inventory": channel_int4_inventory,
    "metrics": metrics,
    "fp32_reference_metrics": reference["metrics"]["fp32_clean"],
    "predictions": predictions,
    "inference_pipeline_seconds": elapsed,
    "limitations": [
        "Only 50 development images; no final-test claim.",
        "FP32 baseline reused from the verified same-environment report.",
        "No candidate acceptance threshold was predefined.",
        "Candidate selection occurs after observing earlier candidate results.",
        "Logical quantization with FP32 computation; no hardware INT4 claim.",
    ],
}

payload = json.dumps(report, allow_nan=False, separators=(",", ":")).encode()
with partial.open("xb") as stream:
    with gzip.GzipFile(fileobj=stream, mode="wb", mtime=0) as compressed:
        compressed.write(payload)
with gzip.open(partial, "rb") as stream:
    if stream.read() != payload:
        raise RuntimeError("Compressed report verification failed.")
if output.exists():
    raise FileExistsError("Destination appeared during execution.")
partial.rename(output)

print("\nPer-channel INT4:", metrics)
print("FP32 reference:", report["fp32_reference_metrics"])
print(
    "AP50:95 change:",
    metrics["AP50_95"] - report["fp32_reference_metrics"]["AP50_95"],
)
print("Saved:", output)
print("SHA256:", hashlib.sha256(output.read_bytes()).hexdigest())
print("No fault injected. No held-out evaluation. No commit or push.")

Per-channel INT4: 10/50
Per-channel INT4: 20/50
Per-channel INT4: 30/50
Per-channel INT4: 40/50
Per-channel INT4: 50/50
loading annotations into memory...
Done (t=0.50s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.27s).
Accumulating evaluation results...
DONE (t=0.22s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.245
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.381
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.275
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.140
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.248
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.306
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.232
 Average Recall     (AR) 

In [14]:
import hashlib
import torch
from ultralytics import YOLO

checkpoint = project / "models/yolov8s.pt"
expected_checkpoint_sha = (
    "1f47a78bf100391c2a140b7ac73a1caae18c32779be7d310658112f7ac9aa78a"
)
if hashlib.sha256(checkpoint.read_bytes()).hexdigest() != expected_checkpoint_sha:
    raise RuntimeError("Checkpoint identity mismatch.")

channel_int8_yolo = YOLO(str(checkpoint))
channel_int8_model = channel_int8_yolo.model.cpu().float().eval()

eligible = [
    (name, module)
    for name, module in channel_int8_model.named_modules()
    if isinstance(module, torch.nn.Conv2d)
    and name != "model.22.dfl.conv"
]
if len(eligible) != 63:
    raise RuntimeError("Expected 63 eligible layers.")

target_keys = {name + ".weight" for name, _ in eligible}
preserved = {
    name: tensor.detach().clone()
    for name, tensor in channel_int8_model.state_dict().items()
    if name not in target_keys
}

channel_int8_inventory = []
with torch.no_grad():
    for name, module in eligible:
        weight = module.weight.detach()
        channel_max = weight.abs().amax(dim=(1, 2, 3), keepdim=True)
        scale = torch.where(
            channel_max > 0,
            channel_max / 127,
            torch.ones_like(channel_max),
        )
        if not torch.isfinite(scale).all() or not (scale > 0).all():
            raise RuntimeError(f"Invalid scale: {name}")

        q = torch.round(weight / scale).clamp(-127, 127).to(torch.int8)
        dequantized = q.float() * scale
        module.weight.copy_(dequantized)
        if not torch.equal(module.weight, dequantized):
            raise RuntimeError(f"Quantization verification failed: {name}")

        channel_int8_inventory.append({
            "layer": name.removeprefix("model."),
            "shape": list(weight.shape),
            "scale_axis": 0,
            "scales": scale.flatten().tolist(),
            "zero_only_channels": int((channel_max == 0).sum().item()),
            "weight_count": q.numel(),
            "zero_count": int((q == 0).sum().item()),
        })

state = channel_int8_model.state_dict()
for name, original in preserved.items():
    if not torch.equal(state[name], original):
        raise RuntimeError(f"Unexpected non-target state change: {name}")

total = sum(item["weight_count"] for item in channel_int8_inventory)
zeros = sum(item["zero_count"] for item in channel_int8_inventory)

print("Candidate: logical INT8 with per-output-channel scales")
print("Quantized layers:", len(channel_int8_inventory))
print("Quantized weights:", total)
print("Zero-valued quantized weight fraction:", zeros / total)
print("Non-target parameters and buffers preserved: PASS")
print("Clean integer range: [-127, 127]")
print("Inference representation: dequantized FP32 weights")
print("Accuracy has not been evaluated.")
print("No inference. Existing candidates and files unchanged.")

Candidate: logical INT8 with per-output-channel scales
Quantized layers: 63
Quantized weights: 11146080
Zero-valued quantized weight fraction: 0.030653108536812943
Non-target parameters and buffers preserved: PASS
Clean integer range: [-127, 127]
Inference representation: dequantized FP32 weights
Accuracy has not been evaluated.
No inference. Existing candidates and files unchanged.


In [15]:
import copy
import gzip
import hashlib
import json
import platform
import time
from importlib.metadata import version

import cv2
import numpy as np
import torch
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics.data.augment import LetterBox
from ultralytics.utils.nms import non_max_suppression
from ultralytics.utils.ops import scale_boxes

folder = project / "code/results/notebook05"
reference_path = folder / "shared_INT4_candidate_pilot50_cpu_v1.json.gz"
output = folder / "shared_INT8_perchannel_pilot50_cpu_v1.json.gz"
partial = output.with_name(output.name + ".part")

if output.exists() or partial.exists():
    raise FileExistsError("Output or partial file already exists.")

reference_bytes = reference_path.read_bytes()
reference_sha = hashlib.sha256(reference_bytes).hexdigest()
if reference_sha != (
    "d435a9b2989b5f9d13a7d0fa1dc1c875541bc3dc069d1e276015e286f4c84b19"
):
    raise RuntimeError("Reference report changed.")

reference = json.loads(gzip.decompress(reference_bytes))
environment = reference["environment"]
if platform.python_version() != environment["python"]:
    raise RuntimeError("Python version differs from the reference run.")
for name in (
    "torch", "torchvision", "ultralytics", "numpy",
    "pycocotools", "opencv-python", "Pillow"
):
    if version(name) != environment[name]:
        raise RuntimeError(f"Package version differs: {name}")

checkpoint_sha = hashlib.sha256(checkpoint.read_bytes()).hexdigest()
if checkpoint_sha != reference["checkpoint_sha256"]:
    raise RuntimeError("Checkpoint identity mismatch.")

split_path = (
    project / "code/experiments/notebook05/coco2017_split_manifest_v1.json"
)
split_bytes = split_path.read_bytes()
if hashlib.sha256(split_bytes).hexdigest() != reference["split_manifest_sha256"]:
    raise RuntimeError("Split manifest mismatch.")
split = json.loads(split_bytes)
ids = split["image_ids"]["pilot"]
assert len(ids) == len(set(ids)) == 50
assert ids == reference["image_ids"]
assert set(ids) <= set(split["image_ids"]["development"])
assert not set(ids) & set(split["image_ids"]["held_out"])

root = project / "datasets/coco2017"
annotation_path = root / "annotations/instances_val2017.json"
annotation_bytes = annotation_path.read_bytes()
if hashlib.sha256(annotation_bytes).hexdigest() != reference["annotation_sha256"]:
    raise RuntimeError("Annotation file changed.")

coco_data = json.loads(annotation_bytes)
images = {item["id"]: item for item in coco_data["images"]}
categories = sorted(coco_data["categories"], key=lambda item: item["id"])
category_ids = [item["id"] for item in categories]
assert all(
    channel_int8_yolo.names[i] == item["name"]
    for i, item in enumerate(categories)
)

torch.manual_seed(20261111)
torch.use_deterministic_algorithms(True)
channel_int8_model.cpu().float().eval()
transform = LetterBox(
    new_shape=(640, 640), auto=False,
    scale_fill=False, scaleup=True, stride=32,
)
predictions = []
started = time.perf_counter()

with torch.inference_mode():
    for number, image_id in enumerate(ids, start=1):
        record = images[image_id]
        image = cv2.imread(str(root / "images/val2017" / record["file_name"]))
        if image is None or image.shape[:2] != (
            record["height"], record["width"]
        ):
            raise RuntimeError(f"Image validation failed: {image_id}")

        padded = transform(image=image)
        array = np.ascontiguousarray(padded[:, :, ::-1].transpose(2, 0, 1))
        tensor = torch.from_numpy(array).unsqueeze(0).float() / 255.0

        result = channel_int8_model(tensor)
        raw = result[0] if isinstance(result, (tuple, list)) else result
        if tuple(raw.shape) != (1, 84, 8400) or not torch.isfinite(raw).all():
            raise RuntimeError(f"Invalid raw output: {image_id}")

        detections = non_max_suppression(
            raw.clone(), conf_thres=0.001, iou_thres=0.7,
            nc=80, max_det=300,
        )[0].clone()
        detections[:, :4] = scale_boxes(
            tensor.shape[2:], detections[:, :4], image.shape[:2]
        )
        if not torch.isfinite(detections).all():
            raise RuntimeError("Non-finite detections.")

        for row in detections:
            x1, y1, x2, y2, score, cls = row.tolist()
            if x2 <= x1 or y2 <= y1:
                continue
            if not (cls == int(cls) and 0 <= cls < 80 and 0 <= score <= 1):
                raise RuntimeError("Invalid class or confidence.")
            predictions.append({
                "image_id": int(image_id),
                "category_id": int(category_ids[int(cls)]),
                "bbox": [x1, y1, x2 - x1, y2 - y1],
                "score": score,
            })

        if number % 10 == 0:
            print(f"Per-channel INT8: {number}/50", flush=True)

elapsed = time.perf_counter() - started
gt = COCO(str(annotation_path))

if predictions:
    dt = gt.loadRes(copy.deepcopy(predictions))
else:
    dt = COCO()
    dt.dataset = {
        "images": copy.deepcopy(gt.dataset["images"]),
        "categories": copy.deepcopy(gt.dataset["categories"]),
        "annotations": [],
    }
    dt.createIndex()

evaluator = COCOeval(gt, dt, iouType="bbox")
evaluator.params.imgIds = list(ids)
evaluator.params.catIds = category_ids
evaluator.params.maxDets = [1, 10, 100]
evaluator.evaluate()
evaluator.accumulate()
evaluator.summarize()

metrics = {
    key: float(evaluator.stats[index])
    for key, index in {"AP50_95": 0, "AP50": 1, "AP75": 2, "AR100": 8}.items()
}
if not all(np.isfinite(value) for value in metrics.values()):
    raise RuntimeError("Non-finite metrics.")

report = {
    "scope": "development_pilot_clean_candidate_quality_check",
    "candidate": "shared_logical_INT8_per_output_channel",
    "reference_report_sha256": reference_sha,
    "checkpoint_sha256": checkpoint_sha,
    "split_manifest_sha256": reference["split_manifest_sha256"],
    "annotation_sha256": reference["annotation_sha256"],
    "environment": environment,
    "settings": reference["settings"],
    "image_ids": ids,
    "quantization": {
        "clean_integer_range": [-127, 127],
        "scale": "CPU_FP32_per_output_channel_abs_max_divided_by_127",
        "scope": "63_eligible_Conv2d_weights",
        "inference": "dequantized_FP32",
    },
    "quantization_inventory": channel_int8_inventory,
    "metrics": metrics,
    "fp32_reference_metrics": reference["metrics"]["fp32_clean"],
    "predictions": predictions,
    "inference_pipeline_seconds": elapsed,
    "limitations": [
        "Only 50 development images; no final-test claim.",
        "FP32 baseline reused from the verified same-environment report.",
        "No candidate acceptance threshold was predefined.",
        "Candidate selection follows observation of earlier INT4 candidates.",
        "No native INT8 hardware performance claim.",
    ],
}

payload = json.dumps(report, allow_nan=False, separators=(",", ":")).encode()
with partial.open("xb") as stream:
    with gzip.GzipFile(fileobj=stream, mode="wb", mtime=0) as compressed:
        compressed.write(payload)
with gzip.open(partial, "rb") as stream:
    if stream.read() != payload:
        raise RuntimeError("Compressed report verification failed.")
if output.exists():
    raise FileExistsError("Destination appeared during execution.")
partial.rename(output)

print("\nPer-channel INT8:", metrics)
print("FP32 reference:", report["fp32_reference_metrics"])
print(
    "AP50:95 change:",
    metrics["AP50_95"] - report["fp32_reference_metrics"]["AP50_95"],
)
print("Saved:", output)
print("SHA256:", hashlib.sha256(output.read_bytes()).hexdigest())
print("No fault injected. No held-out evaluation. No commit or push.")

Per-channel INT8: 10/50
Per-channel INT8: 20/50
Per-channel INT8: 30/50
Per-channel INT8: 40/50
Per-channel INT8: 50/50
loading annotations into memory...
Done (t=0.56s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.32s).
Accumulating evaluation results...
DONE (t=0.28s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.609
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.754
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.668
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.375
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.573
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.778
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.493
 Average Recall     (AR) 

In [16]:
import gzip
import hashlib
import json
from pathlib import Path

folder = project / "code/results/notebook05"
output = (
    project / "code/docs/research_reports/"
    "Notebook05_Shared_Baseline_Candidate_Review_v1.json"
)
if output.exists():
    raise FileExistsError("Candidate review already exists.")

sources = {
    "int4_per_tensor": (
        "shared_INT4_candidate_pilot50_cpu_v1.json.gz",
        "d435a9b2989b5f9d13a7d0fa1dc1c875541bc3dc069d1e276015e286f4c84b19",
    ),
    "int4_per_channel": (
        "shared_INT4_perchannel_pilot50_cpu_v1.json.gz",
        "a684a61b34b17f1c3cbc556e41f34cf86ff6b2e0ca3d57c7dd4473c8daf44235",
    ),
    "int8_per_channel": (
        "shared_INT8_perchannel_pilot50_cpu_v1.json.gz",
        "f61a979c0daa11c8d07ff113f1b2242aa4d44e1354644aeebf3776c65246b07a",
    ),
}

reports = {}
for name, (filename, expected_sha) in sources.items():
    content = (folder / filename).read_bytes()
    if hashlib.sha256(content).hexdigest() != expected_sha:
        raise RuntimeError(f"Integrity mismatch: {filename}")
    reports[name] = json.loads(gzip.decompress(content))

reference = reports["int4_per_tensor"]
for name, report in reports.items():
    for key in (
        "image_ids", "checkpoint_sha256", "split_manifest_sha256",
        "annotation_sha256", "environment", "settings"
    ):
        if report[key] != reference[key]:
            raise RuntimeError(f"Comparison provenance mismatch: {name}, {key}")

metrics = {
    "fp32": reference["metrics"]["fp32_clean"],
    "int4_per_tensor": reference["metrics"]["shared_int4_clean"],
    "int4_per_channel": reports["int4_per_channel"]["metrics"],
    "int8_per_channel": reports["int8_per_channel"]["metrics"],
}

review = {
    "scope": "clean_baseline_selection_using_50_development_images",
    "source_sha256": {name: value[1] for name, value in sources.items()},
    "metrics": metrics,
    "decision": {
        "int4_candidates": "not_selected_for_broad_fault_screening_at_present",
        "int8_per_channel": "candidate_for_1000_image_clean_validation",
        "final_baseline_selected": False,
        "new_fault_campaign_authorized_by_this_record": False,
    },
    "limitations": [
        "Candidates were examined sequentially after observing earlier results.",
        "No acceptance threshold was predefined for this pilot.",
        "Small AP increase does not establish general improvement.",
        "Logical weight quantization with FP32 inference; activations are not INT8.",
        "No held-out final evaluation or native integer hardware validation.",
        "INT4 results concern these specific methods without retraining.",
    ],
}

content = (json.dumps(review, indent=2, allow_nan=False) + "\n").encode()
output.parent.mkdir(parents=True, exist_ok=True)
with output.open("xb") as stream:
    stream.write(content)

print("Saved:", output)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("INT8 remains a candidate; no new fault campaign started.")
print("No inference. Existing files unchanged. No commit or push.")

Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/docs/research_reports/Notebook05_Shared_Baseline_Candidate_Review_v1.json
SHA256: 7ae1bb8cce9107932722ddd80b591d661bf462d424e19f9b6d4f3bcb0e31b933
INT8 remains a candidate; no new fault campaign started.
No inference. Existing files unchanged. No commit or push.


In [17]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

repo = project / "code"
review_path = (
    repo / "docs/research_reports/"
    "Notebook05_Shared_Baseline_Candidate_Review_v1.json"
)
review_bytes = review_path.read_bytes()
review_sha = hashlib.sha256(review_bytes).hexdigest()

if review_sha != (
    "7ae1bb8cce9107932722ddd80b591d661bf462d424e19f9b6d4f3bcb0e31b933"
):
    raise RuntimeError("Candidate review changed.")

output = (
    repo / "experiments/notebook05/"
    "shared_INT8_clean_quality_gate_v1.json"
)
if output.exists():
    raise FileExistsError("Quality gate already exists.")

gate = {
    "schema_version": "1.0",
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "candidate_review_sha256": review_sha,
    "candidate": "63_eligible_Conv2d_weights_logical_INT8_per_output_channel",
    "reference": "FP32_same_checkpoint_same_images_and_pipeline",
    "images": "1000_frozen_development_images",
    "criteria": {
        "candidate_minus_FP32_AP50_95_minimum": -0.01,
        "candidate_minus_FP32_AR100_minimum": -0.01,
        "both_required": True,
        "finite_predictions_and_complete_image_execution_required": True,
    },
    "threshold_interpretation": (
        "Provisional engineering loss budget of one percentage point "
        "per metric, not a literature-derived universal threshold."
    ),
    "timing_disclosure": (
        "Defined after observing the 50-image candidate pilots and the "
        "existing FP32 development baseline, before the candidate's "
        "1000-image evaluation. The pilot is included in development."
    ),
    "pass_means": (
        "Eligible for further screening-method development, "
        "not proof of equivalence or final deployment suitability."
    ),
    "fail_action": (
        "Record the result and revise the candidate or scope through "
        "an explicit amendment; do not silently relax the thresholds."
    ),
    "report_additionally": [
        "AP50", "AP75", "AP_by_object_size", "AR_by_object_size"
    ],
    "limitations": [
        "Point-estimate gate does not establish statistical non-inferiority.",
        "Development evaluation is not an independent final test.",
        "Weight-only logical INT8 uses FP32 computation.",
    ],
    "held_out_evaluation": False,
    "fault_injection": False,
}

content = (json.dumps(gate, indent=2, allow_nan=False) + "\n").encode()
output.parent.mkdir(parents=True, exist_ok=True)
with output.open("xb") as stream:
    stream.write(content)

print("Saved:", output)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("Loss budgets: 0.01 AP50:95 and 0.01 AR100.")
print("No inference. No existing files modified. No commit or push.")

Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/experiments/notebook05/shared_INT8_clean_quality_gate_v1.json
SHA256: dfc04556e731df68af9375c161778e41b87f66601856265703ea8acd69fb64f3
Loss budgets: 0.01 AP50:95 and 0.01 AR100.
No inference. No existing files modified. No commit or push.


In [18]:
from pathlib import Path
import subprocess

project = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance")
repo = project / "code"

for label, args in [
    ("HEAD", ["rev-parse", "HEAD"]),
    ("Working tree", [
        "--no-optional-locks", "status", "--short",
        "--untracked-files=normal"
    ]),
    ("Staged changes", ["diff", "--cached", "--stat"]),
]:
    result = subprocess.run(
        ["git", *args],
        cwd=repo, capture_output=True, text=True, check=True,
    )
    print(f"\n--- {label} ---")
    print(result.stdout.strip() or "(no output)")

print("\n--- Notebook 05 candidates ---")
for folder in (
    Path("/content/drive/MyDrive/Colab Notebooks"),
    repo / "notebooks",
):
    if folder.is_dir():
        for path in sorted(folder.glob("05*.ipynb")):
            print(path)
            print("Size (bytes):", path.stat().st_size)

print("\n--- Saved main-study files ---")
main_results = repo / "results/notebook05/main_v1"
print("Prediction archives:", len(list(main_results.glob("*_predictions.json.gz"))))
print("Evaluation reports:", len(list(main_results.glob("*_evaluation_v1.json"))))
print("Layer summaries:", len(list(main_results.glob("L*_sensitivity_summary_v1.json"))))

print("\nRead-only inspection. No commit or push performed.")


--- HEAD ---
a8eaad0fdb3acd82320b99c3fe39cf82092f9c61

--- Working tree ---
?? docs/research_reports/Notebook05_Design_Review_Decision_v1.json
?? docs/research_reports/Notebook05_Layer_Coverage_Audit_v1.json
?? docs/research_reports/Notebook05_Shared_Baseline_Candidate_Review_v1.json
?? experiments/notebook05/

--- Staged changes ---
(no output)

--- Notebook 05 candidates ---
/content/drive/MyDrive/Colab Notebooks/05_fault_tolerance_evaluation.ipynb
Size (bytes): 404169

--- Saved main-study files ---
Prediction archives: 64
Evaluation reports: 64
Layer summaries: 3

Read-only inspection. No commit or push performed.


In [19]:
from pathlib import Path
import subprocess

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")

ignore_file = repo / ".gitignore"
print("--- Repository .gitignore ---")
print(
    ignore_file.read_text(encoding="utf-8")
    if ignore_file.is_file() else "NOT FOUND"
)

paths = [
    "results/notebook05/main_v1/FP32-clean_predictions.json.gz",
    "results/notebook05/main_v1/FP32-clean_evaluation_v1.json",
    "results/notebook05/main_v1/L01_sensitivity_summary_v1.json",
    "notebooks/05_fault_tolerance_evaluation.ipynb",
]

print("\n--- Ignore rules affecting publication candidates ---")
result = subprocess.run(
    ["git", "check-ignore", "-v", "--", *paths],
    cwd=repo, capture_output=True, text=True,
)
if result.returncode not in (0, 1):
    raise RuntimeError("Git ignore inspection failed.")
print(result.stdout.strip() or "(no matching ignore rules)")

print("\n--- Notebook 05 experiment files ---")
folder = repo / "experiments/notebook05"
if folder.is_dir():
    for path in sorted(folder.rglob("*")):
        if path.is_file():
            print(path.relative_to(repo), "| bytes:", path.stat().st_size)

print("\nRead-only inspection. No files staged, committed or pushed.")

--- Repository .gitignore ---
# Python
__pycache__/
*.py[cod]
.ipynb_checkpoints/
.pytest_cache/
.venv/
venv/

# Secrets
.env
*.key

# Model weights and large artifacts
*.pt
*.pth
*.onnx
*.engine

# Experiment outputs
artifacts/
datasets/
results/
!results/
results/*
!results/fault_injection/
runs/
wandb/

# OS files
.DS_Store


--- Ignore rules affecting publication candidates ---
.gitignore:24:results/*	results/notebook05/main_v1/FP32-clean_predictions.json.gz
.gitignore:24:results/*	results/notebook05/main_v1/FP32-clean_evaluation_v1.json
.gitignore:24:results/*	results/notebook05/main_v1/L01_sensitivity_summary_v1.json

--- Notebook 05 experiment files ---
experiments/notebook05/coco2017_split_manifest_v1.json | bytes: 71007
experiments/notebook05/int4_main_analysis_protocol_v1.json | bytes: 2133
experiments/notebook05/int4_main_execution_protocol_v1.json | bytes: 22954
experiments/notebook05/int4_main_fault_plan_v1.json | bytes: 97031
experiments/notebook05/shared_INT8_clean_quali

In [20]:
import gzip
import hashlib
import json
from pathlib import Path

repo = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
source = repo / "results/notebook05"
main = source / "main_v1"
package = repo / "docs/research_reports/notebook05_checkpoint_v1"

if package.exists():
    raise FileExistsError("Checkpoint package already exists. Inspect before rerunning.")

def sha256(content):
    return hashlib.sha256(content).hexdigest()

plan_path = repo / "experiments/notebook05/int4_main_fault_plan_v1.json"
plan_bytes = plan_path.read_bytes()
if sha256(plan_bytes) != (
    "906af255a7b46654f4736e4a3f77c8c778570ed3d86e5e12077bf9dbcc6f0fbb"
):
    raise RuntimeError("Fault plan changed.")
plan = json.loads(plan_bytes)
scenarios = {s["scenario_id"]: s for s in plan["scenarios"]}

split_path = repo / "experiments/notebook05/coco2017_split_manifest_v1.json"
split_bytes = split_path.read_bytes()
if sha256(split_bytes) != (
    "143c3273b2bf0cd6393cfe148a74e48fbafb74e770bd3fbdf3f14222355978ee"
):
    raise RuntimeError("Split manifest changed.")
development_ids = json.loads(split_bytes)["image_ids"]["development"]

evaluations = sorted(main.glob("*_evaluation_v1.json"))
if len(evaluations) != 64:
    raise RuntimeError("Expected 64 main-study evaluations.")

small_files = {}
completed_faults = []

for evaluation_path in evaluations:
    evaluation_bytes = evaluation_path.read_bytes()
    evaluation = json.loads(evaluation_bytes)
    run_id = evaluation["run_id"]
    prediction_path = main / f"{run_id}_predictions.json.gz"
    compressed = prediction_path.read_bytes()

    if sha256(compressed) != evaluation["prediction_file_sha256"]:
        raise RuntimeError(f"Prediction hash mismatch: {run_id}")
    predictions = json.loads(gzip.decompress(compressed))

    if (
        evaluation["status"] != "PREDICTIONS_AND_EVALUATION_COMPLETE"
        or evaluation["image_ids"] != development_ids
        or predictions["image_ids"] != development_ids
        or predictions["run_id"] != run_id
        or predictions["fault_plan_sha256"] != sha256(plan_bytes)
        or evaluation["fault_plan_sha256"] != sha256(plan_bytes)
        or predictions["execution_protocol_sha256"]
            != evaluation["execution_protocol_sha256"]
    ):
        raise RuntimeError(f"Run provenance mismatch: {run_id}")

    if [r["image_id"] for r in predictions["per_image"]] != development_ids:
        raise RuntimeError(f"Incomplete execution record: {run_id}")
    if sum(r["prediction_count"] for r in predictions["per_image"]) != len(
        predictions["predictions"]
    ):
        raise RuntimeError(f"Prediction count mismatch: {run_id}")

    if run_id.startswith("N05-"):
        if (
            predictions["metadata"]["fault"] != scenarios[run_id]
            or evaluation["metadata"] != predictions["metadata"]
            or evaluation["model_state_sha256"] != predictions["model_state_sha256"]
        ):
            raise RuntimeError(f"Fault scenario mismatch: {run_id}")
        completed_faults.append(run_id)

    small_files[evaluation_path.name] = evaluation_bytes

if set(completed_faults) != {f"N05-{i:03d}" for i in range(1, 61)}:
    raise RuntimeError("Expected completed fault scenarios N05-001 through N05-060.")

summary_hashes = {
    "L01_sensitivity_summary_v1.json":
        "c851bc9eae00f7bdfa2516c2cab8e5a56c477801fdaec2c0b0e3b8fb5926239b",
    "L02_sensitivity_summary_v1.json":
        "28e81f185dbdaf53dbeb8f2355188ac0bd5a71d0c278014edefc06c28373969a",
    "L03_sensitivity_summary_v1.json":
        "cc65fdabc9a7c8c9999ee12fdad526474f778a15e26de4a346f873fd23142136",
}
for name, expected in summary_hashes.items():
    content = (main / name).read_bytes()
    if sha256(content) != expected:
        raise RuntimeError(f"Summary hash mismatch: {name}")
    small_files[name] = content

inventory = []
for path in sorted(source.rglob("*")):
    if path.is_file():
        content = path.read_bytes()
        inventory.append({
            "repository_relative_path": str(path.relative_to(repo)),
            "bytes": len(content),
            "sha256": sha256(content),
            "partial_file": path.name.endswith(".part"),
        })

manifest = {
    "scope": "interim_checkpoint_not_completed_thesis_validation",
    "completed_fault_scenarios": sorted(completed_faults),
    "planned_fault_scenarios": 240,
    "campaign_status": "paused_after_60_pending_redesign",
    "copied_evaluation_reports": 64,
    "copied_layer_summaries": 3,
    "copied_file_sha256": {
        name: sha256(content) for name, content in small_files.items()
    },
    "drive_artifact_inventory": inventory,
    "limitations": [
        "Hashes identify files; they do not independently reproduce inference.",
        "Large prediction archives remain in Drive and are not bundled here.",
        "A hash manifest does not provide access to externally stored archives.",
        "Shared INT8 has only pilot evaluation; 1000-image evaluation is pending.",
    ],
}

package.mkdir(parents=True, exist_ok=False)
for name, content in small_files.items():
    with (package / name).open("xb") as stream:
        stream.write(content)

manifest_bytes = (
    json.dumps(manifest, indent=2, allow_nan=False) + "\n"
).encode()
with (package / "artifact_manifest_v1.json").open("xb") as stream:
    stream.write(manifest_bytes)

print("Package:", package)
print("Verified fault scenarios:", len(completed_faults))
print("Copied evaluations:", len(evaluations))
print("Copied summaries:", len(summary_hashes))
print("Inventoried result files:", len(inventory))
print("Partial files:", sum(item["partial_file"] for item in inventory))
print("Manifest SHA256:", sha256(manifest_bytes))
print("Original files and ignore rules unchanged. No commit or push.")

Package: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/docs/research_reports/notebook05_checkpoint_v1
Verified fault scenarios: 60
Copied evaluations: 64
Copied summaries: 3
Inventoried result files: 145
Partial files: 0
Manifest SHA256: 60a362b5e6b15e81f9837377c4b07e51b69125c1196ef0a7a151c2ddda25eabd
Original files and ignore rules unchanged. No commit or push.
